# Supplementary tables and figures

Supplementary Tables 2–7 and Supplementary Figs. 1–15 of the paper, from the outputs of the pipeline (`../workflow/`) and of the analyses (`../analyses/`). Set the paths in the first cell and run it, run the Setup cells (a few minutes), then run the cells of any table or figure on their own.

| Item | Saved to |
|---|---|
| Supplementary Table 2. Sequencing summary for each library | `TABLE_DIR/sequencing_summary.tsv` |
| Supplementary Table 3. Read filters for each library | `TABLE_DIR/read_filters.tsv` |
| Supplementary Table 4. Callable sites, post-calling variant filters and variant sets for each library | `TABLE_DIR/post_calling_filters.tsv` |
| Supplementary Table 5. Strand-consensus mismatch rates by substitution type and number of passes | `TABLE_DIR/strand_mismatch_rates.tsv` |
| Supplementary Table 6. Callable base pairs by trinucleotide context | `TABLE_DIR/trinucleotide_opportunity.tsv` |
| Supplementary Table 7. Variant table | `TABLE_DIR/variant_table.xlsx`, `variant_table.tsv` |
| Supplementary Fig. 1. Filter flowchart | `FIG_DIR/filter_flowchart.pdf` |
| Supplementary Fig. 2. Read-length distribution for each library | `FIG_DIR/read_length_distribution.pdf` |
| Supplementary Fig. 3. Relative density of singletons and control variants vs distance from the read end | `FIG_DIR/rate_vs_distance_all.pdf` |
| Supplementary Fig. 4. Cumulative singletons vs distance from the read end | `FIG_DIR/cumulative_singletons_all.pdf` |
| Supplementary Fig. 5. Mismatch rate by the number of passes | `FIG_DIR/strand_mismatch_rate_by_passes.pdf` |
| Supplementary Fig. 6. Mismatch rate of each substitution and its reverse complement | `FIG_DIR/strand_mismatch_rate_both_strands.pdf` |
| Supplementary Fig. 7. VAF of the variants on two or more molecules, with the mixture model | `FIG_DIR/vaf_mixture_model.pdf` |
| Supplementary Fig. 8. Concordance of the short and long libraries of the 60-week-old mouse | `FIG_DIR/library_concordance_60wk.pdf` |
| Supplementary Fig. 9. Coverage distribution for each library | `FIG_DIR/coverage_distribution.pdf` |
| Supplementary Fig. 10. Coverage at the somatic and the germline or early embryonic mutations | `FIG_DIR/coverage_at_mutations.pdf` |
| Supplementary Fig. 11. Point mutations per diploid genome vs age, by minimum coverage | `FIG_DIR/somatic_snvs_per_genome_vs_age_by_min_cov.pdf` |
| Supplementary Fig. 12. Somatic point mutations per diploid genome vs age, ours and Grońska-Pęski et al. (2026) | `FIG_DIR/point_mutations_per_genome_vs_age_ours_and_CR.pdf` |
| Supplementary Fig. 13. Number of passes for each library | `FIG_DIR/passes_distribution.pdf` |
| Supplementary Fig. 14. 96-channel spectrum of the strand-consensus mismatches | `FIG_DIR/strand_mismatch_spectrum.pdf` |
| Supplementary Fig. 15. Neighbour-joining tree of the seven animals | `FIG_DIR/nj_tree_inherited_variants.pdf` |

Run the setup cells first; the other sections do not depend on each other.

In [ ]:
# =============================================================================
# Paths: replace with yours
#   RESULTS    results_dir of config/config.yaml: the outputs of the pipeline and the analyses
#   FIG_DIR    where the figures are saved (PDF)
#   TABLE_DIR  where the tables are saved (TSV; Supplementary Table 7 also as xlsx)
# =============================================================================
RESULTS = "../results"
FIG_DIR = "../figures"
TABLE_DIR = "../tables"

## Setup

Six cells that define the libraries, the input paths, the loaders, the figure style (every figure is drawn at its printed size) and the shared computations: read-end counts (0c), strand-consensus mismatches (0d), the mixture model (0e) and the age regression (0f). The data of 0c–0e are loaded when a figure first asks for them and kept for the next one.

In [ ]:
# =============================================================================
# 0a. SETUP — libraries, names, colours, paths, loaders
#   final_variants()    somatic mutations and germline candidates (Methods, Somatic and germline mutations)
#   animal_genotypes()  genotypes of the seven animals at the sites of the tree (Supplementary Fig. 15)
# =============================================================================
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pysam

# Inputs: the outputs of the pipeline and the analyses, under RESULTS (first cell)
RES        = Path(RESULTS)
READS_DIR  = RES / "read_tables"                  # <library>.reads.npz: one row per strand consensus read
BAM_DIR    = RES / "alignment"                    # <library>.sorted.bam
THRESH_DIR = RES / "filter_thresholds"            # the library thresholds of the read and molecule filters
VC_DIR     = RES / "variant_calling"              # ext0/<library>/informative_coverage_dist.tsv (untrimmed reads)
CALL_DIR   = RES / "call_tables"                  # ext450/<library>/single_strand_calls.tsv.gz, mutations_with_vaf.tsv
MASK_DIR   = RES / "masked_sites"                 # excluded_regions.bed, shared_sites.bed
CB_DIR     = RES / "callable_bps" / "trimmed"     # denominator.tsv, coverage_tables/<library>.npz
MT_DIR     = RES / "mutation_tables"              # untrimmed_reads/, 450b_trimmed_reads/
CAND_DIR   = RES / "somatic_mutation_candidates"
EDGE_DIR   = RES / "read_end_errors"              # <library>.edge_counts.npz
SM_DIR     = RES / "strand_mismatches"            # ext450/<library>.counts.npz, counts/ext450/<library>/<chrom>.npz
JOINT_DIR  = RES / "library_concordance"          # joint_untrimmed_coverage.npz
NJ_DIR     = RES / "animal_relationships"         # nj_sites.tsv, callable_L.tsv
SCRIPTS    = "../workflow/scripts"                # the pipeline's scripts (trinucleotide folding)

# Outputs
FIG_DIR, TABLE_DIR = Path(FIG_DIR), Path(TABLE_DIR)
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

LIBRARIES = [
    "12wk_male_ctx_1_4kb", "12wk_male_ctx_2_4kb",
    "48wk_male_ctx_1_4kb", "48wk_male_ctx_2_4kb",
    "60wk_male_ctx_2_4kb", "60wk_male_ctx_2_10kb",
    "72wk_male_ctx_1_4kb", "72wk_male_ctx_2_4kb",
]

# Library names used in the tables
LABELS = {
    "12wk_male_ctx_1_4kb":  "12wk_1",
    "12wk_male_ctx_2_4kb":  "12wk_2",
    "48wk_male_ctx_1_4kb":  "48wk_1",
    "48wk_male_ctx_2_4kb":  "48wk_2",
    "60wk_male_ctx_2_4kb":  "60wk short",
    "60wk_male_ctx_2_10kb": "60wk long",
    "72wk_male_ctx_1_4kb":  "72wk_1",
    "72wk_male_ctx_2_4kb":  "72wk_2",
}
# Library names used in the figures
FIG_LABELS = {
    "12wk_male_ctx_1_4kb":  "12wk #1",
    "12wk_male_ctx_2_4kb":  "12wk #2",
    "48wk_male_ctx_1_4kb":  "48wk #1",
    "48wk_male_ctx_2_4kb":  "48wk #2",
    "60wk_male_ctx_2_4kb":  "60wk short",
    "60wk_male_ctx_2_10kb": "60wk long",
    "72wk_male_ctx_1_4kb":  "72wk #1",
    "72wk_male_ctx_2_4kb":  "72wk #2",
}
AGE_WK = {lib: int(lib.split("wk")[0]) for lib in LIBRARIES}

# One colour per library
LIB_COLORS = dict(zip(LIBRARIES, ["#0072B2", "#D55E00", "#009E73", "#CC79A7",
                                  "#E69F00", "#56B4E9", "#882255", "#666666"]))

# Panel layout of the per-library figures: 2 x 4 by age, the two 60 wk libraries in one column
GRID = [["12wk_male_ctx_1_4kb", "48wk_male_ctx_1_4kb", "60wk_male_ctx_2_4kb",  "72wk_male_ctx_1_4kb"],
        ["12wk_male_ctx_2_4kb", "48wk_male_ctx_2_4kb", "60wk_male_ctx_2_10kb", "72wk_male_ctx_2_4kb"]]


def load_reads(lib, cols=("zmw", "strand", "np", "length", "mapped", "mapq", "contig",
                          "chrom", "n_eq", "aln_len")):
    """Per-read columns of one library, as a dict of arrays.

    zmw      ZMW hole number
    strand   0 = forward, 1 = reverse
    np       number of passes
    length   read length (bases)
    mapped   has a primary alignment
    mapq     mapping quality (0 if unmapped)
    contig   0 = unmapped, 1 = T2T autosome, 2 = other contig
    chrom    autosome number (1-19), 0 if not on a T2T autosome
    n_eq     matched bases in the alignment
    aln_len  matched + mismatched + inserted + deleted bases; divergence = 1 - n_eq / aln_len
    start    alignment start on the reference, 0-based (-1 if unmapped)
    end      alignment end on the reference, 0-based exclusive (-1 if unmapped)
    sa       has a supplementary alignment (SA tag)
    """
    with np.load(READS_DIR / f"{lib}.reads.npz") as z:
        return {c: z[c] for c in cols}


def length_bounds(lib):
    """(min, max) read length of the read-length filter."""
    lo, hi = (THRESH_DIR / f"{lib}.read_length_bounds.txt").read_text().split()
    return int(lo), int(hi)


def lendiv_cutoff(lib):
    """Maximum length difference (bases) between the forward and reverse reads of a molecule."""
    return int((THRESH_DIR / f"{lib}.strand_length_difference.txt").read_text().split()[0])


def nonov_cutoff(lib):
    """Maximum non-overlap (bases) between the reference spans of a molecule's two strand alignments."""
    return int((THRESH_DIR / f"{lib}.strand_alignment_nonoverlap.txt").read_text().split()[0])


def run_date(lib):
    """Date of the sequencing run (YYYY-MM-DD), from the read groups in the BAM header."""
    with pysam.AlignmentFile(BAM_DIR / f"{lib}.sorted.bam") as bam:
        dates = {re.search(r"RUNID=r\d+_(\d{8})", rg["DS"]).group(1)
                 for rg in bam.header.to_dict()["RG"]}
    (d,) = dates
    return f"{d[:4]}-{d[4:6]}-{d[6:]}"


missing = [lib for lib in LIBRARIES if not (READS_DIR / f"{lib}.reads.npz").exists()]
print("per-read tables missing:", ", ".join(missing) if missing else "none")


# Somatic SNVs and germline candidates (used by the post-calling table, the variant table and several figures)
VARIANTS_TSV  = MT_DIR / "450b_trimmed_reads" / "mutations_450b_trimmed_reads.tsv"
UNTRIMMED_TSV = MT_DIR / "untrimmed_reads" / "mutations_untrimmed_reads.tsv"


def final_variants(noisy_cutoff=2, min_cov=10, germline_min_molecules=2):
    """Somatic SNVs and germline candidates of every library, one row per library and site. A callable site is outside the RepeatMasker mask, has an untrimmed coverage from
    min_cov to the library's untrimmed 99th percentile and one alternative base in the library. A molecule
    carrying more than noisy_cutoff singletons (SINGLETONS_ON_MOLECULE of the trimmed table) does not count.

    Somatic SNVs (trimmed table): the candidate filters of somatic_mutation_candidates.py (pipeline), i.e. one variant
    molecule and no call in another library in both the trimmed and the untrimmed reads, and at most
    noisy_cutoff singletons on the molecule.
    Germline candidates (untrimmed table): callable in the library, not called in another animal, and at
    least germline_min_molecules variant molecules with at most noisy_cutoff singletons each, counted over
    the animal (the two libraries of the 60-week-old mouse pooled).
    TYPE is "Somatic SNV" or "Germline candidate". UNTRIMMED_VARIANT_MOLECULES and UNTRIMMED_COVERAGE are the
    library's, from the untrimmed reads; VAF is their ratio.
    """
    t = pd.read_csv(VARIANTS_TSV, sep="\t", keep_default_na=False, low_memory=False,
                    usecols=["LIBRARY", "ANIMAL", "CHROM", "POS", "REF", "ALT", "TRINUCLEOTIDE_CONTEXT", "ZMW",
                             "VARIANT_MOLECULES", "UNTRIMMED_VARIANT_MOLECULES", "UNTRIMMED_COVERAGE",
                             "UNTRIMMED_ABOVE_LIBRARY_COVERAGE_P99", "MULTIALLELIC_SITE",
                             "UNTRIMMED_MULTIALLELIC_SITE", "IN_REPEATMASKER_MASK",
                             "SHARED_WITH_OTHER_LIBRARIES", "UNTRIMMED_SHARED_WITH_OTHER_LIBRARIES",
                             "SINGLETONS_ON_MOLECULE"])
    somatic = ((t.UNTRIMMED_ABOVE_LIBRARY_COVERAGE_P99 == 0) & (t.UNTRIMMED_COVERAGE >= min_cov)
               & (t.MULTIALLELIC_SITE == 0) & (t.UNTRIMMED_MULTIALLELIC_SITE == 0) & (t.IN_REPEATMASKER_MASK == 0)
               & (t.SHARED_WITH_OTHER_LIBRARIES == 0) & (t.UNTRIMMED_SHARED_WITH_OTHER_LIBRARIES == 0)
               & (t.VARIANT_MOLECULES == 1) & (t.UNTRIMMED_VARIANT_MOLECULES == 1)
               & (pd.to_numeric(t.SINGLETONS_ON_MOLECULE, errors="coerce") <= noisy_cutoff))   # one value on single-molecule rows

    # singletons per molecule, from the trimmed reads (0 for a molecule without a trimmed call)
    per_molecule = {}
    for lib, zmws, spm in zip(t.LIBRARY, t.ZMW, t.SINGLETONS_ON_MOLECULE):
        for z, s in zip(zmws.split(","), spm.split(",")):
            per_molecule[(lib, z)] = max(per_molecule.get((lib, z), 0), int(s))

    u = pd.read_csv(UNTRIMMED_TSV, sep="\t", keep_default_na=False, low_memory=False,
                    usecols=["LIBRARY", "ANIMAL", "CHROM", "POS", "REF", "ALT", "TRINUCLEOTIDE_CONTEXT", "ZMW",
                             "VARIANT_MOLECULES", "COVERAGE", "ABOVE_LIBRARY_COVERAGE_P99", "MULTIALLELIC_SITE",
                             "IN_REPEATMASKER_MASK", "SHARED_WITH_LIBRARIES"])
    u = u.rename(columns={"VARIANT_MOLECULES": "UNTRIMMED_VARIANT_MOLECULES", "COVERAGE": "UNTRIMMED_COVERAGE"})
    callable_u = ((u.ABOVE_LIBRARY_COVERAGE_P99 == 0) & (u.UNTRIMMED_COVERAGE >= min_cov)
                  & (u.MULTIALLELIC_SITE == 0) & (u.IN_REPEATMASKER_MASK == 0))
    animal = dict(zip(u.LIBRARY, u.ANIMAL))
    other = np.array([any(l and animal[l] != animal[lib] for l in libs.split(","))      # called in another animal
                      for lib, libs in zip(u.LIBRARY, u.SHARED_WITH_LIBRARIES)])
    u["_kept"] = [sum(per_molecule.get((lib, z), 0) <= noisy_cutoff for z in zmws.split(","))
                  for lib, zmws in zip(u.LIBRARY, u.ZMW)]
    kept_animal = u.groupby(["ANIMAL", "CHROM", "POS", "ALT"])._kept.transform("sum")    # both 60 wk libraries pooled
    germline = callable_u & ~other & (kept_animal >= germline_min_molecules)

    out = pd.concat([t[somatic].assign(TYPE="Somatic SNV"),
                     u[germline].drop(columns="_kept").assign(TYPE="Germline candidate")])
    out = out.drop_duplicates(["TYPE", "LIBRARY", "CHROM", "POS"])
    out["VAF"] = out.UNTRIMMED_VARIANT_MOLECULES / out.UNTRIMMED_COVERAGE
    return out.reset_index(drop=True)


# Genotypes of the seven animals at the variant sites of the tree (variant table and tree)
ANIMALS = {"12wk_1": ["12wk_male_ctx_1_4kb"], "12wk_2": ["12wk_male_ctx_2_4kb"],
           "48wk_1": ["48wk_male_ctx_1_4kb"], "48wk_2": ["48wk_male_ctx_2_4kb"],
           "60wk":   ["60wk_male_ctx_2_4kb", "60wk_male_ctx_2_10kb"],
           "72wk_1": ["72wk_male_ctx_1_4kb"], "72wk_2": ["72wk_male_ctx_2_4kb"]}


def animal_genotypes(min_cov=10):
    """Genotypes of the seven animals at the variant sites of the tree.

    Sites: positions called in any library in the untrimmed reads with one alternative base
    (NJ_DIR/nj_sites.tsv), outside the RepeatMasker mask, where every library passes the coverage filter
    (untrimmed coverage from min_cov to the library's untrimmed p99), and whose variant is carried by at
    least two molecules in total. The two libraries of the 60-week-old mouse are pooled. Genotype of an
    animal: 0 if none of its molecules carries the variant, 2 if all of them do, 1 otherwise.

    Returns (sites, counts). sites: CHROM, POS, REF, ALT, TRINUCLEOTIDE_CONTEXT and, for each animal,
    <animal>_MOLECULES, <animal>_COVERAGE and <animal>_GENOTYPE (untrimmed reads). counts: genotyped_bp
    (sites where every library passes the coverage filter, outside the repeats), autosome_bp, called (called
    positions there) and singletons (those whose variant is carried by one molecule in total, removed).
    """
    s = pd.read_csv(NJ_DIR / "nj_sites.tsv", sep="\t")
    den = pd.read_csv(CB_DIR / "denominator.tsv", sep="\t").drop_duplicates("sample")
    p99 = den.set_index("sample").untrimmed_p99
    keep = s.IN_REPEATMASKER_MASK == 0
    for lib in LIBRARIES:
        keep &= s[f"{lib}_COVERAGE"].between(min_cov, p99[lib])
    s = s[keep]
    out = s[["CHROM", "POS", "REF", "ALT", "TRINUCLEOTIDE_CONTEXT"]].copy()
    for a, libs in ANIMALS.items():
        out[f"{a}_MOLECULES"] = s[[f"{l}_VARIANT_MOLECULES" for l in libs]].sum(axis=1)
        out[f"{a}_COVERAGE"]  = s[[f"{l}_COVERAGE" for l in libs]].sum(axis=1)
        out[f"{a}_GENOTYPE"]  = np.where(out[f"{a}_MOLECULES"] == 0, 0,
                                         np.where(out[f"{a}_MOLECULES"] == out[f"{a}_COVERAGE"], 2, 1))
    total = out[[f"{a}_MOLECULES" for a in ANIMALS]].sum(axis=1)
    L = pd.read_csv(NJ_DIR / "callable_L.tsv", sep="\t").query("CHROM == 'ALL' and MIN_COVERAGE == @min_cov")
    counts = {"genotyped_bp": int(L.L_OUTSIDE_REPEATS.iloc[0]), "autosome_bp": int(den.autosome_bp.iloc[0]),
              "called": len(out), "singletons": int((total == 1).sum())}
    out = out[total >= 2]
    out = (out.assign(_chrom=out.CHROM.str.replace("T2T", "").astype(int))
              .sort_values(["_chrom", "POS"]).drop(columns="_chrom").reset_index(drop=True))
    return out, counts

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# 0b. SETUP — figure style: page width, font sizes, panel titles, saving
#
# Figures are drawn at their printed size: the text width of the page (PAGE_WIDTH) with the font
# sizes in FONT; every one-panel figure is SINGLE_SIZE (half the text width), its panel at
# SINGLE_MARGINS, with the font sizes in FONT_SINGLE, saved uncropped; single_cells() puts such
# cells side by side. Saved as PDF to FIG_DIR.
# ═════════════════════════════════════════════════════════════════════════════
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D

PAGE_WIDTH = 6.5                  # inches: text width of the page
SINGLE_SIZE = (PAGE_WIDTH / 2, 2.6)          # inches: every one-panel figure, half the text width
SINGLE_MARGINS = (0.62, 0.16, 0.08, 0.42)    # inches around the panel of a one-panel figure: left, right, top, bottom
FONT = dict(title=9,              # panel letter (bold) and panel title
            label=8,              # axis labels
            tick=7,               # tick labels
            legend=7,             # legends and text inside the panels
            small=6,              # small panels and secondary text
            context=4)            # the 96 trinucleotide labels of a spectrum
FONT_SINGLE = dict(label=9,       # one-panel figures: axis labels
                   tick=8,        # tick labels
                   legend=8)      # legends and text inside the panel
LETTER_IN = 0.20                  # figures of one-panel cells (single_cells): strip above the cells for the panel letters, inches
LETTER_XY_IN = (0.04, 0.03)       # panel letter: right of its cell's left edge, below the top of the figure, inches
AXIS_LW  = 0.8                    # spine and tick width, points
TICK_LEN = 3                      # tick length, points
PAD_IN   = 0.02                   # white border kept when a figure is cropped to its content, inches

plt.rcParams.update({
    "pdf.fonttype": 42,            # TrueType fonts in the PDF: text stays editable
    "font.size": FONT["tick"], "axes.labelsize": FONT["label"], "axes.titlesize": FONT["title"],
    "xtick.labelsize": FONT["tick"], "ytick.labelsize": FONT["tick"], "legend.fontsize": FONT["legend"],
    "axes.linewidth": AXIS_LW, "xtick.major.width": AXIS_LW, "ytick.major.width": AXIS_LW,
    "xtick.minor.width": 0.6 * AXIS_LW, "ytick.minor.width": 0.6 * AXIS_LW,
    "xtick.major.size": TICK_LEN, "ytick.major.size": TICK_LEN,
    "xtick.minor.size": 0.6 * TICK_LEN, "ytick.minor.size": 0.6 * TICK_LEN,
    "axes.grid": False,
})


def save_figure(fig, name, crop=True):
    """Save fig as FIG_DIR/<name>.pdf.
    crop=True trims the figure to its content (PAD_IN border); crop=False keeps FIGSIZE exactly."""
    kw = dict(bbox_inches="tight", pad_inches=PAD_IN) if crop else {}
    fig.savefig(FIG_DIR / f"{name}.pdf", **kw)
    print(f"saved {FIG_DIR / name}.pdf")


def panel_title(ax, letter, text="", pad=4, fontsize=None):
    """Panel title above the top spine, starting at the left spine: the bold letter, then the text."""
    title = rf"$\mathbf{{{letter}}}$" + (f"  {text}" if text else "")
    return ax.set_title(title, loc="left", pad=pad, fontsize=fontsize or FONT["title"])


def library_grid(figsize, margins=(0.52, 0.08, 0.22, 0.36), gaps=(0.16, 0.30), top_xticks=False):
    """2 x 4 panels, one per library in GRID order, placed in inches inside figsize.
    margins: left, right, top, bottom; gaps: between columns, between rows. top_xticks=True also
    labels the x ticks of the top row (give the row gap room for them). Returns fig, axes[2, 4]."""
    W, H = figsize
    left, right, top, bottom = margins
    col_gap, row_gap = gaps
    pw = (W - left - right - 3 * col_gap) / 4
    ph = (H - top - bottom - row_gap) / 2
    fig = plt.figure(figsize=figsize)
    axes = np.empty((2, 4), dtype=object)
    for i in range(2):
        for j in range(4):
            x0, y0 = left + j * (pw + col_gap), bottom + (1 - i) * (ph + row_gap)
            axes[i, j] = fig.add_axes([x0 / W, y0 / H, pw / W, ph / H],
                                      sharex=axes[0, 0] if (i or j) else None,
                                      sharey=axes[0, 0] if (i or j) else None)
    for ax in axes.ravel():
        ax.tick_params(labelsize=FONT["tick"], length=TICK_LEN, width=AXIS_LW)
    if not top_xticks:
        for ax in axes[0, :]:
            ax.tick_params(labelbottom=False)
    for ax in axes[:, 1:].ravel():
        ax.tick_params(labelleft=False)
    return fig, axes


def grid_labels(axes, xlabel, ylabel, ylabel_in=0.34, top_xlabel=False):
    """Panel titles a-h (bold letter, then the library name), the x label under the bottom row (under
    both rows with top_xlabel=True), and the y label left of each row, its inner edge ylabel_in
    inches left of the spine."""
    for k, (ax, lib) in enumerate(zip(axes.ravel(), sum(GRID, []))):
        panel_title(ax, "abcdefgh"[k], FIG_LABELS[lib])
    for ax in (axes.ravel() if top_xlabel else axes[1, :]):
        ax.set_xlabel(xlabel, fontsize=FONT["label"])
    for ax in axes[:, 0]:
        width_in = ax.get_position().width * ax.figure.get_figwidth()
        ax.set_ylabel(ylabel, fontsize=FONT["label"])
        ax.yaxis.set_label_coords(-ylabel_in / width_in, 0.5)


def poisson_ci(n, alpha=0.05):
    """Exact (Garwood) Poisson interval of the counts n."""
    from scipy import stats
    n = np.asarray(n, float)
    lo = np.where(n > 0, stats.chi2.ppf(alpha / 2, 2 * n) / 2, 0.0)
    return lo, stats.chi2.ppf(1 - alpha / 2, 2 * n + 2) / 2


def single_cells(figsize, letters):
    """A figure made of one-panel cells side by side, one per letter in letters. Each cell is SINGLE_SIZE with its
    panel at SINGLE_MARGINS, as in a one-panel figure; above the cells, a LETTER_IN strip holds each cell's bold
    panel letter at its top left. figsize must be (len(letters) * SINGLE_SIZE[0], SINGLE_SIZE[1] + LETTER_IN).
    Returns fig, the panels (one per cell) and the cells' left edges (inches)."""
    cw, ch = SINGLE_SIZE
    W, H = figsize
    assert np.isclose(W, len(letters) * cw) and np.isclose(H, ch + LETTER_IN), "figsize does not match the cells"
    left, right, top, bottom = SINGLE_MARGINS
    fig = plt.figure(figsize=figsize)
    lefts = [k * cw for k in range(len(letters))]
    axes = [fig.add_axes([(x + left) / W, bottom / H, (cw - left - right) / W, (ch - top - bottom) / H]) for x in lefts]
    for x, letter in zip(lefts, letters):
        fig.text((x + LETTER_XY_IN[0]) / W, 1 - LETTER_XY_IN[1] / H, letter, fontsize=FONT["title"],
                 fontweight="bold", ha="left", va="top")
    return fig, axes, lefts


print(f"figures: {FIG_DIR} ({PAGE_WIDTH} in wide)")

In [ ]:
# =============================================================================
# 0c. SETUP — read-end counts: singletons, control variants and callable base pairs by distance from
#     the read end (Supplementary Figs. 3 and 4; Methods, Errors at the edges of consensus reads)
#
# edge_effect() loads the counts once and returns them with the model of the edge errors:
#   E.CNT[lib_set], E.CNT_ALL[lib_set]   lib_set "all" (the eight libraries), "short" (the seven
#       4 kb libraries) or "long" (the 10 kb library)
#       CNT: mutations of molecules with > MAX_SINGLETONS singletons in the trimmed reads dropped;
#       CNT_ALL: every molecule. Each holds, at d = 1 .. WINDOW (index d - 1) in reads >= MIN_LEN,
#       summed over the libraries of the set:
#       ["fold"]["callable"]             callable base pairs (none counted at d <= SKIP_D)
#       ["fold"]["singletons"]           singletons (variant on one molecule of the animal)
#       ["fold"]["control"]              supporting molecules of variants on >= 2 molecules of the library
#       ["fold"]["expected_singletons"]  m(d) = sum over libraries l of r0_l x callable_l(d), the
#                                        singletons expected without edge errors; r0_l = S0_l / C0_l,
#                                        the interior density of library l
#       ["fold"]["expected_control"]     the same for the control class, with K0_l / C0_l
#       the interior counts S0 (singletons), K0 (control) and C0 (callable base pairs), and
#       ["per_library"]: the same counters for each library of the set.
#   E.model(c), E.edge_fraction(c), E.obs_exp(n, m)   the model of the edge errors (below)
# Distance d is from the nearer end of the forward strand consensus, in bases; d = 1 is the first or
# the last base of the read. Bases, sites and molecules are those of the somatic mutations, in the
# untrimmed reads.
#
# Inputs (under RESULTS)
#   read_end_errors/<library>.edge_counts.npz
#   mutation_tables/untrimmed_reads/mutations_untrimmed_reads.tsv
#   mutation_tables/450b_trimmed_reads/mutations_450b_trimmed_reads.tsv
#   masked_sites/shared_sites.bed
# =============================================================================
from functools import lru_cache
from types import SimpleNamespace


@lru_cache(maxsize=None)
def edge_effect():
    """Counters and model of the edge-effect figures (see the comment above)."""
    from scipy import stats
    COUNTS_DIR    = str(EDGE_DIR)
    TABLE         = str(UNTRIMMED_TSV)
    TRIMMED_TABLE = str(VARIANTS_TSV)
    SHARED_BED    = str(MASK_DIR / "shared_sites.bed")
    SHORT = ["12wk_male_ctx_1_4kb", "12wk_male_ctx_2_4kb",
             "48wk_male_ctx_1_4kb", "48wk_male_ctx_2_4kb",
             "60wk_male_ctx_2_4kb",
             "72wk_male_ctx_1_4kb", "72wk_male_ctx_2_4kb"]
    LIB_SETS = {"all": SHORT + ["60wk_male_ctx_2_10kb"], "short": SHORT, "long": ["60wk_male_ctx_2_10kb"]}
    MIN_COV = 10               # minimum untrimmed coverage (the cap is the library's untrimmed p99)
    MAX_SINGLETONS = 2         # most singletons a molecule may carry in the trimmed reads
    TRIM = 450                 # bases trimmed from each read end
    WINDOW = 1500              # d <= WINDOW
    SKIP_D = 5                 # callable base pairs at d <= SKIP_D are not counted (no mutation is called there)
    MIN_LEN = 3000             # window reads: 3000, 4000 or 5000
    INTERIOR_MIN_LEN = 4000    # multiple of 250
    INTERIOR_MARGIN = 1500     # multiple of 100, >= WINDOW
    T_STEP = 25                # trims tabulated every T_STEP bases
    TEST_BIN = 150             # bases per bin of the chi-square tests (divides WINDOW)
    Z_UPPER = stats.norm.ppf(0.95)   # one-sided 95% upper bound of the FDR
    ENDS = ("fold", "5", "3")

    # ── inputs ───────────────────────────────────────────────────────────────────
    ALL_LIBS = LIB_SETS["all"]
    COUNTS = {}
    for lib in ALL_LIBS:
        z = np.load(f"{COUNTS_DIR}/{lib}.edge_counts.npz")
        COUNTS[lib] = {k: z[k] for k in z.files}
        assert len(COUNTS[lib]["chroms"]) == 19, f"{lib}: {len(COUNTS[lib]['chroms'])} chromosomes"

    # untrimmed calls, one row per molecule carrying the non-reference allele
    _t = pd.read_csv(TABLE, sep="\t", dtype=str, keep_default_na=False)
    _t = _t[_t.LIBRARY.isin(ALL_LIBS)].copy()
    assert not _t.duplicated(["LIBRARY", "CHROM", "POS", "ALT"]).any()
    _t["VARIANT_MOLECULES_ANIMAL"] = (_t.VARIANT_MOLECULES.astype(np.int64)
                                      .groupby([_t.ANIMAL, _t.CHROM, _t.POS, _t.ALT]).transform("sum"))
    _per_mol = ["ZMW", "FWD_POSITION_IN_READ", "FWD_READ_LENGTH", "REV_POSITION_IN_READ", "REV_READ_LENGTH"]
    for col in _per_mol:
        _t[col] = _t[col].str.split(",")
    CALLS = _t.explode(_per_mol, ignore_index=True)
    for col in ["POS", "VARIANT_MOLECULES", "COVERAGE", "LIBRARY_COVERAGE_P99", "MULTIALLELIC_SITE",
                "IN_REPEATMASKER_MASK"] + _per_mol[1:]:
        CALLS[col] = CALLS[col].astype(np.int64)
    CALLS["pos0"] = CALLS.POS - 1
    CAP = CALLS.groupby("LIBRARY").LIBRARY_COVERAGE_P99.first().to_dict()

    # sites called in more than one animal (excluded for singletons, as for the somatic mutations)
    _bed = pd.read_csv(SHARED_BED, sep="\t", header=None, usecols=[0, 1, 2], names=["chrom", "start", "end"])
    assert (_bed.end - _bed.start == 1).all()
    _shared = set(zip(_bed.chrom, _bed.start))
    CALLS["SHARED_SITE"] = [(c_, p_) in _shared for c_, p_ in zip(CALLS.CHROM, CALLS.pos0)]
    _animal = CALLS.groupby("LIBRARY").ANIMAL.first().to_dict()
    _other = [any(l_ and _animal[l_] != a_ for l_ in s_.split(","))
              for s_, a_ in zip(CALLS.SHARED_WITH_LIBRARIES, CALLS.ANIMAL)]
    assert (CALLS.SHARED_SITE.to_numpy() == np.array(_other)).all(), "shared sites = called in another animal"

    # singletons per molecule in the trimmed reads
    _tt = pd.read_csv(TRIMMED_TABLE, sep="\t", dtype=str, keep_default_na=False,
                      usecols=["LIBRARY", "ZMW", "SINGLETONS_ON_MOLECULE"])
    _tt = _tt[_tt.LIBRARY.isin(ALL_LIBS)]
    for col in ["ZMW", "SINGLETONS_ON_MOLECULE"]:
        _tt[col] = _tt[col].str.split(",")
    _tt = _tt.explode(["ZMW", "SINGLETONS_ON_MOLECULE"]).drop_duplicates()
    assert not _tt.duplicated(["LIBRARY", "ZMW"]).any(), "one SINGLETONS_ON_MOLECULE per molecule"
    _som = _tt.set_index(["LIBRARY", "ZMW"]).SINGLETONS_ON_MOLECULE.astype(np.int64)
    CALLS["SINGLETONS_ON_MOLECULE_TRIMMED"] = (_som.reindex(pd.MultiIndex.from_arrays([CALLS.LIBRARY, CALLS.ZMW]))
                                               .fillna(0).astype(np.int64).to_numpy())

    # every mutation of the table must be a d-call of the counting pass (same read, same position)
    for lib in ALL_LIBS:
        c, d = COUNTS[lib], CALLS[CALLS.LIBRARY == lib]
        a = set(zip(c["dcall_chrom"], c["dcall_pos0"], c["dcall_alt"], c["dcall_fwd_qpos"],
                    c["dcall_fwd_len"], c["dcall_rev_qpos"], c["dcall_rev_len"]))
        b = set(zip(d.CHROM, d.pos0, d.ALT, d.FWD_POSITION_IN_READ, d.FWD_READ_LENGTH,
                    d.REV_POSITION_IN_READ, d.REV_READ_LENGTH))
        assert a == b, f"{lib}: mutation table and counting pass disagree"

    _z = COUNTS[ALL_LIBS[0]]
    WIN_EDGES, COV_MAX = _z["win_len_edges"], int(_z["cov_max"])
    LEN_BIN, FD_BIN = int(_z["len_bin"]), int(_z["fd_bin"])
    assert int(_z["window"]) == WINDOW and MIN_LEN in WIN_EDGES
    assert INTERIOR_MIN_LEN % LEN_BIN == 0 and INTERIOR_MARGIN % FD_BIN == 0 and INTERIOR_MARGIN >= WINDOW
    assert WINDOW % TEST_BIN == 0
    assert max(CAP.values()) < COV_MAX, "the top coverage stratum holds everything >= COV_MAX"
    T_GRID = np.arange(0, WINDOW + 1, T_STEP)
    print(f"{len(ALL_LIBS)} libraries x 19 chromosomes; every mutation matches the counting pass")
    print("untrimmed p99 caps:", CAP)


    # ── counters ─────────────────────────────────────────────────────────────────
    def strata(cap):
        """Boolean (2 repeat, 2 shared, COV_MAX + 1 coverage): the callable strata
        (outside the RepeatMasker mask, MIN_COV <= untrimmed coverage <= cap)."""
        rep, sh, cov = np.meshgrid([False, True], [False, True], np.arange(COV_MAX + 1), indexing="ij")
        return ~rep & (cov >= MIN_COV) & (cov <= cap)


    def classes(d, cap, drop_noisy=True):
        """Singleton and control masks for the calls d of one library. drop_noisy: drop the
        mutations of molecules with > MAX_SINGLETONS singletons in the trimmed reads."""
        cov, n_var = d.COVERAGE.to_numpy(), d.VARIANT_MOLECULES.to_numpy()
        ok = ((cov >= MIN_COV) & (cov <= cap)
              & (d.IN_REPEATMASKER_MASK.to_numpy() == 0)
              & (d.MULTIALLELIC_SITE.to_numpy() == 0))
        if drop_noisy:
            ok &= d.SINGLETONS_ON_MOLECULE_TRIMMED.to_numpy() <= MAX_SINGLETONS
        single = ok & (d.VARIANT_MOLECULES_ANIMAL.to_numpy() == 1) & ~d.SHARED_SITE.to_numpy()
        control = ok & (n_var >= 2)
        return single, control


    def library_counts(lib, frame="fwd", drop_noisy=True):
        """Counters of one library, distances in strand consensus `frame` ("fwd" or "rev").
        'fold' = distance to the nearer end; '5' and '3' = each end. drop_noisy: as in classes().
        Window callable base pairs at d <= SKIP_D are set to 0; callable_all (bases lost) keeps them.
        expected_*: the library's interior rate x its callable base pairs at each d."""
        e5, e3, fi, F = (0, 1, 0, "FWD") if frame == "fwd" else (2, 3, 1, "REV")
        cls = WIN_EDGES >= MIN_LEN
        li, fj = INTERIOR_MIN_LEN // LEN_BIN, INTERIOR_MARGIN // FD_BIN
        c = COUNTS[lib]
        keep = strata(CAP[lib])
        w5 = c["win"][e5][cls].sum(axis=0)[keep].sum(axis=0)
        w3 = c["win"][e3][cls].sum(axis=0)[keep].sum(axis=0)
        out = {e: dict(callable=w.copy()) for e, w in (("5", w5), ("3", w3), ("fold", w5 + w3))}
        out["callable_all"] = c["folded"][fi][keep].sum(axis=0)
        out["C0"] = int(c["folded_len"][fi][keep].sum(axis=0)[li:, fj:].sum())

        d = CALLS[CALLS.LIBRARY == lib]
        single, control = classes(d, CAP[lib], drop_noisy)
        q, L = d[f"{F}_POSITION_IN_READ"].to_numpy(), d[f"{F}_READ_LENGTH"].to_numpy()
        dist = {"5": q, "3": L - 1 - q, "fold": np.minimum(q, L - 1 - q)}     # d - 1
        win = L >= MIN_LEN
        out["n_single_win"] = int((single & win).sum())
        out["fold_single"] = dist["fold"][single]
        interior = (L >= INTERIOR_MIN_LEN) & (dist["fold"] >= INTERIOR_MARGIN)
        out["S0"] = int((single & interior).sum())
        out["K0"] = int((control & interior).sum())
        for e in ENDS:
            m = win & (dist[e] < WINDOW)
            out[e]["singletons"] = np.bincount(dist[e][single & m], minlength=WINDOW)
            out[e]["control"] = np.bincount(dist[e][control & m], minlength=WINDOW)
            assert out[e]["singletons"][:SKIP_D].sum() == 0 and out[e]["control"][:SKIP_D].sum() == 0, \
                f"{lib}: mutations at d <= SKIP_D ({e})"
            out[e]["callable"][:SKIP_D] = 0
            out[e]["expected_singletons"] = out["S0"] / out["C0"] * out[e]["callable"]
            out[e]["expected_control"] = out["K0"] / out["C0"] * out[e]["callable"]
        return out


    def build(libs, frame="fwd", drop_noisy=True):
        """Counters of the libraries `libs`: each library's counters (library_counts) summed;
        'per_library' keeps them one by one."""
        per = [library_counts(lib, frame, drop_noisy) for lib in libs]
        out = {e: {k: sum(p[e][k] for p in per)
                   for k in ("callable", "singletons", "control", "expected_singletons", "expected_control")}
               for e in ENDS}
        for k in ("n_single_win", "callable_all", "S0", "K0", "C0"):
            out[k] = sum(p[k] for p in per)
        out["fold_single"] = np.concatenate([p["fold_single"] for p in per])
        out["libs"], out["per_library"] = list(libs), per
        return out


    CNT = {ls: build(libs) for ls, libs in LIB_SETS.items()}                         # Supplementary Figs. 3 and 4
    CNT_ALL = {ls: build(libs, drop_noisy=False) for ls, libs in LIB_SETS.items()}   # every molecule


    # ── model ────────────────────────────────────────────────────────────────────
    #   singletons(d) = m(d) + edge errors(d),  m(d) = sum over libraries of r0_l x callable_l(d)
    #   R(t)    = sum over t < d <= WINDOW of (singletons(d) - m(d))   edge errors left after trimming t
    #   SE(R)   = sqrt(sum over libraries of [S_l(t) + S0_l x (C_l(t) / C0_l)^2])   Poisson, where
    #             S_l(t), C_l(t) = the library's singletons and callable base pairs at t < d <= WINDOW
    #   FDR(t)  = max(R, 0) / singletons kept in the window reads after trimming t from both ends;
    #   fdr_up  = one-sided 95% upper bound, max(R + Z_UPPER x SE, 0) / kept   (the lower bound is 0)
    #   fdr_raw = R / kept, unbounded
    # end = "fold" (both ends), "5" or "3" (that end's edge errors only; same kept singletons).
    # FDR_all(t): the per-base excess of the window reads, (singletons - m) / callable at each d,
    # applied to the callable bases of ALL reads; divided by all singletons kept.
    def model(c, end="fold", t_grid=T_GRID):
        s, m = c[end]["singletons"], c[end]["expected_singletons"]
        S_t = np.array([s[t:].sum() for t in t_grid])
        R = S_t - np.array([m[t:].sum() for t in t_grid])
        var = sum(np.array([p[end]["singletons"][t:].sum() + p["S0"] * (p[end]["callable"][t:].sum() / p["C0"]) ** 2
                            for t in t_grid]) for p in c["per_library"])
        se = np.sqrt(var)
        kept = c["n_single_win"] - np.r_[0, np.cumsum(c["fold"]["singletons"])][t_grid]
        cf = np.r_[0, np.cumsum(c["callable_all"])]
        excess = ((c["fold"]["singletons"] - c["fold"]["expected_singletons"])
                  / np.maximum(c["fold"]["callable"], 1))
        R_all = np.array([(excess[t:] * c["callable_all"][t:WINDOW]).sum() for t in t_grid])
        kept_all = np.array([(c["fold_single"] >= t).sum() for t in t_grid])
        return pd.DataFrame(dict(t=t_grid, R=R, se=se, kept=kept,
                                 fdr=np.maximum(R, 0) / kept,
                                 fdr_up=np.maximum(R + Z_UPPER * se, 0) / kept,
                                 fdr_raw=R / kept,
                                 fdr_all=np.maximum(R_all, 0) / kept_all, kept_all=kept_all,
                                 bases_lost=cf[t_grid] / c["callable_all"].sum()))


    def edge_fraction(c, end="fold"):
        """f = R(0) / S(0), the fraction of the window's singletons that are edge errors;
        its Poisson SE; the edge errors R(0)."""
        S, M = c[end]["singletons"].sum(), c[end]["expected_singletons"].sum()
        var_M = sum(p["S0"] * (p[end]["callable"].sum() / p["C0"]) ** 2 for p in c["per_library"])
        return 1 - M / S, (M / S) * np.sqrt(var_M / M ** 2 + 1 / S), S - M


    def obs_exp(n, m):
        """Observed / expected count, with the exact 95% Poisson interval of the observed count n."""
        lo, hi = poisson_ci(n)
        return n / m, lo / m, hi / m


    for name, cnt in (("every molecule (CNT_ALL)", CNT_ALL),
                      (f"molecules with > {MAX_SINGLETONS} singletons in the trimmed reads dropped (CNT)", CNT)):
        print(name)
        for ls, c in cnt.items():
            f, se_f, art = edge_fraction(c)
            r0 = ", ".join(f"{p['S0'] / p['C0']:.2e}" for p in c["per_library"])
            print(f"  {ls:5s}: window singletons {c['fold']['singletons'].sum():5,}, "
                  f"interior {c['S0']:4,} singletons (interior rate of each library: {r0}), "
                  f"f {f:.3f} ± {se_f:.3f}, edge artifacts {art:,.0f}; "
                  f"control: window {c['fold']['control'].sum():,}, interior {c['K0']:,}")

    return SimpleNamespace(**{k: v for k, v in locals().items() if not k.startswith("_")})

In [ ]:
# =============================================================================
# 0d. SETUP — strand-consensus mismatches (Supplementary Figs. 6 and 14; Methods, Estimation of the
#     sequencing error rate)
#
# strand_mismatches() loads the mismatches once and returns M with
#     M.mm            one row per mismatch kept by SITE_FILTERS: the columns of single_strand_calls.tsv.gz
#                     plus animal, channel (96-channel label from the pyrimidine, e.g. A[C>T]G), in_mask,
#                     at_mutation_site, err_reverse (the divergent strand consensus is aligned to the
#                     reference minus strand) and read_pyrimidine (the changed base, as read on the
#                     divergent strand consensus 5' to 3', is C or T)
#     M.CHANNELS      the 96 channels, substitution-major; M.SUBS the six pyrimidine substitutions
#     M.COMP          str.translate table of the complementary bases
#     M.LIBS, M.EXT, M.SITE_FILTERS, M.COUNTS_DIR    the libraries, the read set (450 = trimmed), the
#                     site filters and the folder of the mismatch counts (strand_mismatches/)
# A strand-consensus mismatch is a position of a molecule where both strand consensuses pass every
# read and base filter of the variant calling and only one of them carries the reference base. Site
# filters: the masked sites (RepeatMasker Simple_repeat, Low_complexity, Satellite*, plus the sites
# called in >= 2 animals) and the sites where the same animal has a mutation are dropped.
#
# Inputs (under RESULTS)
#   call_tables/ext450/<library>/single_strand_calls.tsv.gz and mutations_with_vaf.tsv
#   masked_sites/excluded_regions.bed
#   strand_mismatches/ext450/<library>.counts.npz
# =============================================================================
from functools import lru_cache
from types import SimpleNamespace


@lru_cache(maxsize=None)
def strand_mismatches():
    """Strand-consensus mismatches of the eight libraries (see the comment above)."""
    import os
    RATES_DIR  = str(CALL_DIR)
    MASK_BED   = str(MASK_DIR / "excluded_regions.bed")
    COUNTS_DIR = str(SM_DIR)
    LIBS = list(LIBRARIES)
    EXT = 450                  # read set: 450 = 450 b trimmed from each read end, 0 = untrimmed
    SITE_FILTERS = dict(
        mask           = True, # drop the masked sites (MASK_BED)
        mutation_sites = True, # drop the sites where the same animal has a mutation
    )
    MM_COLS = ["sample", "chrom", "pos_1based", "ref_base", "triplet", "call_type",
               "err_strand", "err_base", "err_sm", "err_sx", "err_np", "err_qpos", "err_readlen",
               "other_base", "other_sm", "other_sx", "other_np", "other_qpos", "other_readlen", "zmw"]

    # ── 96 channels ──────────────────────────────────────────────────────────────
    SUBS     = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]
    CHANNELS = [f"{a}[{s}]{b}" for s in SUBS for a in "ACGT" for b in "ACGT"]   # substitution-major
    _comp    = str.maketrans("ACGT", "TGCA")
    animal_of = lambda s: re.sub(r"_\d+kb$", "", s)


    def sbs96_channel(triplet, ref, alt):
        """96-channel label, written from the pyrimidine of the base pair."""
        if ref in "AG":
            triplet = triplet.translate(_comp)[::-1]
            ref, alt = ref.translate(_comp), alt.translate(_comp)
        return f"{triplet[0]}[{ref}>{alt}]{triplet[2]}"


    def divergent_read_reverse(d, lib):
        """err_reverse for the rows of one library, from the mismatch records of the counts."""
        path = f"{COUNTS_DIR}/ext{EXT}/{lib}.counts.npz"
        if not os.path.exists(path):
            raise FileNotFoundError(f"{path} is missing; run the pipeline step sequencing_errors")
        z = np.load(path)
        rec = pd.DataFrame({"chrom": z["mm_chrom"], "pos_1based": z["mm_pos0"] + 1,
                            "zmw": np.char.add("r", z["mm_zmw"].astype(str)), "err_reverse": z["mm_err_reverse"]})
        x = d[["chrom", "pos_1based", "zmw"]].merge(rec, on=["chrom", "pos_1based", "zmw"], how="left")
        if x.err_reverse.isna().any():
            raise ValueError(f"{lib}: {x.err_reverse.isna().sum()} mismatches without a record in {path}")
        return x.err_reverse.to_numpy(dtype=bool)


    def in_intervals(chrom, pos0, bed):
        """True where 0-based pos0 on chrom falls inside a BED interval."""
        out = np.zeros(len(pos0), bool)
        for c, g in bed.groupby("chrom", sort=False):
            idx = np.nonzero(chrom == c)[0]
            if not len(idx):
                continue
            st, en = g.start.to_numpy(), g.end.to_numpy()
            k = np.searchsorted(st, pos0[idx], side="right") - 1
            out[idx] = (k >= 0) & (pos0[idx] < en[np.maximum(k, 0)])
        return out


    # ── load ─────────────────────────────────────────────────────────────────────
    _parts, _muts = [], []
    for lib in LIBS:
        d = f"{RATES_DIR}/ext{EXT}/{lib}"
        _x = pd.read_csv(f"{d}/single_strand_calls.tsv.gz", sep="\t", usecols=MM_COLS)
        _x["err_reverse"] = divergent_read_reverse(_x, lib)
        _parts.append(_x)
        _muts.append(pd.read_csv(f"{d}/mutations_with_vaf.tsv", sep="\t", usecols=["chrom", "pos_1based"])
                     .assign(animal=animal_of(lib)))
    mm = pd.concat(_parts, ignore_index=True)
    mm["animal"] = mm["sample"].map(animal_of)

    # 96-channel label, and whether the divergent read carries the pyrimidine
    _key = mm.triplet + mm.err_base
    _lab = {k: sbs96_channel(k[:3], k[1], k[3]) for k in _key.unique()}
    mm["channel"] = _key.map(_lab)
    mm["read_pyrimidine"] = mm.ref_base.isin(["C", "T"]).to_numpy() != mm.err_reverse.to_numpy()

    # site annotations
    _bed = pd.read_csv(MASK_BED, sep="\t", header=None, usecols=[0, 1, 2], names=["chrom", "start", "end"])
    mm["in_mask"] = in_intervals(mm.chrom.to_numpy(), mm.pos_1based.to_numpy() - 1, _bed)
    _mut = pd.MultiIndex.from_frame(pd.concat(_muts)[["animal", "chrom", "pos_1based"]].drop_duplicates())
    mm["at_mutation_site"] = pd.MultiIndex.from_frame(mm[["animal", "chrom", "pos_1based"]]).isin(_mut)

    # site filters and the per-library summary
    _keep = np.ones(len(mm), bool)
    if SITE_FILTERS["mask"]:
        _keep &= ~mm.in_mask.to_numpy()
    if SITE_FILTERS["mutation_sites"]:
        _keep &= ~mm.at_mutation_site.to_numpy()
    mm["kept"] = _keep
    _s = mm.groupby("sample").agg(
        f=("call_type", lambda x: (x == "f").sum()), r=("call_type", lambda x: (x == "r").sum()),
        total=("call_type", "size"), in_mask=("in_mask", "sum"),
        at_mutation_site=("at_mutation_site", "sum"), kept=("kept", "sum")).reindex(LIBS)
    _s.loc["all"] = _s.sum()
    _s = _s.astype(int)
    mm = mm[mm.kept].drop(columns="kept").reset_index(drop=True)

    print(f"strand-consensus mismatches, ext{EXT}, site filters {SITE_FILTERS}")
    print(_s.to_string())
    print(f"mm: {len(mm):,} mismatches kept")
    return SimpleNamespace(mm=mm, CHANNELS=CHANNELS, SUBS=SUBS, COMP=_comp, LIBS=LIBS, EXT=EXT,
                           SITE_FILTERS=SITE_FILTERS, COUNTS_DIR=COUNTS_DIR, summary=_s)

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# 0e. SETUP — germline / somatic mixture model of the variants on two or more molecules
#     (Supplementary Fig. 7)
#
# Variant i is carried by k_i of the n_i molecules covering its site, in the untrimmed reads, both
# summed over the animal's libraries (the two libraries of the 60-week-old mouse pooled). Fitted: the
# germline candidates with 2 <= k <= n - 1. Germline: binomial(n, 1/2) truncated to that range;
# somatic: pure-birth site frequency spectrum, 1 / (k (k - 1)), truncated the same way. The somatic
# fraction p is fitted by maximum likelihood (expectation-maximization).
#
# mixture_fit() returns X with
#     X.k, X.n       the fitted variants
#     X.p, X.se, X.ci, X.w   p, its standard error, its 95% likelihood-ratio interval and the posterior
#                    probability that each variant is somatic
#     X.n_somatic    somatic mutations; X.g1 expected germline variants among them
# log_pg(k, n) and log_ps(k, n) are the two components (log probabilities).
#
# Inputs (under RESULTS): the two mutation tables; animal_relationships/nj_sites.tsv
# ═════════════════════════════════════════════════════════════════════════════
from functools import lru_cache
from types import SimpleNamespace

from scipy.special import expit, gammaln, logit


def log_pg(k, n):
    """Germline: binomial(n, 1/2), truncated to 2 <= k <= n - 1."""
    return (gammaln(n + 1) - gammaln(k + 1) - gammaln(n - k + 1)
            - n * np.log(2) - np.log1p(-(n + 2) * 2.0 ** (-n)))


def log_ps(k, n):
    """Somatic: pure-birth site frequency spectrum, 1 / (k (k - 1)), truncated to 2 <= k <= n - 1."""
    return np.log(n - 1) - np.log(n - 2) - np.log(k) - np.log(k - 1)


def em(k, n, p=0.5, tol=1e-12, max_iter=100_000):
    """Maximum-likelihood somatic fraction p by expectation-maximization.
    Returns p, its standard error (Fisher information), its 95% likelihood-ratio interval and w, the
    posterior probability that each variant is somatic."""
    d = log_ps(k, n) - log_pg(k, n)
    for _ in range(max_iter):
        p_new = np.clip(expit(logit(p) + d).mean(), 1e-12, 1 - 1e-12)   # E step, then M step
        done = abs(p_new - p) < tol
        p = p_new
        if done:
            break
    w = expit(logit(p) + d)
    se = 1 / np.sqrt(((w / p - (1 - w) / (1 - p)) ** 2).sum())
    ps, pg = np.exp(log_ps(k, n)), np.exp(log_pg(k, n))
    grid = np.linspace(1e-6, 1 - 1e-6, 5001)
    ll = np.array([np.log(q * ps + (1 - q) * pg).sum() for q in grid])
    inside = grid[ll >= ll.max() - 1.92]                    # chi2(1) 95% quantile / 2
    return p, se, (inside.min(), inside.max()), w


def germline_singletons(w, n):
    """Expected germline non-reference alleles carried by one molecule: each fitted variant counts (1 - w) times
    P(k = 1 | n) / P(2 <= k <= n - 1 | n) under binomial(n, 1/2)."""
    return ((1 - w) * n * 2.0 ** (-n) / (1 - (n + 2) * 2.0 ** (-n))).sum()


@lru_cache(maxsize=None)
def mixture_fit(MIN_COV=10, NOISY_CUTOFF=2):
    """Variants of the mixture model and its fit (see the comment above)."""
    TRIMMED   = VARIANTS_TSV
    UNTRIMMED = UNTRIMMED_TSV
    SITES     = NJ_DIR / "nj_sites.tsv"     # untrimmed coverage of every library at every called site

    # Trimmed reads: singletons per molecule (0 for a molecule without a trimmed call) and the calls made there
    t = pd.read_csv(TRIMMED, sep="\t", keep_default_na=False, low_memory=False,
                    usecols=["LIBRARY", "CHROM", "POS", "ALT", "ZMW", "SINGLETONS_ON_MOLECULE"])
    singletons = {}
    for lib, zmws, spm in zip(t.LIBRARY, t.ZMW, t.SINGLETONS_ON_MOLECULE):
        for z, s in zip(zmws.split(","), spm.split(",")):
            singletons[(lib, z)] = max(singletons.get((lib, z), 0), int(s))
    trimmed_calls = set(zip(t.LIBRARY, t.CHROM, t.POS, t.ALT))

    # Untrimmed reads: every call, one row per library, site and alternative base
    u = pd.read_csv(UNTRIMMED, sep="\t", keep_default_na=False, low_memory=False,
                    usecols=["LIBRARY", "ANIMAL", "CHROM", "POS", "REF", "ALT", "ZMW", "VARIANT_MOLECULES", "COVERAGE",
                             "ABOVE_LIBRARY_COVERAGE_P99", "MULTIALLELIC_SITE", "IN_REPEATMASKER_MASK",
                             "SHARED_WITH_LIBRARIES"])
    animal_of = dict(zip(u.LIBRARY, u.ANIMAL))

    # Filters applied to every variant:
    #   callable site in the library: outside the RepeatMasker mask, untrimmed coverage from MIN_COV to the
    #   library's untrimmed 99th percentile, one alternative base; not called in another animal
    callable_site = ((u.IN_REPEATMASKER_MASK == 0) & (u.COVERAGE >= MIN_COV)
                     & (u.ABOVE_LIBRARY_COVERAGE_P99 == 0) & (u.MULTIALLELIC_SITE == 0))
    in_other_animal = np.array([any(l and animal_of[l] != animal_of[lib] for l in libs.split(","))
                                for lib, libs in zip(u.LIBRARY, u.SHARED_WITH_LIBRARIES)])
    passing = u[callable_site & ~in_other_animal].copy()

    # Molecules of the animal carrying the non-reference allele (the two libraries of the 60-week-old mouse pooled):
    #   MOLECULES = all of them, KEPT = those with at most NOISY_CUTOFF singletons
    KEY = ["ANIMAL", "CHROM", "POS", "ALT"]
    u["KEPT"] = [sum(singletons.get((lib, z), 0) <= NOISY_CUTOFF for z in zmws.split(","))
                 for lib, zmws in zip(u.LIBRARY, u.ZMW)]
    animal_counts = u.groupby(KEY).agg(MOLECULES=("VARIANT_MOLECULES", "sum"), KEPT=("KEPT", "sum")).reset_index()
    passing = passing.merge(animal_counts, on=KEY, how="left")

    # Somatic SNVs: one molecule of the animal, kept, and called in the trimmed reads
    is_somatic = ((passing.MOLECULES == 1) & (passing.KEPT == 1)
                  & np.array([c in trimmed_calls for c in zip(passing.LIBRARY, passing.CHROM, passing.POS, passing.ALT)]))
    # Germline candidates: two or more kept molecules of the animal
    is_germline = passing.KEPT >= 2
    somatic, germline = passing[is_somatic], passing[is_germline]
    n_somatic = len(somatic)

    # Germline candidates for the model: one row per animal and site; k = kept molecules, n = untrimmed
    # coverage, both summed over the animal's libraries
    sites = pd.read_csv(SITES, sep="\t")
    germ = (germline.drop_duplicates(KEY)[KEY + ["KEPT"]].rename(columns={"KEPT": "k"})
            .merge(sites, on=["CHROM", "POS", "ALT"], how="left", validate="many_to_one"))
    germ["n"] = 0
    for a, libs in u.groupby("ANIMAL").LIBRARY.unique().items():
        rows = germ.ANIMAL == a
        germ.loc[rows, "n"] = germ.loc[rows, [f"{l}_COVERAGE" for l in libs]].sum(axis=1)
    assert (germ.n >= germ.k).all(), "coverage missing from nj_sites.tsv for some germline candidates"
    germ = germ[KEY + ["k", "n"]]

    # Fitted set: 2 <= k <= n - 1 (k = n may be homozygous)
    fit = germ[germ.k < germ.n].reset_index(drop=True)
    k, n = fit.k.to_numpy(float), fit.n.to_numpy(float)

    p, se, ci, w = em(k, n)
    g1 = germline_singletons(w, n)
    print(f"fitted variants (2 <= k <= n - 1): {len(fit):,}; somatic fraction p = {p:.3f}, 95% CI "
          f"[{ci[0]:.3f}, {ci[1]:.3f}]; expected germline singletons {g1:.1f} = {100 * g1 / n_somatic:.2f}% of the "
          f"{n_somatic:,} somatic SNVs")
    return SimpleNamespace(k=k, n=n, p=p, se=se, ci=ci, w=w, n_somatic=n_somatic, g1=g1, fit=fit)

In [ ]:
# =============================================================================
# 0f. SETUP — age regression: somatic point mutations vs age, ours and Grońska-Pęski et al. (2026),
#     corrected to the genome's trinucleotide composition (Supplementary Fig. 12)
#
# ours  somatic_mutation_candidates.tsv with SINGLETONS_ON_MOLECULE <= AGE_MAX_SPM (minimum untrimmed
#       coverage AGE_FLOOR), 7 animals (the two 60 wk libraries pooled). Per animal,
#       rate = sum_c (n_c / D_c) * G_c / G over the 32 folded contexts, with n_c and D_c the animal's
#       mutations and callable bps at context c (context_opportunity.tsv) and G_c / G the context's
#       share of the T2T autosomes; per_genome = rate x 2 x the T2T autosomes' ACGT positions.
# CR    Grońska-Pęski et al. 2026, AL cerebellar neurons at 19 months (82.5 wk): their Table S6 burden
#       per bp and its 95% CI (trinucleotide-corrected, mm10), x the same constant (cr_genome).
# Fit   OLS on our 7 animals with the 95% confidence band of the line (class Fit).
# Helpers: draw_age, broken_age_axes, break_y, ylabel_beside, fit_legend, r2_under_legend, drawn_edges,
# clear_top, check_clear, check_text_apart, report_fit. Their style knobs carry the prefix AGE_.
#
# Inputs (under RESULTS): somatic_mutation_candidates/ (somatic_mutation_candidates.tsv,
#   context_opportunity.tsv, callable_bases.tsv, t2t_reference_context_counts.tsv)
# =============================================================================
import statsmodels.api as sm
from matplotlib import legend as mpl_legend
from matplotlib.colors import to_rgb
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator, MaxNLocator

# ------------------------------ data knobs -----------------------------------
AGE_CAND     = str(CAND_DIR)
AGE_MAX_SPM  = 2                         # <= this many singletons per molecule
AGE_FLOOR    = 10                        # minimum untrimmed coverage of the mutations and callable bps
CR_AGE_WK    = 82.5                      # 19 months
# Grońska-Pęski et al. (2026), Table S6: burden per bp (trinucleotide-corrected) and its 95% CI
CR_RATE, CR_LO, CR_HI = 1.84552220935258e-08, 1.74170191716188e-08, 1.95553107661708e-08

# ------------------------------ style knobs ----------------------------------
AGE_C_OURS, AGE_C_CR = "#4C72B0", "#D55E00"      # blue circles, vermillion squares
AGE_MARKER_SIZE  = 5
AGE_BAND_ALPHA, AGE_EXT_ALPHA = 0.15, 0.07       # 95% band tint across our ages / where extended
AGE_EXT_LS       = (0, (4, 2.5))             # extended line: dashed
AGE_LINE_LW, AGE_ERR_LW, AGE_CAPSIZE = 1.2, 1.0, 2.5
AGE_AXIS_LW, AGE_TICK_LEN = AXIS_LW, TICK_LEN
AGE_LABEL_FONT, AGE_TICK_FONT, AGE_LEGEND_FONT = FONT_SINGLE["label"], FONT_SINGLE["tick"], FONT_SINGLE["legend"]
AGE_XLABEL       = "Age (weeks)"
AGE_XLIM         = (0, 86)                   # weeks: age 0 (the intercept) to room for the CR point at 82.5 wk
AGE_Y_PAD        = 0.05                      # y spans everything drawn, plus this share of it beyond each end
AGE_Y_NBINS      = 6                         # at most this many y tick intervals, round steps
AGE_LEFT, AGE_RIGHT, AGE_TOP, AGE_BOTTOM = 0.50, 0.08, 0.08, 0.42   # panel margins, inches
AGE_FIT_LOC      = "upper left"              # where the fit's legend sits
AGE_LOW_IN       = 0.17                      # height of the strip holding the 0 tick, inches
AGE_GAP_IN       = 0.06                      # gap between the strip and the main panel, inches
AGE_BREAK_IN     = 0.045                     # half-length of each break mark on the left spine, inches
AGE_LEGEND_CLEAR_IN = 0.04                   # what is drawn under a legend sits this far below it, inches
AGE_R2_GAP_PT    = 2                         # R2 sits this far under the fit's legend icon, points

WEEKS_PER_YEAR = 365 / 7

# ------------------------------- data ----------------------------------------
_COMP = str.maketrans("ACGT", "TGCA")
_fold = lambda t: t if t[1] in "CT" else t.translate(_COMP)[::-1]    # TGT -> ACA

age_calls = pd.read_csv(f"{AGE_CAND}/somatic_mutation_candidates.tsv", sep="\t",
                        usecols=["LIBRARY", "ANIMAL", "SINGLETONS_ON_MOLECULE", "TRINUCLEOTIDE_CONTEXT"])
age_calls = age_calls[age_calls.SINGLETONS_ON_MOLECULE <= AGE_MAX_SPM].copy()
age_calls["ctx"] = age_calls.TRINUCLEOTIDE_CONTEXT.map(_fold)
_opp = pd.read_csv(f"{AGE_CAND}/context_opportunity.tsv", sep="\t")
_opp = _opp[_opp.floor == AGE_FLOOR].copy()
_opp["ctx"] = _opp.triplet.map(_fold)
_cb = pd.read_csv(f"{AGE_CAND}/callable_bases.tsv", sep="\t")
_cb = _cb[_cb.floor == AGE_FLOOR].set_index("sample")
T2T_AUTOSOME_BP = int(_cb.autosome_bp.iloc[0])                # length of T2T1..T2T19, per haploid genome
_ref = pd.read_csv(f"{AGE_CAND}/t2t_reference_context_counts.tsv", sep="\t")
_ref["ctx"] = _ref.triplet.map(_fold)
G_CTX = _ref.groupby("ctx").autosomes.sum()                   # T2T autosomes, 32 folded contexts
G_ACGT = int(G_CTX.sum())                                     # haploid ACGT positions with a full triplet

# checks: one opportunity table per library that sums to its callable depth; 32 contexts
_dep = _opp.groupby("sample").callable_depth.sum()
if not (_dep.reindex(_cb.index) == _cb.callable_depth).all():
    raise ValueError("context_opportunity.tsv does not sum to callable_bases.tsv's callable_depth")
if set(_opp.ctx) != set(G_CTX.index) or len(G_CTX) != 32:
    raise ValueError("context sets differ between the opportunity and reference tables")
if not set(age_calls.ctx) <= set(G_CTX.index):
    raise ValueError("a call has a context outside the 32 folded ones")

_lib2animal = age_calls.groupby("LIBRARY").ANIMAL.first()
if set(_lib2animal.index) != set(_cb.index):
    raise ValueError("libraries differ between the candidate table and callable_bases.tsv")
_opp["animal"] = _opp["sample"].map(_lib2animal)
_N = age_calls.groupby(["ANIMAL", "ctx"]).size().unstack(fill_value=0)
_D = _opp.groupby(["animal", "ctx"]).callable_depth.sum().unstack(fill_value=0)
_N = _N.reindex(index=_D.index, columns=_D.columns, fill_value=0)
_r = _N / _D                                                  # per-context rates, per animal

ours = pd.DataFrame(index=_D.index)
ours["age_wk"] = ours.index.str.extract(r"^(\d+)wk", expand=False).astype(int).to_numpy()
ours["mutations"] = _N.sum(axis=1)
ours["callable_depth"] = _D.sum(axis=1)
ours["rate_plain"] = ours.mutations / ours.callable_depth
ours["rate"] = (_r * (G_CTX[_D.columns] / G_ACGT)).sum(axis=1)          # genome composition
ours["per_genome"] = ours.rate * 2 * G_ACGT                             # = sum_c r_c * 2 G_c
ours["per_genome_plain"] = ours.rate_plain * 2 * T2T_AUTOSOME_BP
ours = ours.sort_values("age_wk")
cr_rate = np.array([CR_RATE, CR_LO, CR_HI])
cr_genome = cr_rate * 2 * G_ACGT                               # our length: same constant as ours


# ------------------------------- fit -----------------------------------------
class Fit:
    """OLS of y on x: line, 95% band of the line, slope's 95% CI and P (slope = 0)."""
    def __init__(self, x, y):
        self.x = np.asarray(x, float)
        self.res = sm.OLS(np.asarray(y, float), sm.add_constant(self.x)).fit()
        (self.b0, self.b1), self.ci = self.res.params, self.res.conf_int(0.05)
        self.p = self.res.pvalues[1]
    def line(self, xs):
        return self.b0 + self.b1 * np.asarray(xs, float)
    def band(self, xs):
        return self.res.get_prediction(sm.add_constant(np.asarray(xs, float),
                                                       has_constant="add")).conf_int(alpha=0.05)


INSIDE  = np.linspace(ours.age_wk.min(), ours.age_wk.max(), 200)   # our ages: solid, darker band


# ------------------------------- drawing -------------------------------------
def age_panel(figsize):
    """A figure of this size with one panel inside the fixed margins."""
    W, H = figsize
    assert W > AGE_LEFT + AGE_RIGHT and H > AGE_TOP + AGE_BOTTOM, f"FIGSIZE {figsize} is too small for the margins"
    fig = plt.figure(figsize=figsize)
    ax = fig.add_axes([AGE_LEFT / W, AGE_BOTTOM / H, (W - AGE_LEFT - AGE_RIGHT) / W, (H - AGE_TOP - AGE_BOTTOM) / H])
    return fig, ax


def tint(color, alpha):
    """color laid over white at this opacity, as an opaque colour."""
    return tuple(1 - alpha * (1 - c) for c in to_rgb(color))


class FitKey:
    """Legend key for the fit: the 95% band as a shaded box with the line through
    its middle. Use: ax.legend([FitKey()], [label], handler_map={FitKey: FitKey()})."""
    def legend_artist(self, legend, orig_handle, fontsize, handlebox):
        x0, y0 = -handlebox.xdescent, -handlebox.ydescent
        w, h = handlebox.width, handlebox.height
        box = Rectangle((x0, y0), w, h, facecolor=tint(AGE_C_OURS, AGE_BAND_ALPHA), lw=0,
                        transform=handlebox.get_transform())
        line = plt.Line2D([x0, x0 + w], [y0 + h / 2] * 2, color=AGE_C_OURS, lw=AGE_LINE_LW,
                          solid_capstyle="butt", transform=handlebox.get_transform())
        handlebox.add_artist(box)
        handlebox.add_artist(line)
        return box


def age_p_text(p):
    """p as 'm × 10^e' (mathtext), or plain from 0.001 up."""
    if p >= 1e-3:
        return f"{p:.2g}"
    m, e = f"{p:.1e}".split("e")
    return rf"{m} × 10$^{{{int(e)}}}$"


def fit_legend(ax, fit, label_fmt, unit=1.0, extra=()):
    """The fit's legend: its key, labelled with the slope per year ({slope}, {lo}, {hi},
    in units of `unit`) and the slope's P ({p}). `extra`: (handle, label) pairs listed
    above the fit's key, e.g. the studies' markers. Returns the legend."""
    k = WEEKS_PER_YEAR / unit
    label = label_fmt.format(slope=fit.b1 * k, lo=fit.ci[1][0] * k, hi=fit.ci[1][1] * k, p=age_p_text(fit.p))
    return ax.legend([h for h, _ in extra] + [FitKey()], [t for _, t in extra] + [label],
                     handler_map={FitKey: FitKey()}, handlelength=2.6, handleheight=1.6,
                     loc=AGE_FIT_LOC, frameon=False, fontsize=AGE_LEGEND_FONT)


def draw_age(ax, fit, y, cr=None, unit=1.0, xlim=AGE_XLIM, extend_right=True):
    """Ours (points, fit, 95% band) and the CR point if `cr` is given, y in units of
    `unit`. The fit runs dashed (line only) from our youngest animals back to xlim[0]
    (age 0); with extend_right it also runs dashed, with a lighter band, from our
    oldest animals to xlim[1]. Returns the y range drawn and the handles
    {"ours": the points, "cr": the CR error bar or None}, for a study legend."""
    before = np.linspace(xlim[0], INSIDE[0], 100)                # back to age 0: dashed line only
    outside = np.linspace(INSIDE[-1], xlim[1], 100)              # beyond our ages: dashed, lighter band
    xs = np.concatenate([INSIDE, outside]) if extend_right else INSIDE
    if extend_right:
        b = fit.band(xs) / unit                  # lighter band under the darker one: no seam
        ax.fill_between(xs, b[:, 0], b[:, 1], color=tint(AGE_C_OURS, AGE_EXT_ALPHA), lw=0, zorder=1)
        ax.plot(outside, fit.line(outside) / unit, color=AGE_C_OURS, lw=AGE_LINE_LW, ls=AGE_EXT_LS, zorder=2)
    ax.plot(before, fit.line(before) / unit, color=AGE_C_OURS, lw=AGE_LINE_LW, ls=AGE_EXT_LS, zorder=2)
    b = fit.band(INSIDE) / unit
    ax.fill_between(INSIDE, b[:, 0], b[:, 1], color=tint(AGE_C_OURS, AGE_BAND_ALPHA), lw=0, zorder=1)
    ax.plot(INSIDE, fit.line(INSIDE) / unit, color=AGE_C_OURS, lw=AGE_LINE_LW, zorder=2)
    pts, = ax.plot(ours.age_wk, np.asarray(y) / unit, ls="", marker="o", ms=AGE_MARKER_SIZE, color=AGE_C_OURS,
                   mew=0, zorder=4)
    lo = [np.min(y), fit.band(xs)[:, 0].min(), fit.line(xlim[0])]
    hi = [np.max(y), fit.band(xs)[:, 1].max()]
    eb = None
    if cr is not None:
        eb = ax.errorbar([CR_AGE_WK], [cr[0] / unit], yerr=[[(cr[0] - cr[1]) / unit], [(cr[2] - cr[0]) / unit]],
                         fmt="s", ms=AGE_MARKER_SIZE, color=AGE_C_CR, ecolor=AGE_C_CR, elinewidth=AGE_ERR_LW,
                         capsize=AGE_CAPSIZE, capthick=AGE_ERR_LW, zorder=5)
        eb.lines[0].set_markeredgewidth(0)       # no outline; caps untouched
        lo.append(cr[1])
        hi.append(cr[2])
    return min(lo) / unit, max(hi) / unit, {"ours": pts, "cr": eb}


def style_age(ax, ylabel=None, xlabel=True, xlim=AGE_XLIM):
    """Fonts, ticks and spine widths of the age figures."""
    ax.set_xlim(xlim)
    if xlabel:
        ax.set_xlabel(AGE_XLABEL, fontsize=AGE_LABEL_FONT)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=AGE_LABEL_FONT)
    ax.tick_params(labelsize=AGE_TICK_FONT, length=AGE_TICK_LEN, width=AGE_AXIS_LW)
    for s in ax.spines.values():                 # all four spines kept
        s.set_linewidth(AGE_AXIS_LW)


def check_inside(fig, ax):
    """The panel's text must lie inside the figure."""
    fig.canvas.draw()
    bb = ax.get_tightbbox(fig.canvas.get_renderer())
    assert bb.x0 >= 0 and bb.y0 >= 0 and bb.x1 <= fig.bbox.x1 and bb.y1 <= fig.bbox.y1, \
        "text runs off the figure -- widen the margins or FIGSIZE"


def broken_age_axes(fig, box_in, low_in=AGE_LOW_IN, gap_in=AGE_GAP_IN):
    """A panel with a broken y axis, filling box_in = (x0, y0, width, height) in inches
    from the figure's lower left: the main panel on top, a strip low_in tall at the
    bottom that holds the 0 tick, and gap_in between them. Returns (main panel, strip).
    Draw on the main panel, then call break_y."""
    W, H = fig.get_size_inches()
    x0, y0, w, h = box_in
    main_in = h - low_in - gap_in
    assert main_in > 1, f"the panel box {box_in} leaves too little room for the main panel"
    ax = fig.add_axes([x0 / W, (y0 + low_in + gap_in) / H, w / W, main_in / H])
    axl = fig.add_axes([x0 / W, y0 / H, w / W, low_in / H], sharex=ax)
    return ax, axl


def break_y(ax, axl, lo, hi, y_ticks=None, xlim=AGE_XLIM, break_in=AGE_BREAK_IN, y_break=None, y_top=None):
    """Finish a panel made by broken_age_axes. The main panel runs from y_break (None:
    AGE_Y_PAD of the data range below lo) to y_top (None: AGE_Y_PAD above hi), with ticks at
    y_ticks (None: chosen automatically); the strip shows 0 at the same units per inch.
    Break marks cross the AGE_LEFT spine at the gap; the right spine runs unbroken from the
    bottom of the strip to the top of the main panel."""
    fig = ax.figure
    W, H = fig.get_size_inches()
    pm, ps = ax.get_position(), axl.get_position()
    main_in, low_in, w_in = pm.height * H, ps.height * H, pm.width * W
    y0 = lo - AGE_Y_PAD * (hi - lo) if y_break is None else y_break
    y1 = hi + AGE_Y_PAD * (hi - lo) if y_top is None else y_top
    assert y0 < lo and y1 > hi, \
        f"the main panel ({y0:.3g} to {y1:.3g}) must hold everything drawn ({lo:.3g} to {hi:.3g})"
    ax.set_ylim(y0, y1)
    ax.yaxis.set_major_locator(FixedLocator(y_ticks) if y_ticks is not None
                               else MaxNLocator(nbins=AGE_Y_NBINS, steps=[1, 2, 2.5, 5, 10]))
    axl.set_ylim(0, (y1 - y0) / main_in * low_in)                 # same units per inch as the main panel
    axl.yaxis.set_major_locator(FixedLocator([0]))
    style_age(ax, xlabel=False, xlim=xlim)
    style_age(axl, xlim=xlim)
    for a, side in ((ax, "bottom"), (ax, "right"), (axl, "top"), (axl, "right")):
        a.spines[side].set_visible(False)
    ax.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
    fig.add_artist(Line2D([pm.x1, pm.x1], [ps.y0, pm.y1], transform=fig.transFigure, color="black",
                          lw=AGE_AXIS_LW, solid_capstyle="projecting"))      # the unbroken right spine
    for a, yb, h_in in ((ax, 0.0, main_in), (axl, 1.0, low_in)):     # break marks on the left spine
        dx, dy = break_in / w_in, break_in / h_in
        a.plot([-dx, dx], [yb - dy, yb + dy], transform=a.transAxes, color="black",
               lw=AGE_AXIS_LW, clip_on=False, zorder=6, solid_capstyle="butt")


def ylabel_beside(fig, axes, text, fontsize=AGE_LABEL_FONT):
    """One y label for a stack of panels, centred on their joint height, as far left of
    the leftmost tick label as matplotlib puts a y label (labelpad 4 pt)."""
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    x0 = min(t.get_window_extent(rend).x0 for a in axes for t in a.get_yticklabels() if t.get_text())
    y_lo, y_hi = min(a.get_position().y0 for a in axes), max(a.get_position().y1 for a in axes)
    lab = fig.text((x0 - 4 * fig.dpi / 72) / fig.bbox.width, (y_lo + y_hi) / 2, text,
                   fontsize=fontsize, rotation=90, ha="right", va="center")
    assert lab.get_window_extent(rend).x0 >= 0, "the y label runs off the figure -- widen AGE_LEFT or FIGSIZE"
    return lab


def r2_under_legend(ax, fit, legend, fmt="R² = {r2:.2f}", gap_pt=AGE_R2_GAP_PT):
    """The fit's R2 in black, left-aligned with the legend's fit icon, gap_pt under it."""
    icon = legend.legend_handles[-1]             # the FitKey box: the last entry
    return ax.annotate(fmt.format(r2=fit.res.rsquared), xy=(0, 0), xycoords=icon,
                       xytext=(0, -gap_pt), textcoords="offset points", ha="left", va="top",
                       fontsize=AGE_LEGEND_FONT, color="black", zorder=6)    # drawn after the legend


def drawn_edges(fit, y, xlim=AGE_XLIM, extend_right=True, cr=None):
    """(x, y, pad_in) samples of the top and of the bottom of everything draw_age draws:
    the line, the band's edges, the points and the CR error bar; pad_in is how far the
    mark reaches beyond (x, y), in inches. Returns (tops, bottoms)."""
    r_in, l_in = AGE_MARKER_SIZE / 2 / 72, AGE_LINE_LW / 2 / 72
    xs_line = np.linspace(xlim[0], xlim[1] if extend_right else INSIDE[-1], 400)
    xs_band = np.concatenate([INSIDE, np.linspace(INSIDE[-1], xlim[1], 100)]) if extend_right else INSIDE
    band = fit.band(xs_band)
    x_pts, y_pts = ours.age_wk.to_numpy(float), np.asarray(y, float)
    tops = [(xs_line, fit.line(xs_line), l_in), (xs_band, band[:, 1], 0.0), (x_pts, y_pts, r_in)]
    bottoms = [(xs_line, fit.line(xs_line), l_in), (xs_band, band[:, 0], 0.0), (x_pts, y_pts, r_in)]
    if cr is not None:
        tops += [([CR_AGE_WK], [cr[2]], AGE_ERR_LW / 2 / 72), ([CR_AGE_WK], [cr[0]], r_in)]
        bottoms += [([CR_AGE_WK], [cr[1]], AGE_ERR_LW / 2 / 72), ([CR_AGE_WK], [cr[0]], r_in)]
    return tops, bottoms


def clear_top(ax, y0, tops, artists, clear_in=AGE_LEGEND_CLEAR_IN):
    """The lowest y top of the main panel (its bottom at y0) at which every sample in
    `tops` (drawn_edges) lying under one of `artists` sits clear_in below it. The
    artists (legends, texts) hang from the top of the panel, so their depth in inches
    does not depend on the y limits. Set the x limits first."""
    fig = ax.figure
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    bb = ax.get_window_extent(rend)
    main_in, x_per_in = bb.height / fig.dpi, np.diff(ax.get_xlim())[0] / (bb.width / fig.dpi)
    top = -np.inf
    for art in artists:
        a = art.get_window_extent(rend)
        depth_in = (bb.y1 - a.y0) / fig.dpi
        x_lo, x_hi = ax.transData.inverted().transform([(a.x0, bb.y0), (a.x1, bb.y0)])[:, 0]
        for x, y, pad_in in tops:
            x, y = np.asarray(x, float), np.asarray(y, float)
            reach = (pad_in + clear_in) * x_per_in                 # marks this close to the side count too
            sel = (x >= x_lo - reach) & (x <= x_hi + reach)
            if sel.any():
                g = (depth_in + clear_in + pad_in) / main_in        # (top - y) / (top - y0) = g
                top = max(top, (y[sel].max() - g * y0) / (1 - g))
    return top


def check_clear(ax, artist, edges, above, clear_in=AGE_LEGEND_CLEAR_IN * 0.99):
    """Nothing drawn (edges: drawn_edges' tops if the artist lies above the data, bottoms
    if below) comes within clear_in of the artist. Run after the limits are final."""
    fig = ax.figure
    fig.canvas.draw()
    a = artist.get_window_extent(fig.canvas.get_renderer())
    for x, y, pad_in in edges:
        pad = (pad_in + clear_in) * fig.dpi
        p = ax.transData.transform(np.column_stack([np.asarray(x, float), np.asarray(y, float)]))
        sel = (p[:, 0] >= a.x0 - pad) & (p[:, 0] <= a.x1 + pad)
        bad = p[sel, 1] + pad > a.y0 if above else p[sel, 1] - pad < a.y1
        assert not bad.any(), "a legend or label touches the data -- change FIGSIZE, Y_BREAK or AGE_LEGEND_CLEAR_IN"


def check_text_apart(fig, axes, extra=()):
    """No two pieces of text overlap (tick labels in view, axis labels, texts, and the
    extra artists such as legends), and all of them lie inside the figure. A framed
    legend counts as its frame; a frameless one as its texts and icons, so R2 can sit
    right under the fit's icon."""
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    items = list(fig.texts)
    for t in extra:
        if isinstance(t, mpl_legend.Legend) and not t.get_frame_on():
            items += list(t.get_texts()) + list(t.legend_handles)
        else:
            items.append(t)
    for a in axes:
        (x_lo, x_hi), (y_lo, y_hi) = a.get_xlim(), a.get_ylim()
        items += [t for t in a.get_xticklabels() if x_lo <= t.get_position()[0] <= x_hi]
        items += [t for t in a.get_yticklabels() if y_lo <= t.get_position()[1] <= y_hi]
        items += a.texts + [a.xaxis.label, a.yaxis.label]
    seen, boxes = set(), []
    for t in items:
        if id(t) in seen or not t.get_visible() or not getattr(t, "get_text", lambda: "x")():
            continue
        seen.add(id(t))
        boxes.append((getattr(t, "get_text", lambda: type(t).__name__)(), t.get_window_extent(rend)))
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            assert not boxes[i][1].overlaps(boxes[j][1]), \
                f"text overlaps: {boxes[i][0]!r} and {boxes[j][0]!r} -- change FIGSIZE or the knobs"
    for n, b in boxes:
        assert b.x0 >= 0 and b.y0 >= 0 and b.x1 <= fig.bbox.x1 and b.y1 <= fig.bbox.y1, \
            f"{n!r} runs off the figure -- widen the margins or FIGSIZE"


def age_figure(y, cr, ylabel, fit_label, unit=1.0, figsize=None):
    """One panel: ours, the fit and the CR point, y in units of `unit`; y spans what is
    drawn (AGE_Y_PAD beyond each end); the fit's legend in AGE_FIT_LOC. Returns fig, ax, fit."""
    fit = Fit(ours.age_wk, y)
    fig, ax = age_panel(figsize or FIGSIZE)
    lo, hi, _ = draw_age(ax, fit, y, cr, unit)
    ax.set_ylim(lo - AGE_Y_PAD * (hi - lo), hi + AGE_Y_PAD * (hi - lo))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=AGE_Y_NBINS, steps=[1, 2, 2.5, 5, 10]))
    style_age(ax, ylabel)
    fit_legend(ax, fit, fit_label, unit)
    check_inside(fig, ax)
    return fig, ax, fit


def report_fit(fit, unit, unit_name, scale_yr=True):
    k = WEEKS_PER_YEAR if scale_yr else 1.0
    s, lo, hi = fit.b1 * k / unit, fit.ci[1][0] * k / unit, fit.ci[1][1] * k / unit
    at_cr, band_cr = fit.line(CR_AGE_WK) / unit, fit.band([CR_AGE_WK])[0] / unit
    print(f"   fit: {s:.3g} [{lo:.3g}, {hi:.3g}] {unit_name} per year, intercept {fit.b0 / unit:.3g}, "
          f"R2 {fit.res.rsquared:.3f}, slope P {fit.p:.3g}; at {CR_AGE_WK} wk "
          f"{at_cr:.3g} [{band_cr[0]:.3g}, {band_cr[1]:.3g}]")

# Supplementary tables

Each cell writes a TSV to `TABLE_DIR`, formatted for pasting into a spreadsheet (Supplementary Table 7 also as an xlsx). Supplementary Table 1 (PacBio tags) has no code.

## Supplementary Table 2. Sequencing summary for each library

One row per library: HiFi strand consensus reads (forward and reverse counted separately), molecules with both strands (% of the ZMWs with at least one HiFi read), yield, and read length and passes per strand consensus read.

In [ ]:
# =============================================================================
# Supplementary Table 2. Sequencing summary for each library
#   Input   read_tables/<library>.reads.npz; the read groups of alignment/<library>.sorted.bam (run date)
#   Output  TABLE_DIR/sequencing_summary.tsv
# =============================================================================
SAVE_TSV = TABLE_DIR / "sequencing_summary.tsv"

# Number format of each column
FORMATS = {
    "HiFi reads":                 "{:,.0f}",
    "Forward-strand reads":       "{:,.0f}",
    "Reverse-strand reads":       "{:,.0f}",
    "Molecules (both strands)":   "{:,.0f}",
    "Molecules (% of ZMWs)":      "{:.1f}",
    "HiFi yield (Gb)":            "{:.1f}",
    "Mean read length (b)":       "{:,.0f}",
    "Median read length (b)":     "{:,.0f}",
    "Mean passes":                "{:.1f}",
    "Median passes":              "{:g}",
}

rows = []
for lib in LIBRARIES:
    r   = load_reads(lib, ("zmw", "strand", "np", "length"))
    fwd = r["strand"] == 0

    # Sort reads by (ZMW, strand): the two reads of a molecule end up next to each other
    key = np.sort(r["zmw"] * 2 + r["strand"])
    assert not (np.diff(key) == 0).any(), f"{lib}: more than one read per ZMW and strand"
    zmw   = key >> 1
    n_mol = int((zmw[1:] == zmw[:-1]).sum())       # ZMWs with both a forward and a reverse read
    n_zmw = len(zmw) - n_mol                       # ZMWs with at least one read

    rows.append({
        "Library":                    LABELS[lib],
        "Age (weeks)":                AGE_WK[lib],
        "Sequencing run":             run_date(lib),
        "HiFi reads":                 len(fwd),
        "Forward-strand reads":       int(fwd.sum()),
        "Reverse-strand reads":       int((~fwd).sum()),
        "Molecules (both strands)":   n_mol,
        "Molecules (% of ZMWs)":      100 * n_mol / n_zmw,
        "HiFi yield (Gb)":            r["length"].sum() / 1e9,
        "Mean read length (b)":       r["length"].mean(),
        "Median read length (b)":     np.median(r["length"]),
        "Mean passes":                r["np"].mean(),
        "Median passes":              np.median(r["np"]),
    })
    del r

summary = pd.DataFrame(rows)

table = summary.copy()
for col, fmt in FORMATS.items():
    table[col] = table[col].map(fmt.format)

table.to_csv(SAVE_TSV, sep="\t", index=False)
print(f"saved {SAVE_TSV}")
display(table)

## Supplementary Table 3. Read filters for each library

The read and molecule filters applied before variant calling, in their order, with the library thresholds (Methods, Read filters, Molecule filters); each row gives what remains after that filter and every filter above it in its block.

In [ ]:
# =============================================================================
# Supplementary Table 3. Read filters for each library; columns are libraries
#   Thresholds   filter_thresholds/: read length bounds (mean ± 2 s.d.), maximum forward–reverse length
#                difference and maximum strand alignment non-overlap (99th percentiles)
#   Input        read_tables/<library>.reads.npz; filter_thresholds/
#   Output       TABLE_DIR/read_filters.tsv
# =============================================================================
MAPQ_MIN     = 60          # minimum mapping quality
NP_MIN       = 5           # minimum number of passes
MAX_DIV      = 0.01        # maximum divergence from the reference
PCT_DECIMALS = 1
SAVE_TSV     = TABLE_DIR / "read_filters.tsv"

# Block headers (rows with empty cells)
HEAD_THRESHOLDS = "Thresholds"
HEAD_READS      = "Strand consensus reads remaining (% of HiFi reads, Table S2)"
HEAD_MOLECULES  = "Molecules remaining (% of molecules with both strands, Table S2)"


def count_pct(n, total):
    """'count (pct%)'; a percentage below 100 that would round to 100 gets two decimals."""
    pct = 100 * n / total
    txt = f"{pct:.{PCT_DECIMALS}f}"
    if n < total and float(txt) >= 100:
        txt = f"{pct:.2f}"
    return f"{int(n):,} ({txt}%)"


columns = {}
for lib in LIBRARIES:
    r = load_reads(lib, ("zmw", "strand", "np", "length", "mapped", "mapq",
                         "chrom", "n_eq", "aln_len", "start", "end", "sa"))
    lo, hi    = length_bounds(lib)
    max_dlen  = lendiv_cutoff(lib)
    max_nonov = nonov_cutoff(lib)
    n_reads  = len(r["zmw"])

    # Read filters, each applied on top of the ones before it
    aligned  = r["mapped"]
    autosome = aligned  & (r["chrom"] > 0)
    mapq_ok  = autosome & (r["mapq"] >= MAPQ_MIN)
    np_ok    = mapq_ok  & (r["np"] >= NP_MIN)
    len_ok   = np_ok    & (r["length"] >= lo) & (r["length"] <= hi)
    div      = np.where(r["aln_len"] > 0, 1 - r["n_eq"] / np.maximum(r["aln_len"], 1), 0.0)
    read_ok  = len_ok   & (div <= MAX_DIV)

    # Molecules: sort reads by (ZMW, strand) so a ZMW's forward and reverse reads are adjacent
    order = np.argsort(r["zmw"] * 2 + r["strand"], kind="stable")
    zmw   = r["zmw"][order]
    is_pair = zmw[1:] == zmw[:-1]
    f, v  = order[:-1][is_pair], order[1:][is_pair]      # forward and reverse read of each molecule
    n_mol = len(f)

    both_ok = read_ok[f] & read_ok[v] & (r["chrom"][f] == r["chrom"][v])
    dlen    = np.abs(r["length"][f].astype(np.int64) - r["length"][v])
    dlen_ok = both_ok & (dlen <= max_dlen)
    not_chimeric = dlen_ok & ~r["sa"][f] & ~r["sa"][v]

    # bases covered by only one of the two strand alignments
    sf, ef = r["start"][f].astype(np.int64), r["end"][f].astype(np.int64)
    sv, ev = r["start"][v].astype(np.int64), r["end"][v].astype(np.int64)
    nonov  = (np.maximum(ef, ev) - np.minimum(sf, sv)) - np.maximum(0, np.minimum(ef, ev) - np.maximum(sf, sv))
    used   = not_chimeric & (nonov <= max_nonov)

    columns[LABELS[lib]] = {
        HEAD_THRESHOLDS:                                    "",
        "Read length bounds (bases)":                       f"{lo:,}–{hi:,}",
        "Maximum forward–reverse length difference (bases)": f"{max_dlen:,}",
        "Maximum strand alignment non-overlap (bases)":      f"{max_nonov:,}",
        HEAD_READS:                                         "",
        "Primary alignment":                                count_pct(aligned.sum(), n_reads),
        "On a T2T autosome":                                count_pct(autosome.sum(), n_reads),
        f"Mapping quality ≥ {MAPQ_MIN}":                    count_pct(mapq_ok.sum(), n_reads),
        f"≥ {NP_MIN} passes":                               count_pct(np_ok.sum(), n_reads),
        "Read length within bounds":                        count_pct(len_ok.sum(), n_reads),
        f"Divergence ≤ {100 * MAX_DIV:g}%":                 count_pct(read_ok.sum(), n_reads),
        HEAD_MOLECULES:                                     "",
        "Both strands passed the read filters":             count_pct(both_ok.sum(), n_mol),
        "Forward–reverse length difference within maximum": count_pct(dlen_ok.sum(), n_mol),
        "No supplementary alignment on either strand":     count_pct(not_chimeric.sum(), n_mol),
        "Strand alignment non-overlap within maximum (used for calling)": count_pct(used.sum(), n_mol),
    }
    del r

filters = pd.DataFrame(columns)
filters.index.name = "Filter"

filters.to_csv(SAVE_TSV, sep="\t")
print(f"saved {SAVE_TSV}")
display(filters)

## Supplementary Table 4. Callable sites, post-calling variant filters and variant sets for each library

Columns are libraries: the sites covered and callable (Methods, Coverage filter), the callable base pairs, the molecules removed for carrying more than `NOISY_CUTOFF` singletons, and the numbers of somatic and germline mutation candidates (`final_variants()`, setup 0a).

In [ ]:
# =============================================================================
# Supplementary Table 4. Callable sites, post-calling variant filters and variant sets for each library
#   Sites covered by >= 1 molecule   untrimmed coverage >= 1, % of the T2T autosomes; their mean coverage
#   Callable sites                   unmasked, untrimmed coverage MIN_COV..the library's 99th percentile,
#                                    >= 1 trimmed molecule; callable base pairs = their trimmed coverage
#   Molecules with >= 3 singletons   molecules carrying more than NOISY_CUTOFF singletons
#                                    (SINGLETONS_ON_MOLECULE of the trimmed mutation table)
#   Variant sets                     final_variants() (setup 0a)
# Checks: the covered sites, callable sites and callable base pairs against callable_bps/trimmed/, the
# somatic mutation candidates against somatic_mutation_candidates.tsv.
#   Input   callable_bps/trimmed/ (denominator.tsv, coverage_tables/), variant_calling/ext0/<library>/
#           informative_coverage_dist.tsv, the trimmed mutation table, somatic_mutation_candidates/
#   Output  TABLE_DIR/post_calling_filters.tsv
# =============================================================================
MIN_COV                = 10      # minimum untrimmed coverage of the coverage filter
NOISY_CUTOFF           = 2       # maximum singletons per molecule
GERMLINE_MIN_MOLECULES = 2       # germline candidates: at least this many molecules carrying the non-reference allele
PCT_DECIMALS           = 1
SAVE_TSV               = TABLE_DIR / "post_calling_filters.tsv"

COV_TABLES     = CB_DIR / "coverage_tables"                                  # <library>.npz
DENOM_TSV      = CB_DIR / "denominator.tsv"
UNTRIMMED_DIST = VC_DIR / "ext0"                                             # <library>/informative_coverage_dist.tsv
CANDIDATES_TSV = CAND_DIR / "somatic_mutation_candidates.tsv"
CALLS_TSV      = VARIANTS_TSV
AUTOSOME_BP    = int(pd.read_csv(CAND_DIR / "t2t_autosome_lengths.tsv", sep="\t")
                       .set_index("chrom").loc["autosomes", "length_bp"])

# Block headers (rows with empty cells)
HEAD_CALLABLE = "Callable sites"
HEAD_POST     = "Post-calling variant filters"
HEAD_SETS     = "Variant sets"

den = pd.read_csv(DENOM_TSV, sep="\t").query("floor == @MIN_COV").set_index("sample")


def covered_sites(lib):
    """Autosomal sites covered by at least one molecule in the untrimmed reads, and their mean
    untrimmed coverage."""
    d = pd.read_csv(UNTRIMMED_DIST / lib / "informative_coverage_dist.tsv", sep="\t").query("coverage >= 1")
    n = int(d.n_sites.sum())
    h = np.load(COV_TABLES / f"{lib}.npz")["hist"]                        # [mask, context, trimmed, untrimmed]
    assert n == int(h[..., 1:].astype(np.int64).sum()), \
        f"{lib}: sites covered by at least one molecule differ between the coverage distribution and coverage table"
    return n, (d.coverage * d.n_sites).sum() / n


def callable_window(lib):
    """Untrimmed cap, callable sites and callable base pairs: unmasked sites with untrimmed coverage
    MIN_COV..cap and at least one trimmed molecule; base pairs are their trimmed coverage."""
    h = np.load(COV_TABLES / f"{lib}.npz")["hist"][0].astype(np.int64).sum(axis=0)   # [trimmed, untrimmed coverage]
    cov = np.arange(h.shape[0])
    cap = int(den.loc[lib, "untrimmed_p99"])
    sub = h[1:, (cov >= MIN_COV) & (cov <= cap)]
    sites, base_pairs = int(sub.sum()), int((cov[1:, None] * sub).sum())
    assert (sites, base_pairs) == (int(den.loc[lib, "callable_sites"]), int(den.loc[lib, "callable_depth"])), \
        f"{lib}: callable sites or base pairs differ from {DENOM_TSV.name}"
    return cap, sites, base_pairs


def pct(n, total):
    return f"{100 * n / total:.{PCT_DECIMALS}f}%"


# Singletons per molecule: one value per molecule carrying the non-reference allele in the mutation table
calls = pd.read_csv(CALLS_TSV, sep="\t", keep_default_na=False, dtype=str,
                    usecols=["LIBRARY", "ZMW", "SINGLETONS_ON_MOLECULE"])
per_molecule = (calls.assign(ZMW=calls.ZMW.str.split(","), SPM=calls.SINGLETONS_ON_MOLECULE.str.split(","))
                .explode(["ZMW", "SPM"]).astype({"SPM": int})
                .groupby(["LIBRARY", "ZMW"]).SPM.max())
noisy = per_molecule[per_molecule > NOISY_CUTOFF].groupby(level="LIBRARY").size()

candidates = pd.read_csv(CANDIDATES_TSV, sep="\t")
final_8 = final_variants(NOISY_CUTOFF, MIN_COV, GERMLINE_MIN_MOLECULES)
n_final = final_8.groupby(["TYPE", "LIBRARY"]).size()
n_germline_variants = len(final_8[final_8.TYPE == "Germline candidate"]
                          .drop_duplicates(["ANIMAL", "CHROM", "POS", "ALT"]))   # the 60 wk mouse's variants once

columns = {}
for lib in LIBRARIES:
    cap, sites, base_pairs = callable_window(lib)
    covered, mean_cov = covered_sites(lib)
    somatic = int(n_final.get(("Somatic SNV", lib), 0))
    assert somatic == int(((candidates.LIBRARY == lib) & (candidates.SINGLETONS_ON_MOLECULE <= NOISY_CUTOFF)).sum()), \
        f"{lib}: somatic SNVs differ from {CANDIDATES_TSV.name}"
    columns[LABELS[lib]] = {
        HEAD_CALLABLE:                                     "",
        "Sites covered by ≥ 1 molecule (% of autosomes)":  pct(covered, AUTOSOME_BP),
        "Mean untrimmed coverage (molecules)":             f"{mean_cov:.1f}",
        "Minimum–maximum untrimmed coverage (molecules)":  f"{MIN_COV}–{cap}",
        "Callable sites (% of autosomes)":                 pct(sites, AUTOSOME_BP),
        "Callable base pairs (×10⁹)":                      f"{base_pairs / 1e9:.2f}",
        HEAD_POST:                                         "",
        f"Molecules with ≥ {NOISY_CUTOFF + 1} singletons":  f"{int(noisy.get(lib, 0)):,}",
        HEAD_SETS:                                         "",
        "Somatic mutation candidates":                     f"{somatic:,}",
        "Germline mutation candidates":                    f"{int(n_final.get(('Germline candidate', lib), 0)):,}",
    }

table = pd.DataFrame(columns)
table.index.name = "Quantity"
table.to_csv(SAVE_TSV, sep="\t")
print(f"saved {SAVE_TSV}")
print(f"molecules with ≥ {NOISY_CUTOFF + 1} singletons: {int(noisy.sum()):,}; "
      f"somatic mutation candidates: {int(n_final['Somatic SNV'].sum()):,}; "
      f"germline mutation candidates: {int(n_final['Germline candidate'].sum()):,} over the libraries, "
      f"{n_germline_variants:,} "
      f"variants (the 60-week-old mouse's variants found in both its libraries counted once)")
display(table)

## Supplementary Table 5. Strand-consensus mismatch rates by substitution type and number of passes

For each substitution with a pyrimidine changed on the read and its reverse complement: mismatches, opportunities and their rate, by the number of passes of the divergent read (Methods, Estimation of the sequencing error rate).

In [ ]:
# =============================================================================
# Supplementary Table 5. Strand-consensus mismatch rates — 192 substitution types × number of passes
# =============================================================================
# One row per substitution with a pyrimidine changed on the read and class of the
# number of passes (np) of the divergent read. For the substitution as the divergent
# strand consensus read shows it (5'→3') and for its reverse complement (the same
# base-pair change as a read of the opposite strand shows it): the mismatches, the
# opportunities (both strands passed the base filters; each counted once per
# strand) at which the read shows the same trinucleotide, and their ratio. Last
# column: the product of the two rates, the rate per bp per molecule at which both
# strands carry the change if they err independently. Trimmed reads, unmasked sites
# away from mutations of the same animal; all libraries. The block "All" pools the
# classes.
#
# Checks: the mismatches equal those of the single-strand call tables with the
# same site filters, channel by channel after folding to the 96 pyrimidine
# channels; each class's opportunities equal the opportunities of its np range;
# over all classes, a trinucleotide and its reverse complement have the same
# opportunities.
#
# Input   strand_mismatches/ext450/<library>.counts.npz; call_tables/ext450/<library>/
#         (single_strand_calls.tsv.gz, mutations_with_vaf.tsv); masked_sites/excluded_regions.bed
# Output  TABLE_DIR/strand_mismatch_rates.tsv
from numpy import format_float_positional

COUNTS_DIR = SM_DIR / "ext450"                                              # <library>.counts.npz
CALLS_DIR  = CALL_DIR / "ext450"                                            # <library>/single_strand_calls.tsv.gz
MASK_BED   = MASK_DIR / "excluded_regions.bed"
STRATUM     = 0             # unmasked sites, no mutation of the same animal
RATE_UNIT   = 1e-6          # per-strand rates in multiples of this
BOTH_UNIT   = 1e-12         # both-strand rates in multiples of this
RATE_DIGITS = 3             # significant digits of the rates
SAVE_TSV    = TABLE_DIR / "strand_mismatch_rates.tsv"

ACGT  = "ACGT"
COMPL = str.maketrans(ACGT, "TGCA")
rc    = lambda s: s.translate(COMPL)[::-1]
tri_i = lambda s: 16 * ACGT.index(s[0]) + 4 * ACGT.index(s[1]) + ACGT.index(s[2])
PYR_SUBS = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]
PUR_SUBS = [s.translate(COMPL) for s in PYR_SUBS]                         # G>T, G>C, G>A, A>T, A>G, A>C
TYPES = [f"{a}[{s}]{b}" for subs in (PYR_SUBS, PUR_SUBS) for s in subs for a in ACGT for b in ACGT]
rev_compl = lambda c: f"{c[6].translate(COMPL)}[{c[2].translate(COMPL)}>{c[4].translate(COMPL)}]{c[0].translate(COMPL)}"
fold96 = lambda tri, ref, alt: (f"{tri[0]}[{ref}>{alt}]{tri[2]}" if ref in "CT"
                                else f"{rc(tri)[0]}[{ref.translate(COMPL)}>{alt.translate(COMPL)}]{rc(tri)[2]}")

# counts of strand_mismatch_counts.py (pipeline)
C_bin, C_np, recs = 0, 0, []
for lib in LIBRARIES:
    with np.load(COUNTS_DIR / f"{lib}.counts.npz") as z:
        C_bin = C_bin + z["context"][:, STRATUM].sum(axis=(2, 3, 4))       # (np class, 64 read trinucleotides)
        C_np  = C_np + z["compared"][:, STRATUM].sum(axis=(0, 2, 3))       # (np,) compared strand-bases
        edges = z["context_np_edges"]
        keep = z["mm_stratum"] == STRATUM
        recs.append(pd.DataFrame({k: z[f"mm_{k}"][keep] for k in
                                  ("read_trinuc", "err_base", "err_reverse", "err_np", "triplet", "ref")}))
mm = pd.concat(recs, ignore_index=True)
CLASSES = [f"{e}" if (k + 1 < len(edges) and edges[k + 1] == e + 1) else
           (f"{e}–{edges[k + 1] - 1}" if k + 1 < len(edges) else f"≥{e}") for k, e in enumerate(edges)]
cls = np.maximum(np.searchsorted(edges, mm.err_np.to_numpy(), side="right") - 1, 0)
alt_read = np.where(mm.err_reverse, mm.err_base.str.translate(COMPL), mm.err_base)
N_bin = np.zeros(C_bin.shape + (4,), dtype=np.int64)
np.add.at(N_bin, (cls, mm.read_trinuc.to_numpy(), [ACGT.index(a) for a in alt_read]), 1)

# check 1: mismatches against the single-strand call tables with the same site filters
mask = pd.read_csv(MASK_BED, sep="\t", header=None, usecols=[0, 1, 2], names=["chrom", "start", "end"])
animal = lambda lib: re.sub(r"_\d+kb$", "", lib)
tab = []
for lib in LIBRARIES:
    d = pd.read_csv(CALLS_DIR / lib / "single_strand_calls.tsv.gz", sep="\t",
                    usecols=["chrom", "pos_1based", "triplet", "ref_base", "err_base"])
    tab.append(d.assign(animal=animal(lib)))
tab = pd.concat(tab, ignore_index=True)
muts = pd.concat([pd.read_csv(CALLS_DIR / lib / "mutations_with_vaf.tsv", sep="\t", usecols=["chrom", "pos_1based"])
                  .assign(animal=animal(lib)) for lib in LIBRARIES]).drop_duplicates()
in_mask = np.zeros(len(tab), bool)
for c, g in mask.groupby("chrom"):
    i = np.flatnonzero(tab.chrom.to_numpy() == c)
    p0 = tab.pos_1based.to_numpy()[i] - 1
    k = np.searchsorted(g.start.to_numpy(), p0, side="right") - 1
    in_mask[i] = (k >= 0) & (p0 < g.end.to_numpy()[np.maximum(k, 0)])
at_mut = pd.MultiIndex.from_frame(tab[["animal", "chrom", "pos_1based"]]).isin(
    pd.MultiIndex.from_frame(muts[["animal", "chrom", "pos_1based"]]))
tab = tab[~in_mask & ~at_mut]
from_tables = pd.Series([fold96(t, r, a) for t, r, a in zip(tab.triplet, tab.ref_base, tab.err_base)]).value_counts()
from_counts = pd.Series([fold96(t, r, a) for t, r, a in zip(mm.triplet, mm.ref, mm.err_base)]).value_counts()
assert len(tab) == len(mm) == N_bin.sum(), (len(tab), len(mm), N_bin.sum())
assert from_tables.sort_index().equals(from_counts.sort_index()), "mismatches differ from the call tables"

# check 2: each class's opportunities equal those of its np range
upper = list(edges[1:]) + [len(C_np)]
assert all(C_bin[k].sum() == C_np[edges[k]:upper[k]].sum() for k in range(len(edges))), "opportunities differ"

# check 3: over all classes, a trinucleotide and its reverse complement have the same opportunities
rc_idx = np.array([tri_i(rc(a + b + c)) for a in ACGT for b in ACGT for c in ACGT])
C_all = C_bin.sum(0)
asym = (np.abs(C_all - C_all[rc_idx]) / C_all).max()
assert asym < 1e-4, f"opportunities not strand-symmetric ({asym:.1e})"

# table
sup = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")
unit = lambda u: f"(×10{str(int(round(np.log10(u)))).translate(sup)})"
fmt = lambda x, u: "0" if x == 0 else format_float_positional(x / u, precision=RATE_DIGITS, unique=False,
                                                              fractional=False, trim="-")
PYR_TYPES = TYPES[:96]
blocks = [("All", C_bin.sum(0), N_bin.sum(0))] + [(CLASSES[k], C_bin[k], N_bin[k]) for k in range(len(edges))]
rows = []
for label, C, N in blocks:
    for t_ in PYR_TYPES:
        p = rev_compl(t_)
        (ti, ai), (pi, qi) = [(tri_i(s[0] + s[2] + s[6]), ACGT.index(s[4])) for s in (t_, p)]
        r_a, r_b = N[ti, ai] / C[ti], N[pi, qi] / C[pi]
        rows.append({"Number of passes": label,
                     "Substitution (pyrimidine changed, as read 5′→3′)": t_,
                     "Mismatches": f"{int(N[ti, ai]):,}", "Opportunities": f"{int(C[ti]):,}",
                     f"Rate {unit(RATE_UNIT)}": fmt(r_a, RATE_UNIT),
                     "Reverse complement (purine changed, as read 5′→3′)": p,
                     "Reverse complement mismatches": f"{int(N[pi, qi]):,}",
                     "Reverse complement opportunities": f"{int(C[pi]):,}",
                     f"Reverse complement rate {unit(RATE_UNIT)}": fmt(r_b, RATE_UNIT),
                     f"Both strands, per bp per molecule {unit(BOTH_UNIT)}": fmt(r_a * r_b, BOTH_UNIT)})
table = pd.DataFrame(rows)
table.to_csv(SAVE_TSV, sep="\t", index=False)

rate = lambda t: N_bin.sum(0)[tri_i(t[0] + t[2] + t[6]), ACGT.index(t[4])] / C_bin.sum(0)[tri_i(t[0] + t[2] + t[6])]
print(f"saved {SAVE_TSV}: {len(table):,} rows ({len(blocks)} blocks × {len(PYR_TYPES)} substitutions and their reverse complements)")
print(f"mismatches {int(N_bin.sum()):,} (= the call tables, channel by channel); "
      f"opportunities {int(C_bin.sum()):,}; largest strand asymmetry of opportunities {asym:.1e}")
print(f"e.g. C[C>A]A {rate('C[C>A]A'):.2e}, its reverse complement T[G>T]G {rate('T[G>T]G'):.2e} per base per strand")
display(table.head(8))

## Supplementary Table 6. Callable base pairs by trinucleotide context

The callable base pairs of each library and the sites of the T2T autosomes, whole and unmasked, in each of the 32 pyrimidine-centred trinucleotides, as counts and as percentages of each column's total.

In [ ]:
# =============================================================================
# Supplementary Table 6. Callable base pairs by trinucleotide context — each library and the T2T autosomes
# =============================================================================
# One row per pyrimidine-centred trinucleotide (each counted together with its
# reverse complement): the callable base pairs of each library (trimmed bases at
# sites that pass the coverage filter, unmasked; the denominator of the somatic
# mutation rate) and the sites of the T2T autosomes, whole and unmasked; then the
# same as percentages of each column's total (the share of the library's callable
# base pairs, or of the genome's sites, in that context). Last row: totals.
#
# Checks: each library's total equals its callable base pairs in
# callable_bps/trimmed/denominator.tsv; each library's column is recomputed from
# its coverage table (callable_bps/trimmed/coverage_tables) and must match exactly.
#
# Input   somatic_mutation_candidates/ (context_opportunity.tsv, t2t_reference_context_counts.tsv,
#         t2t_autosome_lengths.tsv); callable_bps/trimmed/
# Output  TABLE_DIR/trinucleotide_opportunity.tsv
import sys
sys.path.insert(0, SCRIPTS)
from callable_bps_untrimmed import fold_triplet, all_triplets

CONTEXT_TSV = CAND_DIR / "context_opportunity.tsv"
REF_TSV     = CAND_DIR / "t2t_reference_context_counts.tsv"
LENGTHS_TSV = CAND_DIR / "t2t_autosome_lengths.tsv"
DENOM_TSV   = CB_DIR / "denominator.tsv"
COV_TABLES  = CB_DIR / "coverage_tables"                                    # <library>.npz
MIN_COV     = 10            # minimum untrimmed coverage of the coverage filter
PCT_DECIMALS = 3            # decimals of the percentages
SAVE_TSV    = TABLE_DIR / "trinucleotide_opportunity.tsv"

ROW_ORDER = sorted(all_triplets(fold=True), key=lambda t: (t[1], t[0], t[2]))   # C-centred, then T-centred
REF_COLS  = {"autosomes": "T2T autosomes", "outside_mask": "T2T autosomes outside the mask"}
LIB_UNIT, REF_UNIT = "callable bp", "sites"

opp = pd.read_csv(CONTEXT_TSV, sep="\t")
assert set(opp.floor) == {MIN_COV}, f"{CONTEXT_TSV.name} holds minimum coverage {sorted(set(opp.floor))}"
opp["context"] = opp.triplet.map(fold_triplet)
lib_cols = opp.pivot_table(index="context", columns="sample", values="callable_depth", aggfunc="sum")
ref = pd.read_csv(REF_TSV, sep="\t")
ref["context"] = ref.triplet.map(fold_triplet)
ref_cols = ref.groupby("context")[list(REF_COLS)].sum()

# check 1: totals against the denominator of the somatic mutation rate
den = pd.read_csv(DENOM_TSV, sep="\t").query("floor == @MIN_COV").set_index("sample")
for lib in LIBRARIES:
    assert lib_cols[lib].sum() == den.loc[lib, "callable_depth"], f"{lib}: total differs from {DENOM_TSV.name}"

# check 2: each library's column recomputed from its coverage table
CONTEXTS = all_triplets(fold=False) + ["non-ACGT"]                         # the coverage tables' context axis
for lib in LIBRARIES:
    h = np.load(COV_TABLES / f"{lib}.npz")["hist"][0].astype(np.int64)    # unmasked sites: [context, trimmed, untrimmed]
    cov = np.arange(h.shape[1])
    in_window = (cov >= MIN_COV) & (cov <= den.loc[lib, "untrimmed_p99"])
    depth = (cov[None, 1:, None] * h[:, 1:, :][:, :, in_window]).sum(axis=(1, 2))
    assert depth[-1] == 0, f"{lib}: callable bases at non-ACGT contexts"
    recomputed = pd.Series(depth[:-1], index=CONTEXTS[:-1]).groupby(fold_triplet).sum()
    assert recomputed.reindex(ROW_ORDER).equals(lib_cols[lib].reindex(ROW_ORDER).astype(np.int64)), \
        f"{lib}: context table differs from the coverage table"

# table: counts, then each column as a percentage of its total
counts = pd.DataFrame({LABELS[lib]: lib_cols[lib].reindex(ROW_ORDER) for lib in LIBRARIES})
for c, name in REF_COLS.items():
    counts[name] = ref_cols[c].reindex(ROW_ORDER)
counts = counts.astype(np.int64)
shares = 100 * counts / counts.sum()
assert np.allclose(shares.sum(), 100), "percentages do not sum to 100"
unit = lambda col: REF_UNIT if col in REF_COLS.values() else LIB_UNIT
table = pd.concat([counts.map(lambda v: f"{v:,}").rename(columns=lambda c: f"{c}: {unit(c)}"),
                   shares.map(lambda v: f"{v:.{PCT_DECIMALS}f}%").rename(columns=lambda c: f"{c}: % of {unit(c)}")],
                  axis=1)
table.loc["Total"] = ([f"{v:,}" for v in counts.sum()] + [f"{100:.{PCT_DECIMALS}f}%"] * counts.shape[1])
table.index.name = "Trinucleotide (pyrimidine-centred)"
table.to_csv(SAVE_TSV, sep="\t")

lengths = pd.read_csv(LENGTHS_TSV, sep="\t").set_index("chrom").loc["autosomes"]
print(f"saved {SAVE_TSV}: {len(ROW_ORDER)} contexts + total")
print("library totals = denominator.tsv, and every column = its coverage table")
print(f"T2T autosomes: {int(ref_cols.autosomes.sum()):,} sites with a full ACGT triplet "
      f"(ACGT bases {int(lengths.acgt_bp):,}); outside the mask {int(ref_cols.outside_mask.sum()):,} "
      f"(ACGT bases {int(lengths.outside_mask_acgt_bp):,})")
display(table)

## Supplementary Table 7. Variant table

One row per variant: the somatic mutations, the germline or early embryonic mutations (each variant once) and the non-reference alleles shared among animals, with the molecules carrying the non-reference allele and the coverage in each library, in the untrimmed reads; "Yes" marks the sites of the tree (Supplementary Fig. 15).

In [ ]:
# =============================================================================
# Supplementary Table 7. Variant table — somatic, germline or early embryonic and shared variants, with
#   the number of molecules carrying the non-reference allele and the coverage in each library
#   (untrimmed reads)
#   Somatic                      the somatic mutations of final_variants() (setup 0a)
#   Germline or early embryonic  the germline candidates of final_variants(), each variant once
#   Shared among animals         non-reference alleles called in two or more animals, outside the
#                                RepeatMasker mask, callable where SHARED_CALLABLE says
#   Molecule columns are blank where the library is not callable at the site (untrimmed coverage
#   outside MIN_COV..the library's untrimmed p99); coverage columns are always given.
#   Input   animal_relationships/nj_sites.tsv; callable_bps/trimmed/denominator.tsv; the mutation tables
#   Output  TABLE_DIR/variant_table.xlsx (two header rows, group names merged over their libraries,
#           panes frozen) and TABLE_DIR/variant_table.tsv (the same two header rows)
# =============================================================================
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, Side

# ── KNOBS ────────────────────────────────────────────────────────────────────
MIN_COV                = 10      # coverage filter: untrimmed coverage from MIN_COV to the library's untrimmed p99
NOISY_CUTOFF           = 2       # somatic and germline sets: molecules with more singletons do not count
GERMLINE_MIN_MOLECULES = 2       # germline or early embryonic: at least this many molecules of one animal
SHARED_CALLABLE        = "carrier"   # shared variants kept where callable in: "carrier" = at least one library
                                     # carrying them; "any" = any library; "all" = every library
CLASS_LABELS = {"Somatic SNV": "Somatic", "Germline candidate": "Germline or early embryonic",
                "Shared": "Shared among animals"}
ID_COLUMNS   = [("CHROM", "Chromosome"), ("POS", "Position"), ("REF", "Reference"), ("ALT", "Alternative"),
                ("TRINUCLEOTIDE_CONTEXT", "Trinucleotide context"), ("CLASS", "Class"), ("TREE", "In the tree")]
GROUPS       = [("VARIANT_MOLECULES", "Molecules carrying the non-reference allele"), ("COVERAGE", "Coverage")]
SAVE_XLSX    = TABLE_DIR / "variant_table.xlsx"
SAVE_TSV     = TABLE_DIR / "variant_table.tsv"     # same two header rows, for pasting

# ── THE THREE CLASSES ────────────────────────────────────────────────────────
KEY = ["CHROM", "POS", "ALT"]
nj  = pd.read_csv(NJ_DIR / "nj_sites.tsv", sep="\t")     # every site called in any library, untrimmed reads
p99 = (pd.read_csv(CB_DIR / "denominator.tsv", sep="\t")
         .drop_duplicates("sample").set_index("sample").untrimmed_p99)
callable_ = pd.DataFrame({l: nj[f"{l}_COVERAGE"].between(MIN_COV, p99[l]) for l in LIBRARIES})
carries   = pd.DataFrame({l: nj[f"{l}_VARIANT_MOLECULES"] > 0 for l in LIBRARIES})
n_animals = pd.DataFrame({a: carries[libs].any(axis=1) for a, libs in ANIMALS.items()}).sum(axis=1)
where = {"carrier": (callable_ & carries).any(axis=1), "any": callable_.any(axis=1),
         "all": callable_.all(axis=1)}[SHARED_CALLABLE]
shared = nj.loc[(nj.IN_REPEATMASKER_MASK == 0) & (n_animals >= 2) & where, KEY].assign(TYPE="Shared")

fv = final_variants(NOISY_CUTOFF, MIN_COV, GERMLINE_MIN_MOLECULES)        # one row per library and site
classes = pd.concat([fv[KEY + ["TYPE"]].drop_duplicates(KEY), shared])     # one row per variant
assert not classes.duplicated(KEY).any(), "a variant falls in two classes"
tree_sites, _ = animal_genotypes(MIN_COV)
in_tree = set(zip(tree_sites.CHROM, tree_sites.POS, tree_sites.ALT))

# ── ONE ROW PER VARIANT ──────────────────────────────────────────────────────
t = classes.merge(nj, on=KEY, how="left", validate="1:1")
assert t.REF.notna().all(), "a variant is missing from nj_sites.tsv"
for l in LIBRARIES:                         # molecules left blank where the library is not callable at the site
    ok = t[f"{l}_COVERAGE"].between(MIN_COV, p99[l])
    t[f"{l}_VARIANT_MOLECULES"] = t[f"{l}_VARIANT_MOLECULES"].where(ok).astype("Int64")
t["CLASS"] = t.TYPE.map(CLASS_LABELS)
t["TREE"]  = ["Yes" if k in in_tree else "" for k in zip(t.CHROM, t.POS, t.ALT)]
t = (t.assign(_c=t.CHROM.str.replace("T2T", "").astype(int)).sort_values(["_c", "POS"])
      .drop(columns="_c").reset_index(drop=True))
value_cols = [f"{l}_{g}" for g, _ in GROUPS for l in LIBRARIES]

# ── WRITE: two header rows, the group names merged over their libraries ─────
head1 = [h for _, h in ID_COLUMNS] + [x for _, name in GROUPS for x in [name] + [""] * (len(LIBRARIES) - 1)]
head2 = [""] * len(ID_COLUMNS) + [LABELS[l] for _ in GROUPS for l in LIBRARIES]
body  = t[[c for c, _ in ID_COLUMNS] + value_cols]

wb = Workbook()
ws = wb.active
ws.title = "Table S7"
ws.append(head1)
ws.append(head2)
for row in body.itertuples(index=False):
    ws.append([None if pd.isna(v) else (int(v) if isinstance(v, (int, np.integer)) else v) for v in row])
bold, centre = Font(bold=True), Alignment(horizontal="center", vertical="center", wrap_text=True)
thin = Side(style="thin", color="999999")
n_id = len(ID_COLUMNS)
for j in range(1, n_id + 1):                                         # identifiers span both header rows
    ws.merge_cells(start_row=1, end_row=2, start_column=j, end_column=j)
for g in range(len(GROUPS)):                                         # group names span their libraries
    c0 = n_id + 1 + g * len(LIBRARIES)
    ws.merge_cells(start_row=1, end_row=1, start_column=c0, end_column=c0 + len(LIBRARIES) - 1)
    for r in range(1, ws.max_row + 1):                               # line before each group
        ws.cell(r, c0).border = Border(left=thin)
for r in (1, 2):
    for j in range(1, ws.max_column + 1):
        cell = ws.cell(r, j)
        cell.font, cell.alignment = bold, centre
        cell.border = Border(left=cell.border.left, bottom=thin if r == 2 else None)
widths = {"Chromosome": 11, "Position": 12, "Reference": 10, "Alternative": 11, "Trinucleotide context": 13,
          "Class": 26, "In the tree": 9}
for j, (_, h) in enumerate(ID_COLUMNS, start=1):
    ws.column_dimensions[ws.cell(1, j).column_letter].width = widths.get(h, 12)
for j in range(n_id + 1, ws.max_column + 1):
    ws.column_dimensions[ws.cell(2, j).column_letter].width = 10
ws.row_dimensions[1].height = 32
ws.freeze_panes = ws.cell(3, n_id + 1)
wb.save(SAVE_XLSX)
with open(SAVE_TSV, "w") as fh:
    fh.write("\t".join(head1) + "\n" + "\t".join(head2) + "\n")
    body.to_csv(fh, sep="\t", index=False, header=False)
print(f"saved {SAVE_XLSX} and {SAVE_TSV}: {len(t):,} variants")

# ── NUMBERS ──────────────────────────────────────────────────────────────────
print(t.CLASS.value_counts().to_string())
print(f"in the tree: {(t.TREE == 'Yes').sum()} ({', '.join(f'{k} {v}' for k, v in t[t.TREE == 'Yes'].CLASS.value_counts().items())})")
blank = t[[f"{l}_VARIANT_MOLECULES" for l in LIBRARIES]].isna()
print(f"library-variant cells left blank (not callable): {int(blank.to_numpy().sum()):,} of {blank.size:,}")

# Supplementary figures

In the order of the paper. Each cell saves a PDF to `FIG_DIR`; `FIGSIZE`, the first line of each cell, is the printed size in inches.

## Supplementary Fig. 1. Filter flowchart

The filters from the consensus reads mapped to the T2T genome to the somatic mutations and the germline or early embryonic mutations, with the total of each set over the seven animals (`final_variants()`, setup 0a). The figure is drawn at its own size (about 7.1 × 6.8 in), set by the box widths and gaps.

In [ ]:
# =============================================================================
# Supplementary Fig. 1. Filter flowchart — filters from the mapped reads to the somatic and germline
#   mutation candidates. The figure size follows from the box widths and gaps below (about 7.1 in wide).
# =============================================================================
from itertools import combinations

from matplotlib.patches import FancyBboxPatch, PathPatch, Polygon
from matplotlib.path import Path

# ── CONTENT ──────────────────────────────────────────────────────────────────
# Boxes are (text, description), top to bottom in the order the filters are applied.
# An empty description gives a one-part box; "\n" in a description starts a new line.
# A section title may carry a subtitle after "\n".
INPUT_NODE = ("Consensus reads mapped to the T2T genome", "")
CALL_NODE  = ("Variant calling", "Same non-reference base on both strands, in trimmed and untrimmed reads")

PRE_TITLE      = "Pre-calling variant filters"
CALLABLE_TITLE = "Callable sites"
POST_TITLE     = "Post-calling variant filters"

PRE_SECTIONS = [
    ("Consensus read QC", [
        ("Only primary alignments", ""),
        ("On an autosome", ""),
        ("Mapping quality of 60", ""),
        ("≥ 5 passes", ""),
        ("Within the library's read-length range", ""),
        ("≤ 1% divergence from the reference", ""),
    ]),
    ("Molecule QC", [
        ("Both strands available", ""),
        ("Strand length agreement", ""),
        ("Not chimeric", ""),
        ("Strand alignments cover the same reference span", ""),
    ]),
    ("Base QC\non both strands", [
        ("Phred base quality ≥ 40", ""),
        ("sm ≥ 5 and sm/(sm + sx) > 0.8", ""),
        ("Not within 3 bases of an indel", ""),
        ("No N in the reference at or beside the base", ""),
    ]),
]
# The last base filter branches into the two read sets: scissors (trimmed) and crossed-out scissors (untrimmed)
TRIMMED   = ("Trimmed reads", "450 bases cut\nfrom each end")
UNTRIMMED = ("Untrimmed reads", "")

# Callable sites: a frame of its own, left of the post-calling filters (no arrows in or out); the two
# conditions are joined by SITE_JOIN
SITE_BOXES = [
    ("Unmasked sites", "Not in simple repeats, low-complexity sequence or satellite DNA"),
    ("Pass coverage filter", "Untrimmed coverage from 10 molecules to the library's 99th percentile"),
]
SITE_JOIN = "&"

# Post-calling: one filter box, then one labelled arrow per variant set
POST_BOX = ("Molecules with ≤ 2 singletons", "")
BRANCHES = [                                   # (arrow label, output node, key of the total in final_variants())
    ("Seen in one molecule in trimmed and untrimmed reads, not shared among libraries",
     ("Somatic mutations", ""), "Somatic SNV"),
    ("Seen in more than one molecule in untrimmed reads, not shared among animals",
     ("Germline or early embryonic mutations", ""), "Germline candidate"),
]
# Totals, from final_variants() (setup 0a). TOTALS_BY = "library": one per library and site (the sums of
# Supplementary Table 4); "variant": each variant once (a germline candidate found in both 60-week
# libraries counts once)
TOTALS_BY   = "variant"
TOTALS_NOTE = {"library": "Total over the eight libraries",
               "variant": "Total over the seven animals"}   # left of the totals box
NOISY_CUTOFF           = 2       # maximum singletons per molecule
MIN_COV                = 10      # minimum untrimmed coverage of the coverage filter
GERMLINE_MIN_MOLECULES = 2       # germline candidates: at least this many molecules carrying the non-reference allele

# ── STYLE KNOBS (lengths in inches) ──────────────────────────────────────────
BOX_W        = 1.20        # filter box width
BASE_W       = 1.54        # filter box width in the base QC section (holds the two read-set boxes)
NODE_W       = 0.92        # input / calling node width
OUT_W        = 1.70        # output node width (also the width of the branch labels)
SITE_W       = 1.70        # callable-site box width
POST_BOX_W   = 1.50        # post-calling filter box width
COL_GAP      = 0.20        # gap between columns (arrow length)
SECTION_GAP  = 0.17        # gap between the pre-calling sections
BOX_GAP      = 0.17        # vertical gap between filter boxes (arrow length)
ROW_GAP      = 0.30        # vertical gap between the pre-calling frame and the lower row (callable sites, post-calling)
ROUTE_GAP    = 0.15        # minimum gap between the post-calling frame and the arrow from the variant calling
READ_GAP     = 0.10        # gap between the trimmed and untrimmed boxes
READ_DROP    = 0.30        # vertical gap between the last base filter and the read-set boxes
OUT_GAP      = 0.30        # gap between the output nodes
FORK_STEM    = 0.08        # stem of a branching arrow, before it splits
LABEL_PAD    = 0.10        # space above and below the branch labels, inside the post-calling frame
TOTAL_DROP   = 0.22        # arrows from the output nodes to the totals box
SECTION_PAD  = 0.07        # section padding around its boxes
FRAME_PAD    = 0.07        # frame padding around its sections
TEXT_PAD     = 0.05        # padding inside boxes
MAIN_SIZE    = 7           # box text
DESC_SIZE    = 6           # box description and section subtitle
LABEL_SIZE   = 9.5         # labels on the branch arrows
TOTAL_SIZE   = 10          # numbers in the totals box (bold, in the colour of their set)
TOTAL_NOTE_SIZE = 8.5      # TOTALS_NOTE, left of the totals box
TOTAL_NOTE_GAP  = 0.12     # gap between TOTALS_NOTE and the totals box
SITE_SIZE    = 8.5         # callable-site box text
SITE_DESC_SIZE = 7.5       # callable-site box description
SITE_JOIN_SIZE = 13        # SITE_JOIN between the two callable-site boxes (bold)
SITE_JOIN_GAP  = 0.30      # vertical gap between the two callable-site boxes, holding SITE_JOIN
ICON_SIZE    = 12          # scissors
SECTION_SIZE = 7.5         # section titles (bold)
FRAME_SIZE   = 8.5         # frame titles (bold)
NODE_SIZE    = 7           # node text (bold); node descriptions use DESC_SIZE
LINE_SPACING = 1.22        # line height as a multiple of the font size
PRE_COLORS   = dict(frame="#EEF2F8", node="#4C72B0",
                    sections=["#D9E2F0", "#D3E8E6", "#E0DBEF"])   # one per pre-calling section
CALLABLE_COLOR = "#EDF3EF"                     # callable-sites frame
POST_COLORS  = dict(frame="#FBF1EA")
OUT_COLORS   = ["#C8414B", "#7B4FA0"]   # somatic, germline
CALL_COLOR   = "#5E6B7D"
CROSS_COLOR  = "#C0392B"   # cross over the untrimmed scissors
CROSS_LW     = 1.4
BOX_EDGE     = "0.62"
ARROW_COLOR  = "0.30"
ARROW_LW     = 0.9         # arrow shaft width (points)
CORNER_R     = 0.06        # corner radius of the stepped arrow from the variant calling
HEAD_L       = 0.065       # arrowhead length
HEAD_W       = 0.060       # arrowhead width
HEAD_ROUND   = 1.6         # arrowhead outline width (points); sets how round its corners are
TEXT_COLOR   = "0.10"
DESC_COLOR   = "0.38"
SAVE_NAME    = "filter_flowchart"

# ── TOTALS ───────────────────────────────────────────────────────────────────
_fv = final_variants(NOISY_CUTOFF, MIN_COV, GERMLINE_MIN_MOLECULES)
if TOTALS_BY == "variant":
    _fv = _fv.drop_duplicates(["TYPE", "CHROM", "POS", "ALT"])
totals = _fv.TYPE.value_counts()
print("totals:", ", ".join(f"{key} {int(totals.get(key, 0)):,}" for _, _, key in BRANCHES))

# ── TEXT MEASUREMENT (wraps text to the box width) ───────────────────────────
_mfig = plt.figure()
_rend = _mfig.canvas.get_renderer()


def text_width(s, size, weight="normal"):
    t = _mfig.text(0, 0, s, fontsize=size, fontweight=weight)
    w = t.get_window_extent(_rend).width / _mfig.dpi
    t.remove()
    return w


def wrap(s, size, width, weight="normal"):
    """Word wrap of s to the fewest lines no wider than width (inches), with balanced line lengths."""
    words = s.split()
    lines, cur = [], ""
    for word in words:
        trial = f"{cur} {word}".strip()
        if cur and text_width(trial, size, weight) > width:
            lines.append(cur)
            cur = word
        else:
            cur = trial
    lines += [cur] if cur else []
    n = len(lines)
    if n < 2:
        return lines
    best = None
    for cuts in combinations(range(1, len(words)), n - 1):
        parts = [" ".join(words[a:b]) for a, b in zip((0,) + cuts, cuts + (len(words),))]
        widest = max(text_width(x, size, weight) for x in parts)
        if widest <= width and (best is None or widest < best[0]):
            best = (widest, parts)
    return best[1] if best else lines


def line_h(size):
    return size * LINE_SPACING / 72


def layout(text, desc, w, node=False, icon=False):
    """Wrapped text and description lines, and the box height (with room for an icon on top)."""
    size, weight = (NODE_SIZE, "bold") if node else (MAIN_SIZE, "normal")
    inner = w - 2 * TEXT_PAD
    t = wrap(text, size, inner, weight)
    d = [x for part in desc.split("\n") for x in wrap(part, DESC_SIZE, inner)] if desc else []
    h = 2 * TEXT_PAD + len(t) * line_h(size) + len(d) * line_h(DESC_SIZE) + (line_h(ICON_SIZE) if icon else 0)
    return t, d, h


def site_layout(text, desc, w):
    """Wrapped text and description lines of a callable-site box, and the height of the text block."""
    inner = w - 2 * TEXT_PAD
    t = wrap(text, SITE_SIZE, inner)
    d = [x for part in desc.split("\n") for x in wrap(part, SITE_DESC_SIZE, inner)] if desc else []
    return t, d, len(t) * line_h(SITE_SIZE) + len(d) * line_h(SITE_DESC_SIZE)


# ── DRAWING HELPERS ──────────────────────────────────────────────────────────
def rounded(ax, x, y, w, h, r, face, edge="none", lw=0.0, ls="-"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=f"round,pad=0,rounding_size={r}",
                                facecolor=face, edgecolor=edge, lw=lw, ls=ls))


def draw_box(ax, x, y_top, text, desc, w, node_color=None, icon=None):
    """Filter box (white) or node (filled with node_color); top-left corner at (x, y_top).
    icon: "cut" (scissors) or "uncut" (scissors with a red cross) on top. Returns the box height."""
    node = node_color is not None
    t, d, h = layout(text, desc, w, node, icon is not None)
    if node:
        rounded(ax, x, y_top - h, w, h, 0.07, node_color)
    else:
        rounded(ax, x, y_top - h, w, h, 0.05, "white", BOX_EDGE, 0.6)
    ink = "white" if node else TEXT_COLOR
    size, weight = (NODE_SIZE, "bold") if node else (MAIN_SIZE, "normal")
    y = y_top - TEXT_PAD
    if icon is not None:
        cx, cy = x + w / 2, y - line_h(ICON_SIZE) / 2
        ax.text(cx, cy, "✂", ha="center", va="center", fontsize=ICON_SIZE, color=TEXT_COLOR)
        if icon == "uncut":
            s = 0.42 * ICON_SIZE / 72
            for sgn in (1, -1):
                ax.plot([cx - s, cx + s], [cy - sgn * s, cy + sgn * s], color=CROSS_COLOR,
                        lw=CROSS_LW, solid_capstyle="round")
        y -= line_h(ICON_SIZE)
    for line in t:
        ax.text(x + w / 2, y - line_h(size) / 2, line, ha="center", va="center", fontsize=size,
                fontweight=weight, color=ink)
        y -= line_h(size)
    for line in d:
        ax.text(x + w / 2, y - line_h(DESC_SIZE) / 2, line, ha="center", va="center",
                fontsize=DESC_SIZE, color="white" if node_color is not None else DESC_COLOR)
        y -= line_h(DESC_SIZE)
    return h


def head(ax, tip, u):
    """Round-cornered arrowhead with its tip at `tip`, pointing along the unit vector u."""
    ux, uy = u
    r = HEAD_ROUND / 2 / 72                       # outline half-width (inches)
    tx, ty = tip[0] - ux * r, tip[1] - uy * r     # tip of the head's outline path
    bx, by = tx - ux * (HEAD_L - 2 * r), ty - uy * (HEAD_L - 2 * r)
    hw = HEAD_W / 2 - r
    ax.add_patch(Polygon([(tx, ty), (bx - uy * hw, by + ux * hw), (bx + uy * hw, by - ux * hw)],
                         closed=True, facecolor=ARROW_COLOR, edgecolor=ARROW_COLOR,
                         lw=HEAD_ROUND, joinstyle="round"))
    return bx, by


def arrow(ax, x0, y0, x1, y1):
    """Straight arrow from (x0, y0) to (x1, y1) with a round-cornered head."""
    L = np.hypot(x1 - x0, y1 - y0)
    bx, by = head(ax, (x1, y1), ((x1 - x0) / L, (y1 - y0) / L))
    ax.plot([x0, bx], [y0, by], color=ARROW_COLOR, lw=ARROW_LW, solid_capstyle="butt")


def fork(ax, x0, y0, xs, y1):
    """Arrow down from (x0, y0) that splits into one arrow per x in xs, each ending at y1."""
    ym = y0 - FORK_STEM
    ax.plot([x0, x0], [y0, ym], color=ARROW_COLOR, lw=ARROW_LW, solid_capstyle="butt")
    ax.plot([min(xs), max(xs)], [ym, ym], color=ARROW_COLOR, lw=ARROW_LW, solid_capstyle="projecting")
    for x in xs:
        arrow(ax, x, ym, x, y1)
    return ym


def route_arrow(ax, pts):
    """Solid arrow along the straight segments through pts, with rounded corners."""
    pts = [np.asarray(p, float) for p in pts]
    u_end = (pts[-1] - pts[-2]) / np.linalg.norm(pts[-1] - pts[-2])
    pts[-1] = np.array(head(ax, pts[-1], u_end))
    verts, codes = [pts[0]], [Path.MOVETO]
    for a, b, c in zip(pts[:-2], pts[1:-1], pts[2:]):
        u_in, u_out = (b - a) / np.linalg.norm(b - a), (c - b) / np.linalg.norm(c - b)
        verts += [b - CORNER_R * u_in, b, b + CORNER_R * u_out]
        codes += [Path.LINETO, Path.CURVE3, Path.CURVE3]
    verts.append(pts[-1])
    codes.append(Path.LINETO)
    ax.add_patch(PathPatch(Path(verts, codes), facecolor="none", edgecolor=ARROW_COLOR, lw=ARROW_LW,
                           capstyle="butt"))


def chain_height(boxes, w):
    return sum(layout(t, d, w)[2] for t, d in boxes) + BOX_GAP * (len(boxes) - 1)


def section_title_h(title):
    lines = title.split("\n")
    return 2 * TEXT_PAD + line_h(SECTION_SIZE) + (len(lines) - 1) * line_h(DESC_SIZE)


def draw_section(ax, x, y_top, h, section, color, box_w):
    """Shaded section with its title and chain of boxes of width box_w; returns the bottom y of the last box."""
    title, boxes = section
    w = box_w + 2 * SECTION_PAD
    rounded(ax, x, y_top - h, w, h, 0.08, color)
    head_line, *sub = title.split("\n")
    y = y_top - (SECTION_H - section_title_h(title)) / 2 - TEXT_PAD
    ax.text(x + w / 2, y - line_h(SECTION_SIZE) / 2, head_line, ha="center", va="center",
            fontsize=SECTION_SIZE, fontweight="bold", color=TEXT_COLOR)
    for line in sub:
        y -= line_h(SECTION_SIZE) / 2 + line_h(DESC_SIZE) / 2
        ax.text(x + w / 2, y - line_h(SECTION_SIZE) / 2, line, ha="center", va="center",
                fontsize=DESC_SIZE, color=DESC_COLOR)
    xb, y = x + SECTION_PAD, y_top - SECTION_H
    for i, (t, d) in enumerate(boxes):
        if i:
            arrow(ax, xb + box_w / 2, y + BOX_GAP, xb + box_w / 2, y)
        y -= draw_box(ax, xb, y, t, d, box_w) + BOX_GAP
    return y + BOX_GAP


def draw_frame(ax, x0, y_top, w, h, lines, color):
    """Shaded frame with its bold title on top."""
    rounded(ax, x0, y_top - h, w, h, 0.12, color)
    ax.text(x0 + w / 2, y_top - frame_title_h / 2, "\n".join(lines), ha="center", va="center",
            fontsize=FRAME_SIZE, fontweight="bold", color=TEXT_COLOR, linespacing=LINE_SPACING)


# ── LAYOUT ───────────────────────────────────────────────────────────────────
n_pre     = len(PRE_SECTIONS)
box_ws    = [BOX_W] * (n_pre - 1) + [BASE_W]     # the base QC section is the last one
sec_ws    = [w + 2 * SECTION_PAD for w in box_ws]
READ_W    = (BASE_W - READ_GAP) / 2
read_h    = max(layout(*TRIMMED, READ_W, icon=True)[2], layout(*UNTRIMMED, READ_W, icon=True)[2])
read_ext  = READ_DROP + read_h                    # the read-set boxes under the last base filter
SECTION_H = max(section_title_h(t) for t, _ in PRE_SECTIONS)
chains    = [chain_height(b, w) + (read_ext if i == n_pre - 1 else 0)
             for i, ((_, b), w) in enumerate(zip(PRE_SECTIONS, box_ws))]
sec_h     = SECTION_H + max(chains) + SECTION_PAD
w_pre     = sum(sec_ws) + (n_pre - 1) * SECTION_GAP + 2 * FRAME_PAD
w_site    = SITE_W + 2 * FRAME_PAD
w_right   = max(POST_BOX_W, len(BRANCHES) * OUT_W + (len(BRANCHES) - 1) * OUT_GAP) + 2 * FRAME_PAD
pre_lines      = wrap(PRE_TITLE, FRAME_SIZE, w_pre - 2 * FRAME_PAD, "bold")
callable_lines = wrap(CALLABLE_TITLE, FRAME_SIZE, w_site - 2 * FRAME_PAD, "bold")
post_lines     = wrap(POST_TITLE, FRAME_SIZE, w_right - 2 * FRAME_PAD, "bold")
frame_title_h  = max(len(pre_lines), len(callable_lines), len(post_lines)) * line_h(FRAME_SIZE) + 2 * FRAME_PAD
frame_h   = frame_title_h + sec_h + FRAME_PAD
post_box_h = layout(*POST_BOX, POST_BOX_W)[2]
label_lines = [wrap(label, LABEL_SIZE, OUT_W) for label, _, _ in BRANCHES]
label_h   = max(len(x) for x in label_lines) * line_h(LABEL_SIZE)
post_h    = frame_title_h + post_box_h + FORK_STEM + label_h + 2 * LABEL_PAD   # ends below the branch labels
out_h     = max(layout(*node, OUT_W, node=True)[2] for _, node, _ in BRANCHES)
total_h   = 2 * TEXT_PAD + line_h(TOTAL_SIZE)
site_text = [site_layout(t, d, SITE_W) for t, d in SITE_BOXES]
# the callable-sites frame reaches down to the bottom of the output nodes; its two boxes share the height
callable_h = max(frame_title_h + 2 * FRAME_PAD + SITE_JOIN_GAP + 2 * max(h for _, _, h in site_text) + 2 * TEXT_PAD,
                 post_h + 2 * BOX_GAP + out_h)
site_box_h = (callable_h - frame_title_h - 2 * FRAME_PAD - SITE_JOIN_GAP) / 2
top_h     = frame_h + ROW_GAP                                 # upper row and the gap below it
low_h     = callable_h + TOTAL_DROP + total_h

# upper row: input -> pre-calling filters -> variant calling; lower row: callable sites, post-calling filters
x_in   = 0.0
x_pre  = x_in + NODE_W + COL_GAP
x_call = x_pre + w_pre + COL_GAP
x_site = x_in
x_post = x_site + w_site + COL_GAP
x_drop = max(x_call + NODE_W / 2, x_post + w_right + ROUTE_GAP)   # vertical part of the arrow from the variant calling
W      = max(x_call + NODE_W, x_drop + ROUTE_GAP)
H      = top_h + low_h + BOX_GAP

# ── PLOT ─────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(W, H))
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, W)
ax.set_ylim(0, H)
ax.axis("off")

y_top   = H
y_sec   = y_top - frame_title_h                 # top of the sections
y_flow  = y_sec - SECTION_H / 2                 # section-title row: the main flow
y_bot   = y_top - frame_h                       # bottom of the pre-calling frame
y_ptop  = y_top - top_h                        # top of the lower row (callable sites, post-calling filters)
y_post  = y_ptop - post_h                       # bottom of the post-calling frame
y_out   = y_post - 2 * BOX_GAP                  # top of the output nodes
y_total = y_ptop - callable_h - TOTAL_DROP      # top of the totals box
x_mid   = x_post + w_right / 2                  # centre of the post-calling frame
x_smid  = x_site + w_site / 2                   # centre of the callable-sites frame

draw_frame(ax, x_pre, y_top, w_pre, frame_h, pre_lines, PRE_COLORS["frame"])
draw_frame(ax, x_site, y_ptop, w_site, callable_h, callable_lines, CALLABLE_COLOR)
draw_frame(ax, x_post, y_ptop, w_right, post_h, post_lines, POST_COLORS["frame"])

# input -> pre-calling sections -> variant calling
in_h = layout(*INPUT_NODE, NODE_W, node=True)[2]
draw_box(ax, x_in, y_flow + in_h / 2, *INPUT_NODE, NODE_W, PRE_COLORS["node"])
arrow(ax, x_in + NODE_W, y_flow, x_pre + FRAME_PAD, y_flow)

x = x_pre + FRAME_PAD
for i, section in enumerate(PRE_SECTIONS):
    last_bottom = draw_section(ax, x, y_sec, sec_h, section, PRE_COLORS["sections"][i], box_ws[i])
    if i:
        arrow(ax, x - SECTION_GAP, y_flow, x, y_flow)
    xb = x + SECTION_PAD                          # left edge of this section's boxes
    x += sec_ws[i] + SECTION_GAP

# trimmed and untrimmed reads under the last base filter
x_tr, x_un = xb, xb + READ_W + READ_GAP
y_read = last_bottom - READ_DROP
fork(ax, xb + BASE_W / 2, last_bottom, [x_tr + READ_W / 2, x_un + READ_W / 2], y_read)
draw_box(ax, x_tr, y_read, *TRIMMED, READ_W, icon="cut")
draw_box(ax, x_un, y_read, *UNTRIMMED, READ_W, icon="uncut")

call_h = layout(*CALL_NODE, NODE_W, node=True)[2]
arrow(ax, x_pre + w_pre - FRAME_PAD, y_flow, x_call, y_flow)
draw_box(ax, x_call, y_flow + call_h / 2, *CALL_NODE, NODE_W, CALL_COLOR)

# callable sites: the two site conditions, one under the other, joined by SITE_JOIN
y = y_ptop - frame_title_h - FRAME_PAD
for i, (t, d, h) in enumerate(site_text):
    if i:
        ax.text(x_smid, y + SITE_JOIN_GAP / 2, SITE_JOIN, ha="center", va="center", fontsize=SITE_JOIN_SIZE,
                fontweight="bold", color=TEXT_COLOR)
    rounded(ax, x_smid - SITE_W / 2, y - site_box_h, SITE_W, site_box_h, 0.05, "white", BOX_EDGE, 0.6)
    yt = y - (site_box_h - h) / 2                # text block centred in the box
    for line in t:
        ax.text(x_smid, yt - line_h(SITE_SIZE) / 2, line, ha="center", va="center", fontsize=SITE_SIZE, color=TEXT_COLOR)
        yt -= line_h(SITE_SIZE)
    for line in d:
        ax.text(x_smid, yt - line_h(SITE_DESC_SIZE) / 2, line, ha="center", va="center", fontsize=SITE_DESC_SIZE,
                color=DESC_COLOR)
        yt -= line_h(SITE_DESC_SIZE)
    y -= site_box_h + SITE_JOIN_GAP

# variant calling -> down the right side, then left into the post-calling filters
y_pbox = y_ptop - frame_title_h
y_in   = y_pbox - post_box_h / 2
x_cmid = x_call + NODE_W / 2
pts = [(x_cmid, y_flow - call_h / 2)]
if x_drop > x_cmid:                              # step right first if the call node is above the post-calling frame
    y_step = (y_bot + y_ptop) / 2
    pts += [(x_cmid, y_step), (x_drop, y_step)]
route_arrow(ax, pts + [(x_drop, y_in), (x_mid + POST_BOX_W / 2, y_in)])
draw_box(ax, x_mid - POST_BOX_W / 2, y_pbox, *POST_BOX, POST_BOX_W)

# one labelled arrow per variant set, then the output nodes and the totals box
xs = [x_mid + (i - (len(BRANCHES) - 1) / 2) * (OUT_W + OUT_GAP) for i in range(len(BRANCHES))]
y_fork = fork(ax, x_mid, y_pbox - post_box_h, xs, y_out)
y_label = (y_fork + y_post) / 2
for x, lines, (_, node, _), color in zip(xs, label_lines, BRANCHES, OUT_COLORS):
    ax.text(x, y_label, "\n".join(lines), ha="center", va="center", fontsize=LABEL_SIZE,
            color=TEXT_COLOR, linespacing=LINE_SPACING,
            bbox=dict(boxstyle="round,pad=0.25", facecolor=POST_COLORS["frame"], edgecolor="none"))
    h = draw_box(ax, x - OUT_W / 2, y_out, *node, OUT_W, color)
    arrow(ax, x, y_out - h, x, y_total)

w_total = xs[-1] - xs[0] + OUT_W
rounded(ax, x_mid - w_total / 2, y_total - total_h, w_total, total_h, 0.05, "white", BOX_EDGE, 0.6)
y_num = y_total - total_h / 2
for x, (_, _, key), color in zip(xs, BRANCHES, OUT_COLORS):
    ax.text(x, y_num, f"{int(totals.get(key, 0)):,}", ha="center", va="center", fontsize=TOTAL_SIZE,
            fontweight="bold", color=color)
x_note = x_mid - w_total / 2 - TOTAL_NOTE_GAP
note = wrap(TOTALS_NOTE[TOTALS_BY], TOTAL_NOTE_SIZE, x_note)
ax.text(x_note, y_num, "\n".join(note), ha="right", va="center", fontsize=TOTAL_NOTE_SIZE, color=TEXT_COLOR,
        linespacing=LINE_SPACING)

plt.close(_mfig)
save_figure(fig, SAVE_NAME)
plt.show()

## Supplementary Fig. 2. Read-length distribution for each library

Histogram of the HiFi strand consensus read lengths, one panel per library; dashed line: the mean; grey lines: the bounds of the read-length filter.

In [ ]:
FIGSIZE = (PAGE_WIDTH, 3.25)          # width, height in inches; the panels fill it minus fixed margins
# =============================================================================
# Supplementary Fig. 2. Read-length distribution — one panel per library
#   Input   read_tables/<library>.reads.npz; filter_thresholds/<library>.read_length_bounds.txt
# =============================================================================
ROW_GAP_IN = 0.60                     # between the rows: the top row's x labels and the bottom row's titles

# ── STYLE KNOBS ──────────────────────────────────────────────────────────────
BIN_WIDTH    = 200              # bases per histogram bin
X_MAX        = 20_000           # x-axis limit (bases)
SHOW_BOUNDS  = True             # grey lines at the read-length filter bounds
Y_DIV        = 1e6              # y tick labels are drawn in units of this
Y_LABEL      = r"Reads ($\times 10^6$)"
X_LABEL      = "Read length (kb)"
BAR_COLOR    = "#4C72B0"        # histogram
MEAN_COLOR   = "#B5452B"        # mean line and its legend text
MEAN_STYLE   = "--"
MEAN_LW      = 1.0
FILTER_COLOR = "#595959"        # filter lines
FILTER_LW    = 0.8
SAVE_NAME    = "read_length_distribution"

# ── PLOT ─────────────────────────────────────────────────────────────────────
edges = np.arange(0, X_MAX + BIN_WIDTH, BIN_WIDTH)
fig, axes = library_grid(FIGSIZE, gaps=(0.16, ROW_GAP_IN), top_xticks=True)

for i, row in enumerate(GRID):
    for j, lib in enumerate(row):
        ax = axes[i, j]
        L  = load_reads(lib, ("length",))["length"]

        heights, _ = np.histogram(L, bins=edges)
        ax.stairs(heights, edges, fill=True, color=BAR_COLOR, lw=0)

        if SHOW_BOUNDS:
            for bound in length_bounds(lib):
                ax.axvline(bound, color=FILTER_COLOR, lw=FILTER_LW)

        mean = L.mean()
        ax.axvline(mean, color=MEAN_COLOR, lw=MEAN_LW, ls=MEAN_STYLE, label=f"Mean: {mean / 1e3:.2f} kb")
        ax.legend(loc="upper right", fontsize=FONT["legend"], labelcolor=MEAN_COLOR, handlelength=1.6,
                  handletextpad=0.5, borderaxespad=0.3, facecolor="white", edgecolor="none", framealpha=1)
        del L

axes[0, 0].set_xlim(0, X_MAX)
axes[0, 0].xaxis.set_major_locator(ticker.MultipleLocator(5_000))
axes[0, 0].xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v / 1e3:g}"))
axes[0, 0].yaxis.set_major_locator(ticker.MultipleLocator(0.5 * Y_DIV))
axes[0, 0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v / Y_DIV:g}"))
grid_labels(axes, X_LABEL, Y_LABEL, top_xlabel=True)

save_figure(fig, SAVE_NAME, crop=False)
plt.show()

## Supplementary Fig. 3. Relative density of singletons and control variants vs distance from the read end

Singletons (a) and control molecules (b) in each bin, divided by the number expected from the read interiors (the sum over the libraries of the library's interior density × its callable base pairs in the bin), in the eight libraries (Methods, Errors at the edges of consensus reads). Uses `edge_effect()` (setup 0c).

In [ ]:
FIGSIZE = (2 * SINGLE_SIZE[0], SINGLE_SIZE[1] + LETTER_IN)   # width, height in inches: two one-panel cells side by side under the panel letters (setup 0b)
# =============================================================================
# Supplementary Fig. 3. Relative density vs distance from the read end — singletons (a) and the
# control class (b)
#
# a: singletons in each bin / the singletons expected from the interior, m = sum over the libraries
#    of (the library's interior singleton density x its callable base pairs in the bin).
# b: the same for the control molecules, with each library's interior control density.
# x: distance from the nearer end of the forward strand consensus, BIN-base bins, library set
# LIB_SET, molecules as in MOLECULES. Error bars: exact 95% Poisson on the bin's count. Dashed
# line: 1 (= the interior density). Grey line: the trim. Both panels share the y axis. Callable base
# pairs at d <= 5 are not counted, so the first bin uses d = 6-25.
#
# Printed: observed / expected in the COARSE ranges (95% CI of the count), and for each class a
# chi-square test that observed / expected is the same in every TEST_BIN-base bin.
# =============================================================================
from matplotlib.ticker import FixedLocator, FuncFormatter, MaxNLocator, NullLocator
from scipy import stats

# ── KNOBS ────────────────────────────────────────────────────────────────────
LIB_SET   = "all"                      # "all" (eight libraries), "short" (seven 4 kb) or "long" (10 kb)
MOLECULES = "final"                    # "final": molecules with > 2 singletons in the trimmed reads dropped
                                       # (E.CNT, as in Supplementary Fig. 4); "every": every molecule (E.CNT_ALL)
BIN       = 25                         # bases per plotted bin: 25 or 50
COARSE    = [0, 25, 50, 100, 150, 250, 450, 750, 1000, 1500]    # printed ranges: d = a + 1 .. b

# ── STYLE KNOBS ──────────────────────────────────────────────────────────────
LOG_TICKS  = [0.2, 0.5, 1, 2, 5, 10, 20, 50]
COLORS     = {"singletons": "#4C72B0", "control": "#3E8E57"}
INK        = "0.15"                    # dashed line at 1
TRIM_LINE  = dict(ls="-", lw=1.0, color="0.6")
MARKER_SIZE, ERR_LW, CAP_SIZE = 3, 0.8, 1.5
X_LABEL    = "Distance from the read end (bases)"
Y_LABEL    = "Relative density"
SAVE_NAME  = f"rate_vs_distance_{LIB_SET}"

# ── PLOT ─────────────────────────────────────────────────────────────────────
E = edge_effect()
c = (E.CNT if MOLECULES == "final" else E.CNT_ALL)[LIB_SET]
fig, axes, _ = single_cells(FIGSIZE, "ab")
rows = (("singletons", c["S0"], "Singletons"), ("control", c["K0"], "Control class"))
ylo, yhi = np.inf, 0
for ax, (what, n0, label) in zip(axes, rows):
    num, exp = c["fold"][what], c["fold"][f"expected_{what}"]
    nb = num.reshape(-1, BIN).sum(axis=1)
    mb = exp.reshape(-1, BIN).sum(axis=1)
    mid = np.arange(len(nb)) * BIN + (BIN + 1) / 2     # bin centres, d = 1 .. WINDOW
    lo, hi = poisson_ci(nb)
    y, y_lo, y_hi = nb / mb, lo / mb, hi / mb
    y = np.where(nb > 0, y, np.nan)
    eb = ax.errorbar(mid, y, yerr=[y - y_lo, y_hi - y], fmt="o", ms=MARKER_SIZE, color=COLORS[what],
                     ecolor=COLORS[what], elinewidth=ERR_LW, capsize=CAP_SIZE, capthick=ERR_LW)
    eb.lines[0].set_markeredgewidth(0)
    ax.axhline(1, ls="--", lw=1.2, color=INK)
    ax.axvline(E.TRIM, **TRIM_LINE)
    ax.set_xlim(0, E.WINDOW)
    ax.set_xlabel(X_LABEL, fontsize=FONT_SINGLE["label"])
    ylo, yhi = min(ylo, np.nanmin(y_lo[nb > 0])), max(yhi, np.nanmax(y_hi))

    cells = []
    for a, b in zip(COARSE[:-1], COARSE[1:]):
        r, l_, h_ = E.obs_exp(num[a:b].sum(), exp[a:b].sum())
        cells.append(f"{a + 1}-{b}: {r:.2f} [{l_:.2f}, {h_:.2f}]")
    print(f"{label} ({LIB_SET}, {MOLECULES} molecules; window n = {num.sum():,}, interior n = {n0:,}), "
          f"observed / expected:\n    " + " | ".join(cells))
    k = num.reshape(-1, E.TEST_BIN).sum(axis=1)
    e = k.sum() * exp.reshape(-1, E.TEST_BIN).sum(axis=1) / exp.sum()
    chi = stats.chisquare(k, e)
    p = f"= {chi.pvalue:.3g}" if chi.pvalue > 0 else "< 1e-300"
    print(f"    {what}: observed / expected equal in every {E.TEST_BIN}-base bin: chi2 = {chi.statistic:.1f}, "
          f"df = {len(k) - 1}, p {p}")
for ax in axes:
    ax.set_yscale("log")
    ax.set_ylim(ylo / 1.2, yhi * 1.2)
    ax.yaxis.set_major_locator(FixedLocator([v for v in LOG_TICKS if ylo / 1.2 <= v <= yhi * 1.2]))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    ax.yaxis.set_minor_locator(NullLocator())
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4, steps=[1, 2, 2.5, 5, 10]))
    ax.tick_params(labelsize=FONT_SINGLE["tick"])
axes[0].set_ylabel(Y_LABEL, fontsize=FONT_SINGLE["label"])

save_figure(fig, SAVE_NAME, crop=False)
plt.show()

## Supplementary Fig. 4. Cumulative singletons vs distance from the read end

Fraction of the singletons, and of the expected singletons (the sum over the libraries of the library's interior density × its callable base pairs), at distance ≤ *d* from the read end, with the model of the edge errors, in the eight libraries; the FDR after trimming 450 bases and its one-sided 95% upper bound are printed under the cell. Uses `edge_effect()` (setup 0c).

In [ ]:
FIGSIZE = SINGLE_SIZE                  # width, height in inches (setup 0b); the panel fills it minus MARGINS_IN
# =============================================================================
# Supplementary Fig. 4. Cumulative singletons vs distance from the read end, and the FDR at the trim
#
# Distance d from the nearer end of the forward strand consensus, reads >= E.MIN_LEN, library set
# LIB_SET, molecules with more than 2 singletons in the trimmed reads dropped (E.CNT). One panel.
#     F_s(d) = singletons at distance <= d / singletons in the window               (blue curve)
#     F_m(d) = expected singletons m at distance <= d / m in the window, where m = sum over the
#              libraries of (interior density x callable base pairs)
# Dotted: no edge errors, F_s = F_m. Grey vertical line: the trim. Dashed: every edge error at the
# read end, F_s = f + (1 - f) F_m, with f = R(0) / S(0) the fraction of the window's singletons
# that are edge errors. The gap dashed - blue at d = t, times the window's singletons, is R(t),
# the edge errors left after trimming t bases from each end.
# Printed: f, the edge errors, and the FDR at the trim with its one-sided 95% upper bound.
# =============================================================================
from matplotlib.ticker import MaxNLocator

# ── KNOBS ────────────────────────────────────────────────────────────────────
LIB_SET = "all"                        # "all" (eight libraries), "short" (seven 4 kb) or "long" (10 kb)

# ── STYLE KNOBS ──────────────────────────────────────────────────────────────
MARGINS_IN = SINGLE_MARGINS           # left, right, top, bottom, inches (setup 0b)
BLUE, INK  = "#4C72B0", "0.15"
TRIM_LINE  = dict(ls="-", lw=1.0, color="0.6")
X_LABEL    = "Distance from the read end (bases)"
Y_LABEL    = "Cumulative fraction of singletons"
SAVE_NAME  = f"cumulative_singletons_{LIB_SET}"

# ── PLOT ─────────────────────────────────────────────────────────────────────
E = edge_effect()
c = E.CNT[LIB_SET]
W, H = FIGSIZE
left, right, top, bottom = MARGINS_IN
fig = plt.figure(figsize=FIGSIZE)
ax = fig.add_axes([left / W, bottom / H, (W - left - right) / W, (H - top - bottom) / H])
d = np.arange(1, E.WINDOW + 1)
s, m = c["fold"]["singletons"], c["fold"]["expected_singletons"]
Fs, Fm = np.cumsum(s) / s.sum(), np.cumsum(m) / m.sum()
f, se_f, art = E.edge_fraction(c)
row = E.model(c).set_index("t").loc[E.TRIM]
gap = f + (1 - f) * Fm[E.TRIM - 1] - Fs[E.TRIM - 1]
assert np.isclose(gap * s.sum(), row.R), "gap at the trim x window singletons = R(trim)"
ax.plot(d, Fm, ls=":", lw=1.2, color="0.55")
ax.plot(d, f + (1 - f) * Fm, ls="--", lw=1.2, color=INK)
ax.plot(d, Fs, lw=1.5, color=BLUE)
ax.axvline(E.TRIM, **TRIM_LINE)
ax.set_xlim(0, E.WINDOW)
ax.set_ylim(0, 1)
ax.xaxis.set_major_locator(MaxNLocator(nbins=4, steps=[1, 2, 2.5, 5, 10]))
ax.yaxis.set_major_locator(MaxNLocator(nbins=4, steps=[1, 2, 2.5, 5, 10]))
ax.tick_params(labelsize=FONT_SINGLE["tick"])
ax.set_xlabel(X_LABEL, fontsize=FONT_SINGLE["label"])
ax.set_ylabel(Y_LABEL, fontsize=FONT_SINGLE["label"])
print(f"{LIB_SET}: {s.sum():,} window singletons, interior {c['S0']:,}; f = {f:.3f} ± {se_f:.3f}, "
      f"{art:,.0f} edge artifacts; after trimming {E.TRIM}: {row.R:.1f} ± {row.se:.1f} left of "
      f"{row.kept:,.0f} kept -> FDR {100 * row.fdr:.2f}% (one-sided 95% upper bound {100 * row.fdr_up:.2f}%); "
      f"gap at {E.TRIM} = {100 * gap:.2f}% of the window's singletons")

save_figure(fig, SAVE_NAME, crop=False)
plt.show()

## Supplementary Fig. 5. Mismatch rate by the number of passes of the strand consensus

Mismatches per compared base per strand, pooled over the libraries and both strands, by the number of passes (np) of the strand consensus (Methods, Estimation of the sequencing error rate); error bars: exact 95% Poisson intervals.

In [ ]:
FIGSIZE = SINGLE_SIZE                  # width, height in inches (setup 0b); the panel fills it minus MARGINS_IN
# =============================================================================
# Supplementary Fig. 5. Mismatch rate by the number of passes (np) of the strand consensus
#
# Reads the counts of strand_mismatch_counts.py (pipeline step sequencing_errors) from
# strand_mismatches/counts/ext{EXT}/<library>/<chrom>.npz, every chromosome present, and keeps
# the site strata allowed by SITE_FILTERS.
#
# rate = mismatches / opportunities, per strand. An opportunity is a site where both strand
# consensuses of the molecule pass every filter; it counts once for each strand, at that strand's np.
# A mismatch counts for the strand carrying the non-reference base. Pooled over the libraries and both
# strands. Error bars: exact 95% Poisson interval of the bin's mismatch count, divided by the bin's
# opportunities. Not drawn: the cap bin (np >= np_max), bins with fewer than MIN_COMPARED
# opportunities and, on the log y axis, bins without mismatches.
#
# Printed: the chromosomes used; per-library rate; the rate by np.
# =============================================================================
import glob
import os

# ── KNOBS ────────────────────────────────────────────────────────────────────
EXT          = 450                     # read set: 450 = 450 b trimmed from each read end
SITE_FILTERS = dict(mask=True,         # drop the masked sites
                    mutation_sites=True)   # drop the sites where the same animal has a mutation
MIN_COMPARED = 1e6                     # bins with fewer opportunities are not drawn

# ── STYLE KNOBS ──────────────────────────────────────────────────────────────
MARGINS_IN   = SINGLE_MARGINS            # left, right, top, bottom, inches (setup 0b)
COLOR        = "#4C72B0"
MARKER_SIZE  = 4
ERR_LW, CAP_SIZE = 0.8, 1.8
X_LABEL      = "Number of passes (np)"
Y_LABEL      = "mismatches per base per strand"
SAVE_NAME    = "strand_mismatch_rate_by_passes"

# ── LOAD ─────────────────────────────────────────────────────────────────────
COUNTS_ROOT = SM_DIR / "counts" / f"ext{EXT}"
_strata = [s for s in range(4)
           if not (SITE_FILTERS["mask"] and s & 1) and not (SITE_FILTERS["mutation_sites"] and s & 2)]
rate_C = rate_N = None
_used, _per_lib = {}, {}
for lib in LIBRARIES:
    files = sorted(glob.glob(str(COUNTS_ROOT / lib / "T2T*.npz")))
    if not files:
        raise FileNotFoundError(f"no counts for {lib} in {COUNTS_ROOT / lib}")
    _used[lib] = [os.path.basename(f)[:-4] for f in files]
    for f in files:
        z = np.load(f)
        c = z["compared"][:, _strata].sum(axis=(0, 1))          # (np, sm, sx)
        n = z["mismatch"][:, _strata].sum(axis=(0, 1))
        rate_C = c if rate_C is None else rate_C + c
        rate_N = n if rate_N is None else rate_N + n
        _pc, _pn = _per_lib.get(lib, (0, 0))
        _per_lib[lib] = (_pc + c.sum(), _pn + n.sum())

lo_ci, hi_ci = poisson_ci(rate_N.sum(axis=(1, 2)))
_c, _n = rate_C.sum(axis=(1, 2)), rate_N.sum(axis=(1, 2))
with np.errstate(divide="ignore", invalid="ignore"):
    rates = pd.DataFrame({"compared": _c, "mismatches": _n, "rate": _n / _c, "lo": lo_ci / _c, "hi": hi_ci / _c})
rates = rates.iloc[:-1]                                         # drop the cap bin
rates = rates[(rates.compared >= MIN_COMPARED) & (rates.mismatches > 0)]

# ── PLOT ─────────────────────────────────────────────────────────────────────
W, H = FIGSIZE
left, right, top, bottom = MARGINS_IN
fig = plt.figure(figsize=FIGSIZE)
ax = fig.add_axes([left / W, bottom / H, (W - left - right) / W, (H - top - bottom) / H])
eb = ax.errorbar(rates.index, rates.rate, yerr=[rates.rate - rates.lo, rates.hi - rates.rate], fmt="o",
                 ms=MARKER_SIZE, color=COLOR, ecolor=COLOR, elinewidth=ERR_LW, capsize=CAP_SIZE, capthick=ERR_LW)
eb.lines[0].set_markeredgewidth(0)
ax.set_yscale("log")
ax.set_ylim(rates.lo[rates.lo > 0].min() / 2, rates.hi.max() * 2)
ax.set_xlabel(X_LABEL, fontsize=FONT_SINGLE["label"])
ax.set_ylabel(Y_LABEL, fontsize=FONT_SINGLE["label"])
ax.tick_params(labelsize=FONT_SINGLE["tick"])

save_figure(fig, SAVE_NAME, crop=False)
plt.show()

# ── PRINTED ──────────────────────────────────────────────────────────────────
_chroms = {tuple(v) for v in _used.values()}
print(f"ext{EXT}, site filters {SITE_FILTERS}; chromosomes used: "
      + (", ".join(next(iter(_chroms))) if len(_chroms) == 1 else str(_used)))
print("per library: opportunities (per strand, both strands summed), mismatches, rate")
for lib, (c, n) in _per_lib.items():
    print(f"  {lib:22s} {c:15,} {n:9,} {n / c:.2e}")
_c, _n = sum(v[0] for v in _per_lib.values()), sum(v[1] for v in _per_lib.values())
print(f"  {'all':22s} {_c:15,} {_n:9,} {_n / _c:.2e}")
print("\nby np:")
print(rates.to_string(formatters={"compared": "{:,.0f}".format, "mismatches": "{:,.0f}".format,
                                  "rate": "{:.2e}".format, "lo": "{:.2e}".format, "hi": "{:.2e}".format}))

## Supplementary Fig. 6. Mismatch rate of each substitution and its reverse complement

The mismatch rate of each of the 96 channels with a pyrimidine changed on the read (a) and, under each bar, of its reverse complement (b), all eight libraries. Uses `strand_mismatches()` (setup 0d); prints the expected false mutations of each library.

In [ ]:
FIGSIZE = (PAGE_WIDTH, 3.6)            # width, height in inches; the panels fill it minus fixed margins
# =============================================================================
# Supplementary Fig. 6. Mismatch rate per channel, each change under its reverse complement
#
# Uses strand_mismatches() (setup 0d). Reads the context
# opportunity from COUNTS_DIR/ext{EXT}/{lib}.counts.npz: compared strand-bases
# by the trinucleotide as each strand consensus read shows it (5' to 3') and by
# np bin, in the site strata allowed by SITE_FILTERS, summed over LIBS.
#
# rate of a channel x[r>a]y as read = mismatches showing it / compared
# strand-bases whose read shows the trinucleotide xry.
#   a  the 96 channels with a pyrimidine changed on the read (C>N, T>N)
#   b  under each bar of a, its reverse complement: the change with a purine
#      changed on the read that a read of the opposite strand shows for the same
#      base-pair change (C[C>A]A has partner T[G>T]G), labelled with its own
#      change. Both rows are rates over all reads, not over the same molecules.
# Bars in mismatches per base per strand (x RATE_UNIT), one y axis. RATE_NP
# draws one np range only; it must start and end on bin edges of the counts.
#
# Printed: false single-molecule mutations expected if the two strands err
# independently. Every position has one read with a pyrimidine and one with a
# purine in the middle, so at a position whose pyrimidine-read context is xry,
# both strands carry the same wrong base at rate(a bar) x rate(b bar).
#   rate per position = sum over channels of opportunity(xry) x rate(a) x rate(b)
#                       / sum of opportunity over the 32 pyrimidine contexts
# within each np bin (both strands of a molecule taken to share the np bin),
# and pooled over np for comparison.
# Expected false SNVs per library: for each of the 32 pyrimidine contexts, the
# false-mutation rate of each np bin (rates from all libraries) is averaged over
# the library's own opportunities in that context by np bin, then multiplied by
# the library's callable bases in that context
# (somatic_mutation_candidates/context_opportunity.tsv; each reference triplet
# counted under its pyrimidine-centred form) and summed. Somatic SNVs per library:
# somatic_mutation_candidates.tsv with at most NOISY_CUT singletons per molecule.
# =============================================================================
from matplotlib.patches import Rectangle

M = strand_mismatches()
mm, CHANNELS, SUBS, _comp = M.mm, M.CHANNELS, M.SUBS, M.COMP
LIBS, EXT, SITE_FILTERS, COUNTS_DIR = M.LIBS, M.EXT, M.SITE_FILTERS, M.COUNTS_DIR
CONTEXT_OPP_TSV = CAND_DIR / "context_opportunity.tsv"
CANDIDATES_TSV  = CAND_DIR / "somatic_mutation_candidates.tsv"

# ------------------------------ knobs ----------------------------------------
RATE_NP = None             # None = all np; (lo, hi) = np lo..hi on bin edges, e.g. (5, 9) or (25, 60)
NOISY_CUT = 2              # somatic mutations: at most this many singletons per molecule

# ------------------------------ style knobs ----------------------------------
RT_COLORS     = {"C>A": "#02BCED", "C>G": "#010101", "C>T": "#E32926",   # COSMIC palette
                 "T>A": "#CAC9C9", "T>C": "#A1CE63", "T>G": "#EDC6C5"}
RT_MARGINS_IN = (0.64, 0.03, 0.42, 0.42)  # left, right, top, bottom (inches)
RT_GAP_IN     = 0.80       # between the panels (holds the context labels, the x label and b's title)
RT_BARW       = 0.78       # bar width
RT_YPAD       = 1.18       # headroom above the tallest bar, for the colour bands
RT_TITLE_IN   = 0.26       # panel titles: inches above the top spine
RT_YLABEL    = "mismatches per base\nper strand (×10$^{{{e}}}$)"   # y label of each panel; {e} the exponent of RATE_UNIT
RATE_UNIT     = 1e-6       # y axis in multiples of this
RT_SAVE_NAME  = "strand_mismatch_rate_both_strands"

# ------------------------------ opportunity and counts -----------------------
_strata = [s for s in range(4)
           if not (SITE_FILTERS["mask"] and s & 1) and not (SITE_FILTERS["mutation_sites"] and s & 2)]
rt_C_lib = {}
for lib in LIBS:
    z = np.load(f"{COUNTS_DIR}/ext{EXT}/{lib}.counts.npz")
    rt_C_lib[lib] = z["context"][:, _strata].sum(axis=(1, 3, 4, 5))       # (np bin, 64 read trinucleotides)
rt_C = sum(rt_C_lib.values())
rt_edges = z["context_np_edges"]
_idx = {b: i for i, b in enumerate("ACGT")}
_rev = mm.err_reverse.to_numpy(dtype=bool)
_trip = np.where(_rev, mm.triplet.map(lambda s: s.translate(_comp)[::-1]), mm.triplet)
_alt = np.where(_rev, mm.err_base.str.translate(_comp), mm.err_base)
_t = np.array([16 * _idx[s[0]] + 4 * _idx[s[1]] + _idx[s[2]] for s in _trip])
_a = np.array([_idx[b] for b in _alt])
_bin = np.maximum(np.searchsorted(rt_edges, mm.err_np.to_numpy(), side="right") - 1, 0)
rt_N = np.zeros(rt_C.shape + (4,))
np.add.at(rt_N, (_bin, _t, _a), 1)

# channel -> (trinucleotide index, wrong base) for the panel a bars and their partners
_tri = lambda s: 16 * _idx[s[0]] + 4 * _idx[s[1]] + _idx[s[2]]
A_KEY = [(_tri(c[0] + c[2] + c[6]), _idx[c[4]]) for c in CHANNELS]
B_KEY = [(_tri((c[6] + c[2] + c[0]).translate(_comp)), _idx[c[4].translate(_comp)]) for c in CHANNELS]
SUBS_PUR = [s.translate(_comp) for s in SUBS]
CTX_PUR = [c[6].translate(_comp) + c[0].translate(_comp) for c in CHANNELS]


def channel_rates(C, N):
    """rates of the panel a bars and of their partners, from opportunity C (64,) and counts N (64, 4)."""
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.where(C[:, None] > 0, N / C[:, None], 0.0)
    return np.array([r[t, a] for t, a in A_KEY]), np.array([r[t, a] for t, a in B_KEY])


def false_rate(C, N):
    """rate per position of both strands showing the same wrong base, if independent."""
    ra, rb = channel_rates(C, N)
    opp = np.array([C[t] for t, _ in A_KEY])               # positions by pyrimidine-read context
    pyr = [t for t in range(64) if (t // 4) % 4 in (1, 3)]
    return (opp * ra * rb).sum() / C[pyr].sum()


if RATE_NP is None:
    _sel, _np_txt = np.ones(len(rt_edges), bool), ""
else:
    lo, hi = RATE_NP
    _upper = np.r_[rt_edges[1:] - 1, np.inf]                 # last np of each bin; the last bin is open
    _sel = (rt_edges >= lo) & ((_upper <= hi) | ((_upper == np.inf) & (hi >= rt_edges[-1])))
    if lo not in rt_edges or (hi < rt_edges[-1] and hi not in _upper) or not _sel.any():
        raise ValueError(f"RATE_NP must start on a bin edge {list(rt_edges)} and end one below an edge "
                         f"or at {rt_edges[-1]} or above")
    _np_txt = f"np {lo}–{hi}" if hi < rt_edges[-1] else f"np ≥ {lo}"
rt_A, rt_B = channel_rates(rt_C[_sel].sum(0), rt_N[_sel].sum(0))

# ------------------------------ figure ---------------------------------------
_np_sfx = f" ({_np_txt})" if _np_txt else ""
panels = [("a", f"Pyrimidine changed on the read{_np_sfx}", rt_A, SUBS, [c[0] + c[6] for c in CHANNELS]),
          ("b", f"Purine changed on the read{_np_sfx}", rt_B, SUBS_PUR, CTX_PUR)]
W, H = FIGSIZE
left, right, top, bottom = RT_MARGINS_IN
pw, ph = W - left - right, (H - top - bottom - RT_GAP_IN) / 2
fig = plt.figure(figsize=FIGSIZE)
axes = [fig.add_axes([left / W, (bottom + (1 - k) * (ph + RT_GAP_IN)) / H, pw / W, ph / H]) for k in range(2)]
ymax = max(rt_A.max(), rt_B.max()) / RATE_UNIT * RT_YPAD
for ax, (letter, label, vals, subs, ctx) in zip(axes, panels):
    ax.bar(range(96), vals / RATE_UNIT, width=RT_BARW, color=[RT_COLORS[c[2:5]] for c in CHANNELS], linewidth=0)
    ax.set_ylim(0, ymax); ax.set_xlim(-0.8, 95.8)
    ax.set_ylabel(RT_YLABEL.format(e=int(round(np.log10(RATE_UNIT)))), fontsize=FONT["label"])
    ax.tick_params(axis="y", labelsize=FONT["tick"])
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.text(0, 1 + RT_TITLE_IN / ph, rf"$\mathbf{{{letter}}}$  {label}", transform=ax.transAxes, ha="left",
            va="baseline", fontsize=FONT["title"])
    for j, sub in enumerate(subs):
        ax.add_patch(Rectangle((j * 16 - .5, ymax * 1.005), 16, ymax * .055,
                               color=RT_COLORS[SUBS[j]], clip_on=False, zorder=3))
        ax.text(j * 16 + 7.5, ymax * 1.075, sub, ha="center", va="bottom", fontsize=FONT["legend"],
                fontweight="bold", clip_on=False)
    ax.set_xticks(range(96))
    ax.set_xticklabels(ctx, fontsize=FONT["context"], family="monospace", rotation=90)
    ax.tick_params(axis="x", length=1.5, pad=1)
    ax.set_xlabel("trinucleotide context (5' and 3' base)", fontsize=FONT["label"])

save_figure(fig, RT_SAVE_NAME)
plt.show()

# ------------------------------ printed --------------------------------------
_pooled = false_rate(rt_C.sum(0), rt_N.sum(0))
_by_np = sum(rt_C[k].sum() / rt_C.sum() * false_rate(rt_C[k], rt_N[k]) for k in range(len(rt_edges)))
print(f"ext{EXT}, pooled over {len(LIBS)} libraries, site filters {SITE_FILTERS}")
print("both strands with the same wrong base, if the strands err independently, per position:")
print(f"  trinucleotide rates within np bins:  {_by_np:.2e}")
print(f"  pooled over np, for comparison:      {_pooled:.2e}")

# false-mutation rate of each pyrimidine context in each np bin
PYR_T = [t for t in range(64) if (t // 4) % 4 in (1, 3)]
_ctx_of = np.array([_tri(c[0] + c[2] + c[6]) for c in CHANNELS])
F_bin = np.zeros((len(rt_edges), 64))
for k in range(len(rt_edges)):
    ra, rb = channel_rates(rt_C[k], rt_N[k])
    np.add.at(F_bin[k], _ctx_of, ra * rb)

_opp = pd.read_csv(CONTEXT_OPP_TSV, sep="\t")
if _opp.floor.nunique() != 1:
    raise ValueError(f"{CONTEXT_OPP_TSV} holds more than one minimum coverage: {sorted(_opp.floor.unique())}")
_opp["pyr_t"] = [_tri(s if s[1] in "CT" else s.translate(_comp)[::-1]) for s in _opp.triplet]
_cand = pd.read_csv(CANDIDATES_TSV, sep="\t")
_snv = _cand[_cand.SINGLETONS_ON_MOLECULE <= NOISY_CUT].LIBRARY.value_counts()
rows = []
for lib in LIBS:
    Cl = rt_C_lib[lib].astype(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        F_lib = np.where(Cl.sum(0) > 0, (Cl * F_bin).sum(0) / Cl.sum(0), 0.0)   # library's np mix per context
    o = _opp[_opp["sample"] == lib]
    exp = float((o.callable_depth.to_numpy() * F_lib[o.pyr_t.to_numpy()]).sum())
    rows.append(dict(library=lib, callable_bases=o.callable_depth.sum(), expected_false_SNVs=exp,
                     somatic_SNVs=int(_snv.get(lib, 0))))
fe = pd.DataFrame(rows).set_index("library")
fe.loc["all"] = fe.sum()
fe["expected / somatic (%)"] = (100 * fe.expected_false_SNVs / fe.somatic_SNVs).round(2)
print(f"\nexpected false SNVs per library (minimum coverage {int(_opp.floor.iloc[0])}, at most {NOISY_CUT} singletons per molecule):")
print(fe.to_string(formatters={"callable_bases": "{:.3e}".format, "expected_false_SNVs": "{:.2f}".format,
                               "somatic_SNVs": "{:,.0f}".format}))

_lab = [f"{c} with {CTX_PUR[i][0]}[{SUBS_PUR[i // 16]}]{CTX_PUR[i][1]}" for i, c in enumerate(CHANNELS)]
_prod = pd.Series(rt_A * rt_B * np.array([rt_C.sum(0)[t] for t, _ in A_KEY]), index=_lab)
print("\nchannels contributing most (opportunity x rate a x rate b, share of the total, np pooled):")
print((_prod / _prod.sum()).sort_values(ascending=False).head(8).round(3).to_string())

## Supplementary Fig. 7. VAF of the variants on two or more molecules, with the mixture model

The fitted variants (*k*/*n*) with the germline, somatic and mixture expectations. Uses `mixture_fit()` (setup 0e), which prints the somatic fraction with its 95% CI and the expected germline singletons.

In [ ]:
FIGSIZE = SINGLE_SIZE                  # width, height in inches (setup 0b); the panel fills it minus MARGINS
# =============================================================================
# Supplementary Fig. 7. VAF of the variants on two or more molecules, with the germline, somatic and
#   mixture expectations (linear y axis). Uses mixture_fit() (setup 0e).
#   Data: k/n of the fitted variants, binned (BIN_WIDTH, one bin centred on 0.5 with CENTRE_ON_HALF);
#   error bars: sqrt(count).
# =============================================================================

X = mixture_fit()
k, n, p = X.k, X.n, X.p

# ── KNOBS ────────────────────────────────────────────────────────────────────
BIN_WIDTH        = 0.05         # width of the VAF bins
CENTRE_ON_HALF   = True         # True: a bin centred on VAF 0.5; False: bins from 0 (0-0.05, 0.05-0.10, ...)
SPREAD_OVER_STEP = True         # True: spread each k/n evenly over [(k - 1/2)/n, (k + 1/2)/n], data and model alike
COMPONENTS       = "weighted"   # "weighted": (1 - p) germline + p somatic = mixture; "alone": p = 0 and p = 1

# ── STYLE KNOBS ──────────────────────────────────────────────────────────────
X_LABEL     = "Fraction of molecules (k / n)"
Y_LABEL     = "Number of variants"
X_TICKS     = [0, 0.25, 0.5, 0.75, 1]
Y_TICKS     = [0, 50, 100, 150, 200]
Y_MAX       = 240                    # top of the y axis; raise it (and Y_TICKS) for COMPONENTS = "alone"
COLORS      = {"data": "#4C72B0",       # our data
               "mixture": "black",
               "germline": "#7B4FA0",   # germline
               "somatic": "#C8414B"}    # somatic
TOP_LEFT    = ["data", "mixture"]       # series named in the top-left corner, top to bottom
TOP_RIGHT   = ["germline", "somatic"]   # series named in the top-right corner, top to bottom
LINE_W      = {"mixture": 1.2, "germline": 1.0, "somatic": 1.0}
MARKER_SIZE = 3.5                       # data points
CAP_SIZE    = 1.5                       # error-bar caps
ERR_LW      = 0.8
MARGINS     = dict(zip(("left", "right", "top", "bottom"), SINGLE_MARGINS))   # inches around the axes (setup 0b)
SAVE_NAME   = "vaf_mixture_model"

# ── BINS AND EXPECTED COUNTS ─────────────────────────────────────────────────
if CENTRE_ON_HALF:
    half = int(np.ceil(0.5 / BIN_WIDTH))
    centres = 0.5 + BIN_WIDTH * np.arange(-half, half + 1)
    edges = np.append(centres - BIN_WIDTH / 2, centres[-1] + BIN_WIDTH / 2)
else:
    edges = np.linspace(0, 1, round(1 / BIN_WIDTH) + 1)


def binned(kk, nn, weights):
    """Weights placed at kk / nn, summed in the VAF bins; spread over [(kk - 1/2)/nn, (kk + 1/2)/nn] with SPREAD_OVER_STEP."""
    if not SPREAD_OVER_STEP:
        return np.histogram(kk / nn, edges, weights=weights)[0]
    lo, hi = (kk - 0.5) / nn, (kk + 0.5) / nn
    inside = np.minimum(hi[:, None], edges[None, 1:]) - np.maximum(lo[:, None], edges[None, :-1])
    return (np.clip(inside, 0, None) / (hi - lo)[:, None] * weights[:, None]).sum(axis=0)


germline_only = np.zeros(len(edges) - 1)
somatic_only = np.zeros(len(edges) - 1)
for nn in np.unique(n):                                   # expected counts, binned like the data
    ks = np.arange(2, nn)
    m = (n == nn).sum()
    germline_only += binned(ks, np.full(len(ks), nn), m * np.exp(log_pg(ks, nn)))
    somatic_only += binned(ks, np.full(len(ks), nn), m * np.exp(log_ps(ks, nn)))
mixture = p * somatic_only + (1 - p) * germline_only
observed = binned(k, n, np.ones(len(k)))
if COMPONENTS == "weighted":
    curves = {"germline": (1 - p) * germline_only, "somatic": p * somatic_only, "mixture": mixture}
    names = {"germline": "Germline", "somatic": "Somatic"}
else:
    curves = {"germline": germline_only, "somatic": somatic_only, "mixture": mixture}
    names = {"germline": "Germline only", "somatic": "Somatic only"}
names.update(data="Observed", mixture="Mixture model")

# ── PLOT ─────────────────────────────────────────────────────────────────────
from matplotlib.lines import Line2D

W, H = FIGSIZE
fig = plt.figure(figsize=FIGSIZE)
ax = fig.add_axes([MARGINS["left"] / W, MARGINS["bottom"] / H,
                   1 - (MARGINS["left"] + MARGINS["right"]) / W, 1 - (MARGINS["bottom"] + MARGINS["top"]) / H])
handles = {}
for key in ("germline", "somatic", "mixture"):                 # mixture drawn last, on top of its components
    ax.stairs(curves[key], edges, color=COLORS[key], lw=LINE_W[key])
    handles[key] = Line2D([], [], color=COLORS[key], lw=LINE_W[key])
mid = (edges[:-1] + edges[1:]) / 2
has = observed > 0
handles["data"] = ax.errorbar(mid[has], observed[has], yerr=np.sqrt(observed[has]), fmt="o", color=COLORS["data"],
                              ms=MARKER_SIZE, capsize=CAP_SIZE, elinewidth=ERR_LW, capthick=ERR_LW, zorder=3)
handles["data"].lines[0].set_markeredgewidth(0)

ax.set_xlim(0, 1)
ax.set_ylim(0, Y_MAX)
ax.set_xticks(X_TICKS)
ax.set_xticklabels([f"{t:g}" for t in X_TICKS])
ax.set_yticks(Y_TICKS)
ax.set_xlabel(X_LABEL, fontsize=FONT_SINGLE["label"])
ax.set_ylabel(Y_LABEL, fontsize=FONT_SINGLE["label"])
ax.tick_params(labelsize=FONT_SINGLE["tick"])

# Series names in their colours: two in the top-left corner, two in the top-right corner
for keys, loc in ((TOP_LEFT, "upper left"), (TOP_RIGHT, "upper right")):
    key_box = ax.legend([handles[s] for s in keys], [names[s] for s in keys], loc=loc, frameon=False,
                        fontsize=FONT_SINGLE["legend"], handlelength=1.6, borderaxespad=0.5)
    for text, s in zip(key_box.get_texts(), keys):
        text.set_color(COLORS[s])
    ax.add_artist(key_box)

save_figure(fig, SAVE_NAME, crop=False)
plt.show()

## Supplementary Fig. 8. Concordance of the short and long libraries of the 60-week-old mouse

At the sites callable in both libraries: (a) the variants on at least two molecules seen in one or both libraries, with the numbers expected by chance; (b) the somatic mutations per bp of each library (Methods, Concordance between the short and long libraries of the 60-week-old mouse).

In [ ]:
FIGSIZE = (2 * SINGLE_SIZE[0], SINGLE_SIZE[1] + LETTER_IN)   # width, height in inches: two one-panel cells side by side under the panel letters (setup 0b)
# =============================================================================
# Supplementary Fig. 8. Concordance of the 60-week short and long libraries
#    a: variants on >= 2 molecules seen in one or both libraries (Venn); b: somatic mutations per bp
#    Both panels use only the sites callable in both libraries.
#    Input   library_concordance/joint_untrimmed_coverage.npz (checked against
#            callable_bps/trimmed/denominator.tsv); the mutation tables;
#            somatic_mutation_candidates/somatic_mutation_candidates.tsv; alignment/ (BAM_CHECK only)
#    Printed: the numbers of the text, with the expected variants missing from each library
#    (hypergeometric model); with BAM_CHECK = True, for each variant seen in one library only, the
#    molecules of the other library's BAM that carry it before any filter.
# =============================================================================
from matplotlib.patches import Circle
from scipy import stats
from scipy.optimize import brentq

# ── KNOBS ────────────────────────────────────────────────────────────────────
SHORT, LONG   = "60wk_male_ctx_2_4kb", "60wk_male_ctx_2_10kb"
MIN_COV       = 10      # coverage filter: untrimmed coverage from MIN_COV to the library's untrimmed p99
NOISY_CUTOFF  = 2       # molecules carrying more singletons than this are not counted (both panels)
MIN_MOLECULES = 2       # panel a: variants on at least this many molecules in either library
Y_UNIT        = 1e-8    # panel b: y axis in units of 10^-8 SNVs per bp
BAM_CHECK     = False   # True: print, for each variant seen in one library only, the other library's BAM molecules carrying it

# ── STYLE KNOBS ──────────────────────────────────────────────────────────────
VENN_BOX      = (0.45, 0.28, 2.45)  # panel a: left, bottom, side of its square in its cell, inches (panel b: as a one-panel figure)
LIB_NAMES     = {SHORT: "Short library", LONG: "Long library"}
COLORS        = {lib: LIB_COLORS[lib] for lib in (SHORT, LONG)}
MARKERS       = {SHORT: "o", LONG: "s"}
INK           = "#333333"
EXPECTED_INK  = "0.4"
FILL_ALPHA    = 0.35                # Venn circle fill
EDGE_LW       = 1.2                 # Venn circle outline
AREA_POWER    = 0.3                 # Venn areas proportional to count ** AREA_POWER (0: the three regions equal, 1: exact)
SHOW_EXPECTED = True                # Venn: the number expected by chance, in parentheses under each count
FIT_PAD       = 1.15                # Venn: the circles move apart until each number, enlarged by this factor, fits its region
VENN_PAD      = 0.12                # space around the circles, in radii of the larger circle
NAME_SHIFT    = 0.45                # library names: horizontal shift outwards from the circle centres, in radii
COUNT_SIZE    = FONT_SINGLE["label"]    # numbers in the Venn diagram
EXPECTED_SIZE = FONT_SINGLE["legend"]   # expected numbers in the Venn diagram
NAME_SIZE     = FONT_SINGLE["tick"]     # library names above the circles (the size of panel b's tick labels)
LABEL_SIZE    = FONT_SINGLE["label"]
TICK_SIZE     = FONT_SINGLE["tick"]
BRACKET_SIZE  = FONT_SINGLE["legend"]
MARKER_SIZE   = 5
CAP_SIZE      = 3
ERR_LW        = 1.0
SPINE_LW      = AXIS_LW
SPINE_COLOR   = "0.25"
SAVE_NAME     = "library_concordance_60wk"

# ── DATA ─────────────────────────────────────────────────────────────────────
JOINT_COV      = JOINT_DIR / "joint_untrimmed_coverage.npz"
DENOM_TSV      = CB_DIR / "denominator.tsv"
CANDIDATES_TSV = CAND_DIR / "somatic_mutation_candidates.tsv"

den = pd.read_csv(DENOM_TSV, sep="\t").query("floor == @MIN_COV").set_index("sample")
cap = {lib: int(den.loc[lib, "untrimmed_p99"]) for lib in (SHORT, LONG)}

# Both libraries' untrimmed coverage at every position outside the rate mask (n_pos), with each library's trimmed
# bases summed there (t4_sum, t10_sum), and both coverages at every call position (lookup_*); index [short, long].
# The rate mask is the RepeatMasker mask plus the sites of variants called in two or more mice (masked_sites/).
J = np.load(JOINT_COV)
win_s, win_l = slice(MIN_COV, cap[SHORT] + 1), slice(MIN_COV, cap[LONG] + 1)
assert int(J["t4_sum"][win_s, :].sum()) == int(den.loc[SHORT, "callable_depth"]), "short: callable bases differ from denominator.tsv"
assert int(J["t10_sum"][:, win_l].sum()) == int(den.loc[LONG, "callable_depth"]), "long: callable bases differ from denominator.tsv"
both_sites = int(J["n_pos"][win_s, win_l].sum())
callable_bases = {SHORT: float(J["t4_sum"][win_s, win_l].sum()), LONG: float(J["t10_sum"][win_s, win_l].sum())}
cov = pd.DataFrame({"CHROM": J["lookup_chrom"], "POS": J["lookup_pos"], "u_short": J["lookup_u4"],
                    "u_long": J["lookup_u10"], "masked": J["lookup_masked"]})


def in_both_windows(x):
    """Rows of x whose untrimmed coverage passes the coverage filter in both libraries."""
    return x.u_short.between(MIN_COV, cap[SHORT]) & x.u_long.between(MIN_COV, cap[LONG])


# a: untrimmed calls at sites outside the RepeatMasker mask, including variants called in other mice; a variant's
#    molecules in a library are its molecules carrying the non-reference allele, with <= NOISY_CUTOFF singletons in the trimmed reads
#    (SINGLETONS_ON_MOLECULE of the trimmed table, VARIANTS_TSV; 0 for a molecule not in it)
calls = pd.read_csv(UNTRIMMED_TSV, sep="\t", dtype={"CHROM": str, "ZMW": str}, keep_default_na=False, low_memory=False,
                    usecols=["CHROM", "POS", "REF", "ALT", "LIBRARY", "ANIMAL", "MULTIALLELIC_SITE", "ZMW",
                             "IN_REPEATMASKER_MASK", "SHARED_WITH_LIBRARIES"])
animal_of = dict(zip(calls.LIBRARY, calls.ANIMAL))
calls = calls[calls.LIBRARY.isin([SHORT, LONG])].copy()
calls["other_mice"] = [any(l and animal_of[l] != animal_of[lib] for l in libs.split(","))
                       for lib, libs in zip(calls.LIBRARY, calls.SHARED_WITH_LIBRARIES)]
spm = pd.read_csv(VARIANTS_TSV, sep="\t", dtype=str, keep_default_na=False, usecols=["LIBRARY", "ZMW", "SINGLETONS_ON_MOLECULE"])
spm = spm[spm.LIBRARY.isin([SHORT, LONG])]
spm = (spm.assign(ZMW=spm.ZMW.str.split(","), SPM=spm.SINGLETONS_ON_MOLECULE.str.split(","))
       .explode(["ZMW", "SPM"]).astype({"SPM": int}).groupby(["LIBRARY", "ZMW"]).SPM.max().to_dict())
calls["molecules"] = [sum(spm.get((lib, z), 0) <= NOISY_CUTOFF for z in zmws.split(","))
                      for lib, zmws in zip(calls.LIBRARY, calls.ZMW)]
two_alleles = calls.loc[calls.MULTIALLELIC_SITE == 1, ["CHROM", "POS"]].drop_duplicates().assign(two_alleles=True)
VKEY = ["CHROM", "POS", "REF", "ALT"]
mol = (calls.pivot_table(index=VKEY, columns="LIBRARY", values="molecules", aggfunc="max")
       .reindex(columns=[SHORT, LONG]).fillna(0).astype(int).rename(columns={SHORT: "m_short", LONG: "m_long"}).reset_index())
site = calls.groupby(VKEY).agg(repeat=("IN_REPEATMASKER_MASK", "max"), other_mice=("other_mice", "any")).reset_index()
var = (mol.merge(site, on=VKEY).merge(cov.drop(columns="masked"), on=["CHROM", "POS"], how="left")
       .merge(two_alleles, on=["CHROM", "POS"], how="left"))
var = var[in_both_windows(var) & (var.repeat == 0) & var.two_alleles.isna()
          & ((var.m_short >= MIN_MOLECULES) | (var.m_long >= MIN_MOLECULES))].reset_index(drop=True)
var["k"] = var.m_short + var.m_long
in_both = int(((var.m_short > 0) & (var.m_long > 0)).sum())
only = {SHORT: int((var.m_long == 0).sum()), LONG: int((var.m_short == 0).sum())}


def p_none(k, c_y, c_x):
    """Chance that library y gets none of the k molecules carrying the non-reference allele at a site, if a molecule has the same chance of
    carrying the variant in either library: the k carriers are drawn at random from the c_x + c_y molecules
    (hypergeometric), given that the variant is in the set (MIN_MOLECULES carriers in at least one library)."""
    h = stats.hypergeom(c_x + c_y, k, c_y)
    left_out = np.arange(max(0, k - MIN_MOLECULES + 1), min(k, MIN_MOLECULES - 1) + 1)   # splits not in the set
    return h.pmf(0) / (1 - h.pmf(left_out).sum())


def at_least(ps, x):
    """P(at least x of independent events with probabilities ps)."""
    d = np.array([1.0])
    for q in ps:
        d = np.convolve(d, [1 - q, q])
    return float(d[x:].sum())


p_miss = {SHORT: np.array([p_none(k, s, l) for k, s, l in zip(var.k, var.u_short, var.u_long)]),
          LONG:  np.array([p_none(k, l, s) for k, s, l in zip(var.k, var.u_short, var.u_long)])}


def alt_molecules_in_bam(lib, chrom, pos, alt):
    """Molecules (ZMWs) of lib whose two strand consensus reads both carry alt at pos, among all primary
    alignments in the BAM (before any filter), with their strand lengths."""
    found = {}
    with pysam.AlignmentFile(BAM_DIR / f"{lib}.sorted.bam") as bam:
        for read in bam.fetch(chrom, pos - 1, pos):
            if read.is_secondary or read.is_supplementary:
                continue
            for q, r in read.get_aligned_pairs(matches_only=True):
                if r == pos - 1:
                    if read.query_sequence[q] == alt:
                        found.setdefault(read.get_tag("zm"), []).append(read.query_length)
                    break
    return {zmw: lengths for zmw, lengths in found.items() if len(lengths) == 2}


# b: somatic SNVs (the pipeline's set at NOISY_CUTOFF) at the sites callable in both libraries, outside the rate mask
cand = pd.read_csv(CANDIDATES_TSV, sep="\t", dtype={"CHROM": str})
snv = cand[cand.LIBRARY.isin([SHORT, LONG]) & (cand.SINGLETONS_ON_MOLECULE <= NOISY_CUTOFF)].merge(cov, on=["CHROM", "POS"], how="left")
snv_in = in_both_windows(snv) & ~snv.masked.astype(bool)
n_snv = {lib: int((snv_in & (snv.LIBRARY == lib)).sum()) for lib in (SHORT, LONG)}


def rate_ci(n, d):
    """Exact (Garwood) 95% CI of the rate n / d."""
    lo = stats.chi2.ppf(0.025, 2 * n) / 2 / d if n > 0 else 0.0
    return lo, stats.chi2.ppf(0.975, 2 * n + 2) / 2 / d


rate = {lib: n_snv[lib] / callable_bases[lib] for lib in (SHORT, LONG)}
test = stats.binomtest(n_snv[LONG], n_snv[SHORT] + n_snv[LONG],
                       callable_bases[LONG] / (callable_bases[SHORT] + callable_bases[LONG]))
pl, ph = test.proportion_ci(0.95, method="exact")
ratio = rate[LONG] / rate[SHORT]
odds = callable_bases[SHORT] / callable_bases[LONG]
ratio_ci = (pl / (1 - pl) * odds, ph / (1 - ph) * odds)


# Venn geometry: areas proportional to count ** AREA_POWER; the larger circle has radius 1
def lens_area(r1, r2, d):
    """Area of the intersection of two circles of radii r1 and r2 whose centres are d apart."""
    if d >= r1 + r2:
        return 0.0
    if d <= abs(r1 - r2):
        return np.pi * min(r1, r2) ** 2
    a1 = r1 ** 2 * np.arccos((d ** 2 + r1 ** 2 - r2 ** 2) / (2 * d * r1))
    a2 = r2 ** 2 * np.arccos((d ** 2 + r2 ** 2 - r1 ** 2) / (2 * d * r2))
    return a1 + a2 - 0.5 * np.sqrt((-d + r1 + r2) * (d + r1 - r2) * (d - r1 + r2) * (d + r1 + r2))


part = {"short": only[SHORT] ** AREA_POWER, "both": in_both ** AREA_POWER, "long": only[LONG] ** AREA_POWER}
area = {SHORT: part["short"] + part["both"], LONG: part["long"] + part["both"]}
unit = np.pi / max(area.values())                                  # area of the larger circle = pi
radius = {lib: np.sqrt(area[lib] * unit / np.pi) for lib in (SHORT, LONG)}
r_s, r_l = radius[SHORT], radius[LONG]
gap0 = brentq(lambda d: lens_area(r_s, r_l, d) - part["both"] * unit, abs(r_s - r_l) + 1e-9, r_s + r_l - 1e-9)
counts = {"short": only[SHORT], "both": in_both, "long": only[LONG]}
expected = {"short": p_miss[LONG].sum(), "long": p_miss[SHORT].sum()}   # seen only in the short library = missing from the long
expected["both"] = len(var) - expected["short"] - expected["long"]

# ── PLOT ─────────────────────────────────────────────────────────────────────
W, H = FIGSIZE
fig, (ax_a, ax_b), cell_left = single_cells(FIGSIZE, "ab")


def box(left, bottom, side):
    return [left / W, bottom / H, side / W, side / H]


def draw_venn(ax, gap):
    """Draw the Venn diagram with the circle centres gap apart; return the centres and the number texts by region."""
    ax.cla()
    centre = {SHORT: -gap / 2, LONG: gap / 2}
    for lib in (SHORT, LONG):
        ax.add_patch(Circle((centre[lib], 0), radius[lib], facecolor=COLORS[lib], alpha=FILL_ALPHA, lw=0))
        ax.add_patch(Circle((centre[lib], 0), radius[lib], facecolor="none", edgecolor=COLORS[lib], lw=EDGE_LW))
        shift = -NAME_SHIFT if lib == SHORT else NAME_SHIFT
        ax.text(centre[lib] + shift, max(r_s, r_l) + 0.06, LIB_NAMES[lib], ha="center", va="bottom",
                fontsize=NAME_SIZE, color=INK)
    lens = (centre[LONG] - r_l, centre[SHORT] + r_s)               # x range of the intersection
    count_x = {"short": (centre[SHORT] - r_s + lens[0]) / 2, "both": sum(lens) / 2,
               "long": (lens[1] + centre[LONG] + r_l) / 2}
    texts = {}
    for region in ("short", "both", "long"):
        texts[region] = [ax.text(count_x[region], 0.08 if SHOW_EXPECTED else 0, f"{counts[region]:,}",
                                 ha="center", va="center", fontsize=COUNT_SIZE, color=INK)]
        if SHOW_EXPECTED:
            texts[region].append(ax.text(count_x[region], -0.2, f"({expected[region]:.1f})", ha="center",
                                         va="center", fontsize=EXPECTED_SIZE, color=EXPECTED_INK))
    half = max(centre[LONG] + r_l, r_s - centre[SHORT]) + VENN_PAD
    y_mid = 0.18                                                   # room above the circles for the names
    ax.set_xlim(-half, half)
    ax.set_ylim(y_mid - half, y_mid + half)
    ax.axis("off")
    return centre, texts


def numbers_fit(ax, centre, texts):
    """True if every number, enlarged by FIT_PAD, lies inside its own region of the Venn diagram."""
    fig.canvas.draw()
    to_data = ax.transData.inverted()
    for region, region_texts in texts.items():
        for t in region_texts:
            bb = t.get_window_extent().expanded(FIT_PAD, FIT_PAD)
            for x, y in to_data.transform([[bb.x0, bb.y0], [bb.x0, bb.y1], [bb.x1, bb.y0], [bb.x1, bb.y1]]):
                in_s = (x - centre[SHORT]) ** 2 + y ** 2 <= r_s ** 2
                in_l = (x - centre[LONG]) ** 2 + y ** 2 <= r_l ** 2
                if (in_s, in_l) != {"short": (True, False), "both": (True, True), "long": (False, True)}[region]:
                    return False
    return True


# a: Venn diagram (the square box and equal x and y spans keep the circles round); the circles move apart until
#    every number fits its region
ax = ax_a
ax.set_position(box(cell_left[0] + VENN_BOX[0], VENN_BOX[1], VENN_BOX[2]))
gap = gap0
centre, texts = draw_venn(ax, gap)
while not numbers_fit(ax, centre, texts) and gap < r_s + r_l - 0.05:
    gap += 0.01
    centre, texts = draw_venn(ax, gap)

# b: somatic SNVs per bp at the sites callable in both
ax = ax_b
tops = []
for x, lib in enumerate((SHORT, LONG)):
    lo, hi = rate_ci(n_snv[lib], callable_bases[lib])
    y = rate[lib] / Y_UNIT
    eb = ax.errorbar([x], [y], yerr=[[y - lo / Y_UNIT], [hi / Y_UNIT - y]], fmt=MARKERS[lib], color=COLORS[lib],
                     ms=MARKER_SIZE, capsize=CAP_SIZE, elinewidth=ERR_LW, capthick=ERR_LW, zorder=3)
    eb.lines[0].set_markeredgewidth(0)
    tops.append(hi / Y_UNIT)
y_top = max(tops) * 1.12
ax.set_ylim(0, max(tops) * 1.45)
ax.plot([0, 0, 1, 1], [y_top - 0.06, y_top, y_top, y_top - 0.06], color=INK, lw=1.0)
p_text = f"P = {test.pvalue:.2g}" if test.pvalue >= 0.001 else f"P = {test.pvalue:.1e}"
ax.text(0.5, y_top + 0.04, f"×{ratio:.2f}, {p_text}", ha="center", va="bottom", fontsize=BRACKET_SIZE, color=INK)
ax.set_xlim(-0.6, 1.6)
ax.set_xticks([0, 1])
ax.set_xticklabels([LIB_NAMES[lib] for lib in (SHORT, LONG)])
ax.set_ylabel("Point mutations per bp (×10$^{-8}$)", fontsize=LABEL_SIZE)
ax.tick_params(labelsize=TICK_SIZE, color=SPINE_COLOR)
for spine in ax.spines.values():
    spine.set_color(SPINE_COLOR)
    spine.set_linewidth(SPINE_LW)

save_figure(fig, SAVE_NAME, crop=False)
plt.show()

# ── NUMBERS FOR THE TEXT ─────────────────────────────────────────────────────
print(f"sites callable in both libraries: {both_sites:,} ({both_sites / 2_641_495_036:.1%} of the autosomes); "
      f"untrimmed coverage {MIN_COV}–{cap[SHORT]} (short) and {MIN_COV}–{cap[LONG]} (long)")
print(f"\na. {len(var)} variants on >= {MIN_MOLECULES} molecules in either library ({int(var.other_mice.sum())} also called "
      f"in another mouse): {in_both} in both ({in_both / len(var):.1%}), {only[SHORT]} only in the short library, "
      f"{only[LONG]} only in the long library")
vaf = var.k / (var.u_short + var.u_long)
private = (var.m_short == 0) | (var.m_long == 0)
print(f"   pooled VAF >= 0.3: {int((vaf >= 0.3).sum())}, all in both: {bool((~private)[vaf >= 0.3].all())}; "
      f"the variants seen in one library only are carried by {var.k[private].min()}–{var.k[private].max()} molecules, "
      f"pooled VAF {vaf[private].min():.2f}–{vaf[private].max():.2f}")
for lib in (SHORT, LONG):
    o = int((var.m_short == 0).sum()) if lib == SHORT else int((var.m_long == 0).sum())
    print(f"   missing from the {LIB_NAMES[lib].lower()}: {o} observed, {p_miss[lib].sum():.1f} expected, "
          f"P(>= {o}) = {at_least(p_miss[lib], o):.2f}")
print(f"   Venn circle centres {gap:.2f} radii apart ({gap0:.2f} from AREA_POWER; moved apart for the numbers: {gap > gap0})")
if BAM_CHECK:
    for r in var[private].itertuples():
        lib = SHORT if r.m_short == 0 else LONG            # the library without the variant
        found = alt_molecules_in_bam(lib, r.CHROM, r.POS, r.ALT)
        lo_len, hi_len = length_bounds(lib)
        note = "; ".join(f"ZMW {z}: strand lengths {a:,} and {b:,} b (length bounds {lo_len:,}–{hi_len:,} b)"
                         for z, (a, b) in found.items())
        print(f"   {r.CHROM}:{r.POS:,} {r.REF}>{r.ALT}, missing from the {LIB_NAMES[lib].lower()}: "
              f"{r.m_short} / {r.u_short} short, {r.m_long} / {r.u_long} long molecules; "
              f"molecules carrying it in its BAM: {len(found)}" + (f" ({note})" if note else ""))
print(f"\nb. somatic SNVs per bp at the sites callable in both libraries")
for lib in (SHORT, LONG):
    lo, hi = rate_ci(n_snv[lib], callable_bases[lib])
    print(f"   {LIB_NAMES[lib]}: {n_snv[lib]} SNVs / {callable_bases[lib]:.3e} callable bases = {rate[lib]:.3e} [{lo:.3e}, {hi:.3e}]")
print(f"   long / short = {ratio:.2f} [{ratio_ci[0]:.2f}, {ratio_ci[1]:.2f}], exact conditional test P = {test.pvalue:.2f}")

## Supplementary Fig. 9. Coverage distribution for each library

The fraction of the covered autosomal sites at each untrimmed coverage, one panel per library; dashed line: the mean coverage of the sites with coverage of at least 1.

In [ ]:
FIGSIZE = (PAGE_WIDTH, 3.25)          # width, height in inches; the panels fill it minus fixed margins
# =============================================================================
# Supplementary Fig. 9. Coverage distribution — one panel per library, all autosomal sites, untrimmed reads
#   Input   variant_calling/ext0/<library>/informative_coverage_dist.tsv
# =============================================================================

# ── STYLE KNOBS ──────────────────────────────────────────────────────────────
X_MAX        = 35               # x-axis limit (molecules)
Y_HEADROOM   = 1.25             # y-axis top as a multiple of the tallest bar (room for the mean text)
Y_LABEL      = "Fraction of sites"
X_LABEL      = "Coverage (molecules)"
FILL_COLOR   = "#9FD3CD"        # histogram fill
EDGE_COLOR   = "#2A9D8F"        # histogram outline
EDGE_LW      = 0.8
MEAN_COLOR   = "#C8414B"        # mean line and its text
MEAN_STYLE   = "--"
MEAN_LW      = 1.0
SPINE_COLOR  = "0.25"
SAVE_NAME    = "coverage_distribution"

# ── DATA ─────────────────────────────────────────────────────────────────────
COV_DIR = VC_DIR / "ext0"                                        # <library>/informative_coverage_dist.tsv (untrimmed reads)


def coverage_fractions(lib):
    """Fraction of the sites with coverage >= 1 at each coverage 0..max (0 at coverage 0),
    and their mean coverage."""
    d = pd.read_csv(COV_DIR / lib / "informative_coverage_dist.tsv", sep="\t")
    n = np.zeros(d.coverage.max() + 1)
    n[d.coverage.to_numpy()] = d.n_sites.to_numpy()
    n[0] = 0
    frac = n / n.sum()
    mean = (np.arange(len(n)) * n).sum() / n.sum()
    return frac, mean


# ── PLOT ─────────────────────────────────────────────────────────────────────
edges = np.arange(0.5, X_MAX + 1)           # one bin per coverage value, from 1
fig, axes = library_grid(FIGSIZE, margins=(0.58, 0.08, 0.22, 0.36))
y_top = 0.0

for i, row in enumerate(GRID):
    for j, lib in enumerate(row):
        ax = axes[i, j]
        frac, mean = coverage_fractions(lib)

        heights = np.zeros(X_MAX)
        shown = frac[1:X_MAX + 1]
        heights[:len(shown)] = shown
        y_top = max(y_top, heights.max())
        ax.stairs(heights, edges, fill=True, color=FILL_COLOR, lw=0)
        ax.stairs(heights, edges, color=EDGE_COLOR, lw=EDGE_LW)

        ax.axvline(mean, color=MEAN_COLOR, lw=MEAN_LW, ls=MEAN_STYLE)
        ax.text(0.96, 0.95, f"Mean: {mean:.1f}", transform=ax.transAxes, ha="right", va="top",
                fontsize=FONT["legend"], color=MEAN_COLOR)
        for spine in ax.spines.values():
            spine.set_color(SPINE_COLOR)
        ax.tick_params(color=SPINE_COLOR)

axes[0, 0].set_xlim(0, X_MAX)
axes[0, 0].set_ylim(0, y_top * Y_HEADROOM)
axes[0, 0].xaxis.set_major_locator(ticker.MultipleLocator(10))
axes[0, 0].yaxis.set_major_locator(ticker.MultipleLocator(0.025))
axes[0, 0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:g}"))
grid_labels(axes, X_LABEL, Y_LABEL, ylabel_in=0.41)

save_figure(fig, SAVE_NAME, crop=False)
plt.show()

## Supplementary Fig. 10. Coverage at the somatic and the germline or early embryonic mutations

For each library, the untrimmed coverage of its somatic mutations and of its germline or early embryonic mutations (`final_variants()`, setup 0a), over the untrimmed coverage of the unmasked sites; grey lines: the coverage filter.

In [ ]:
FIGSIZE = (PAGE_WIDTH, 3.25 + 0.36)   # width, height in inches; the read-length figure's, plus LEGEND_IN for the legend
# =============================================================================
# Supplementary Fig. 10. Coverage at the somatic and the germline or early embryonic mutations — one panel
#   per library, untrimmed coverage, over the untrimmed coverage of the unmasked sites
#   Input   callable_bps/trimmed/coverage_tables/<library>.npz; the mutation tables
# =============================================================================
from matplotlib.colors import to_rgba
from matplotlib.patches import Patch

# ── KNOBS ────────────────────────────────────────────────────────────────────
NOISY_CUTOFF           = 2                      # maximum singletons per molecule (somatic and germline)
GERMLINE_MIN_MOLECULES = 2                      # germline candidates: at least this many molecules carrying the non-reference allele
MIN_COV                = 10                     # minimum untrimmed coverage of callable sites
GENOME_MIN_COV         = 10                     # unmasked sites: from this coverage (1 or 10)
SHOW_WINDOW            = True                   # grey lines at MIN_COV and the library's coverage cap

# ── STYLE KNOBS ──────────────────────────────────────────────────────────────
LEGEND_IN      = 0.36           # extra height under the panels, holding the legend (inches); more: more space above the legend
LEGEND_Y_IN    = 0.16           # height of the legend's centre above the bottom of the figure (inches)
X_MAX          = 35             # x-axis limit (molecules)
Y_LABEL        = "Fraction of sites"
X_LABEL        = "Coverage (molecules)"
GENOME_COLOR   = "#2A9D8F"      # unmasked sites (steps)
GENOME_LW      = 1.0
SOMATIC_COLOR  = "#C8414B"
GERMLINE_COLOR = "#7B4FA0"
FILL_ALPHA     = 0.3            # fill of the two mutation sets
EDGE_LW        = 0.9
WINDOW_COLOR   = "0.4"          # grey lines of the callable window
WINDOW_LW      = 0.8
GENOME_LABEL   = "Unmasked sites"
SOMATIC_LABEL  = "Somatic mutations"
GERMLINE_LABEL = "Germline or early embryonic mutations"
SPINE_COLOR    = "0.25"
SAVE_NAME      = "coverage_at_mutations"

# ── DATA ─────────────────────────────────────────────────────────────────────
COV_TABLES = CB_DIR / "coverage_tables"                            # <library>.npz


def genome_fractions(lib):
    """Untrimmed coverage of the unmasked sites: the fraction of the sites with coverage
    >= max(GENOME_MIN_COV, 1) at each coverage 0..64 (0 below it)."""
    with np.load(COV_TABLES / f"{lib}.npz") as z:
        n = z["hist"][0].sum(axis=(0, 1)).astype(float)     # [mask 0, triplet, trimmed, untrimmed] -> untrimmed
    n[:max(GENOME_MIN_COV, 1)] = 0
    return n / n.sum()


def fractions(cov):
    """Fraction of the sites at each coverage 1..X_MAX."""
    return np.bincount(cov, minlength=X_MAX + 1)[1:X_MAX + 1] / len(cov)


variants = final_variants(NOISY_CUTOFF, MIN_COV, GERMLINE_MIN_MOLECULES)
somatic  = variants[variants.TYPE == "Somatic SNV"]
germline = variants[variants.TYPE == "Germline candidate"]
cap = (pd.read_csv(VARIANTS_TSV, sep="\t", usecols=["LIBRARY", "UNTRIMMED_LIBRARY_COVERAGE_P99"])
       .groupby("LIBRARY").UNTRIMMED_LIBRARY_COVERAGE_P99.first().to_dict())   # the cap the filter applies

# ── PLOT ─────────────────────────────────────────────────────────────────────
edges = np.arange(0.5, X_MAX + 1)           # one bin per coverage value, from 1
fig, axes = library_grid(FIGSIZE, margins=(0.52, 0.03, 0.22, 0.36 + LEGEND_IN))
summary = []

for i, row in enumerate(GRID):
    for j, lib in enumerate(row):
        ax = axes[i, j]
        genome_h = genome_fractions(lib)[1:X_MAX + 1]
        som_cov  = somatic.loc[somatic.LIBRARY == lib, "UNTRIMMED_COVERAGE"].to_numpy()
        germ_cov = germline.loc[germline.LIBRARY == lib, "UNTRIMMED_COVERAGE"].to_numpy()

        for cov, color in ((germ_cov, GERMLINE_COLOR), (som_cov, SOMATIC_COLOR)):
            h = fractions(cov)
            ax.stairs(h, edges, fill=True, color=to_rgba(color, FILL_ALPHA), lw=0)
            ax.stairs(h, edges, color=color, lw=EDGE_LW)
        ax.stairs(genome_h, edges, color=GENOME_COLOR, lw=GENOME_LW)
        if SHOW_WINDOW:
            for x in (MIN_COV, cap[lib]):
                ax.axvline(x, color=WINDOW_COLOR, lw=WINDOW_LW)
        for spine in ax.spines.values():
            spine.set_color(SPINE_COLOR)
        ax.tick_params(color=SPINE_COLOR)

        summary.append({"Library": FIG_LABELS[lib],
                        "Somatic mutations": len(som_cov), "Germline or early embryonic": len(germ_cov),
                        "Median coverage, somatic": float(np.median(som_cov)),
                        "Median coverage, germline": float(np.median(germ_cov)),
                        "Median coverage, unmasked sites": int(np.searchsorted(np.cumsum(genome_fractions(lib)), 0.5)),
                        "Coverage cap": int(cap[lib])})

axes[0, 0].set_xlim(0, X_MAX)
axes[0, 0].xaxis.set_major_locator(ticker.MultipleLocator(10))
axes[0, 0].yaxis.set_major_locator(ticker.MultipleLocator(0.05))
axes[0, 0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:g}"))
grid_labels(axes, X_LABEL, Y_LABEL)

# legend in one row under the panels
fig.legend(handles=[Line2D([], [], color=GENOME_COLOR, lw=GENOME_LW),
                    Patch(facecolor=to_rgba(SOMATIC_COLOR, FILL_ALPHA), edgecolor=SOMATIC_COLOR, lw=EDGE_LW),
                    Patch(facecolor=to_rgba(GERMLINE_COLOR, FILL_ALPHA), edgecolor=GERMLINE_COLOR, lw=EDGE_LW)],
           labels=[GENOME_LABEL, SOMATIC_LABEL, GERMLINE_LABEL], loc="center",
           bbox_to_anchor=(0.5, LEGEND_Y_IN / FIGSIZE[1]), ncol=3, fontsize=FONT["legend"],
           frameon=False, handlelength=1.6, columnspacing=2.0)

save_figure(fig, SAVE_NAME, crop=False)
plt.show()
display(pd.DataFrame(summary))

## Supplementary Fig. 11. Point mutations per diploid genome vs age, by minimum coverage

One panel per minimum untrimmed coverage (5 to 10): the somatic mutations and callable base pairs at that minimum coverage, per animal, with the OLS fit and its 95% confidence band.

In [ ]:
FIGSIZE = (PAGE_WIDTH, 4.2)            # width, height in inches; the panels fill it minus margins and gaps
# =============================================================================
# Supplementary Fig. 11. Somatic point mutations per diploid genome vs age, one panel per minimum coverage
#
# For each minimum coverage in FLOORS, the somatic mutations of final_variants() (setup 0a) with that
# minimum untrimmed coverage, and the callable base pairs at the same minimum coverage
# (callable_bps/trimmed/denominator.tsv), summed per animal (the two 60 wk libraries pooled).
#
#   y       somatic mutations / callable base pairs x DIPLOID_BP (both copies of the T2T
#           autosomes): no trinucleotide correction
#   fit     OLS on the animals, with the 95% confidence band of the line across
#           our ages; the line runs dashed (line only) back to age 0, where it
#           meets the y axis at the intercept
#   legend  upper left: the fit's key (the band as a shaded box with the line
#           through it), the slope per year and its P (two-sided t test, slope = 0)
#   R2      in black, lower right, R2_PAD_PT_MC above the x axis
#
# Every panel: x from age 0, and a broken y axis from 0 (a
# short strip at the bottom holds the 0 tick; the main panel starts at Y_BREAK_MC;
# break marks on the LEFT spine only; the right spine unbroken). All panels share
# one y range. Its top is solved so that, in every panel, what is drawn under the
# legend clears it by LEGEND_CLEAR_IN_MC. Panels a, b, ... in reading order,
# NCOL_MC per row, titled "Minimum coverage N"; tick labels on the outer panels
# only, and a y label for each row. The panels fill FIGSIZE minus the margins and gaps; the cell checks that
# no two pieces of text overlap. The saved figure is cropped to its content.
# =============================================================================
import statsmodels.api as sm
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator, MaxNLocator

# ------------------------------ knobs ----------------------------------------
SAVE_NAME_MC  = "somatic_snvs_per_genome_vs_age_by_min_cov"
FLOORS        = [5, 6, 7, 8, 9, 10]      # minimum untrimmed coverages, one panel each
NOISY_CUTOFF  = 2                        # maximum singletons per molecule
WEEKS_PER_YEAR = 365 / 7                 # a year is 365 days
NCOL_MC       = 3                        # panels per row: 3 gives two rows of three
TITLE_MC      = "Minimum coverage {floor}"
YLABEL_MC     = "Point mutations per\ndiploid genome"   # one per row; \n breaks the line so it fits the row
XLABEL_MC     = "Age (weeks)"
FIT_LABEL_MC  = "slope = {slope:.1f} per year\np = {p}"   # {slope}, {lo}, {hi} per year; {p} as text; \n breaks the line
R2_LABEL_MC   = "R² = {r2:.2f}"         # lower right, in black; None: no R2
XLIM_MC       = (0, 74)                  # weeks: age 0 (the intercept) to just past the oldest animals
Y_BREAK_MC    = 40                       # the main panels start here (SNVs per diploid genome); None: Y_PAD_MC below the data
Y_TICKS_MC    = [50, 75, 100]            # main panels' y ticks; None: chosen automatically
Y_PAD_MC      = 0.05                     # share of the data range left above the highest thing drawn (at least)
LEGEND_CLEAR_IN_MC = 0.06                # what is drawn under a legend sits at least this far below it, inches

# ---- layout, inches ----
MARGINS_MC    = (0.62, 0.05, 0.22, 0.40) # left, right, top, bottom
COL_GAP_MC    = 0.15                     # between columns
ROW_GAP_MC    = 0.40                     # between rows (holds the titles)
LOW_IN_MC     = 0.17                     # height of the strip holding the 0 tick
GAP_IN_MC     = 0.06                     # gap between the strip and the main panel
BREAK_IN_MC   = 0.045                    # half-length of each break mark
TITLE_IN_MC   = 0.07                     # title baseline above the top spine

# ---- style ----
C_MC          = "#4C72B0"                # our blue: points, line, band
MARKER_MC, LINE_LW_MC, BAND_ALPHA_MC = 4.5, 1.1, 0.15
EXT_LS_MC     = (0, (4, 2.5))            # the line back to age 0: dashed
TITLE_FONT_MC, LABEL_FONT_MC, TICK_FONT_MC, LEGEND_FONT_MC = FONT["title"], FONT["label"], FONT["tick"], FONT["legend"]
AXIS_LW_MC, TICK_LEN_MC = AXIS_LW, TICK_LEN
R2_PAD_PT_MC  = (5, 4)                   # R2 text inset from the right spine and from the x axis, points

# ---- data: somatic SNVs and callable base pairs per animal at each minimum coverage ----
_den_mc = pd.read_csv(CB_DIR / "denominator.tsv", sep="\t")
DIPLOID_BP = 2 * int(_den_mc.autosome_bp.iloc[0])            # both copies of the T2T autosomes
_rows = []
for F in FLOORS:
    _fv = final_variants(NOISY_CUTOFF, F)
    _n = _fv[_fv.TYPE == "Somatic SNV"].groupby("LIBRARY").size()
    _rows += [dict(sample=lib, floor=F, singletons=int(_n.get(lib, 0))) for lib in LIBRARIES]
ani_spm = pd.DataFrame(_rows).merge(_den_mc[["sample", "floor", "callable_depth"]], on=["sample", "floor"],
                                    how="left", validate="1:1")
assert ani_spm.callable_depth.notna().all(), "denominator.tsv lacks a library or minimum coverage"
ani_spm["animal"] = ani_spm["sample"].str.replace(r"_\d+kb$", "", regex=True)   # the 60 wk libraries: one animal
ani_spm["age"] = ani_spm["sample"].map(AGE_WK)
ani_spm = (ani_spm.groupby(["animal", "age", "floor"], as_index=False)
                  .agg(singletons=("singletons", "sum"), callable_depth=("callable_depth", "sum")))
ani_spm["rate"] = ani_spm.singletons / ani_spm.callable_depth


def _mc_tint(color, alpha):
    """color laid over white at this opacity, as an opaque colour."""
    return tuple(1 - alpha * (1 - c) for c in to_rgb(color))


class _McFitKey:
    """Legend key for the fit: the 95% band as a shaded box with the line through its middle."""
    def legend_artist(self, legend, orig_handle, fontsize, handlebox):
        x0, y0 = -handlebox.xdescent, -handlebox.ydescent
        w, h = handlebox.width, handlebox.height
        box = Rectangle((x0, y0), w, h, facecolor=_mc_tint(C_MC, BAND_ALPHA_MC), lw=0,
                        transform=handlebox.get_transform())
        handlebox.add_artist(box)
        handlebox.add_artist(plt.Line2D([x0, x0 + w], [y0 + h / 2] * 2, color=C_MC, lw=LINE_LW_MC,
                                        solid_capstyle="butt", transform=handlebox.get_transform()))
        return box


def _mc_p(p):
    """p as 'm × 10^e' (mathtext), or plain from 0.001 up."""
    if p >= 1e-3:
        return f"{p:.2g}"
    m, e = f"{p:.1e}".split("e")
    return rf"{m} × 10$^{{{int(e)}}}$"


# ---- layout: NCOL_MC panels per row; each is a main panel over a 0-tick strip ----
W, H = FIGSIZE
L_MC, R_MC, T_MC, B_MC = MARGINS_MC
NROW_MC = -(-len(FLOORS) // NCOL_MC)
PW_MC = (W - L_MC - R_MC - (NCOL_MC - 1) * COL_GAP_MC) / NCOL_MC          # panel width
PH_MC = (H - T_MC - B_MC - (NROW_MC - 1) * ROW_GAP_MC) / NROW_MC          # panel height, strip included
MAIN_MC = PH_MC - LOW_IN_MC - GAP_IN_MC                                   # main panel height
assert PW_MC > 1 and MAIN_MC > 0.6, f"FIGSIZE {FIGSIZE} leaves too little room for the panels"

fig = plt.figure(figsize=FIGSIZE)
panels, lo, hi = [], np.inf, -np.inf
for k, F in enumerate(FLOORS):
    r, c = divmod(k, NCOL_MC)
    x0 = L_MC + c * (PW_MC + COL_GAP_MC)
    y0 = B_MC + (NROW_MC - 1 - r) * (PH_MC + ROW_GAP_MC)
    ax = fig.add_axes([x0 / W, (y0 + LOW_IN_MC + GAP_IN_MC) / H, PW_MC / W, MAIN_MC / H])
    axl = fig.add_axes([x0 / W, y0 / H, PW_MC / W, LOW_IN_MC / H], sharex=ax)

    d = ani_spm[ani_spm.floor == F].sort_values("age")
    x, y = d.age.to_numpy(float), (d.rate * DIPLOID_BP).to_numpy()
    res = sm.OLS(y, sm.add_constant(x)).fit()
    line = lambda xs, res=res: res.params[0] + res.params[1] * np.asarray(xs, float)
    band = lambda xs, res=res: res.get_prediction(sm.add_constant(np.asarray(xs, float),
                                                                  has_constant="add")).conf_int(alpha=0.05)
    inside = np.linspace(x.min(), x.max(), 200)                  # our ages: solid, with the band
    before = np.linspace(XLIM_MC[0], x.min(), 100)               # back to age 0: dashed line only
    b = band(inside)
    ax.fill_between(inside, b[:, 0], b[:, 1], color=_mc_tint(C_MC, BAND_ALPHA_MC), lw=0, zorder=1)
    ax.plot(before, line(before), color=C_MC, lw=LINE_LW_MC, ls=EXT_LS_MC, zorder=2)
    ax.plot(inside, line(inside), color=C_MC, lw=LINE_LW_MC, zorder=2)
    ax.plot(x, y, ls="", marker="o", ms=MARKER_MC, color=C_MC, mew=0, zorder=4)
    lo = min(lo, y.min(), b[:, 0].min(), line(XLIM_MC[0]))
    hi = max(hi, y.max(), b[:, 1].max())
    panels.append(dict(floor=F, row=r, col=c, ax=ax, axl=axl, res=res, d=d, x=x, y=y, line=line, band=band))
    ax.set_xlim(XLIM_MC)

# ---- the fit's legend in each panel (upper left): slope per year and its P ----
k_yr = WEEKS_PER_YEAR
for p in panels:
    res = p["res"]
    b1, (lo1, hi1) = res.params[1], res.conf_int(0.05)[1]
    label = FIT_LABEL_MC.format(slope=b1 * k_yr, lo=lo1 * k_yr, hi=hi1 * k_yr, p=_mc_p(res.pvalues[1]))
    p["ax"].legend([_McFitKey()], [label], handler_map={_McFitKey: _McFitKey()}, handlelength=2.2,
                   handleheight=1.6, loc="upper left", frameon=False, fontsize=LEGEND_FONT_MC,
                   borderaxespad=0.3, handletextpad=0.5)

# ---- one y range for every panel ----
# bottom: Y_BREAK_MC. top: at least Y_PAD_MC of the data range above the highest thing
# drawn, and, in every panel, high enough that whatever lies under the legend (line,
# band, points with their radius) sits LEGEND_CLEAR_IN_MC below the legend's bottom.
Y0_MC = lo - Y_PAD_MC * (hi - lo) if Y_BREAK_MC is None else Y_BREAK_MC
assert Y0_MC < lo, f"Y_BREAK_MC ({Y_BREAK_MC}) must lie below the lowest thing drawn ({lo:.1f})"
Y1_MC = hi + Y_PAD_MC * (hi - lo)
fig.canvas.draw()
_rend = fig.canvas.get_renderer()
for p in panels:
    ax = p["ax"]
    bb, lg = ax.get_window_extent(_rend), ax.get_legend().get_window_extent(_rend)
    depth_in = (bb.y1 - lg.y0) / fig.dpi                         # legend bottom under the top spine
    x_right = XLIM_MC[0] + (lg.x1 - bb.x0) / bb.width * (XLIM_MC[1] - XLIM_MC[0])   # legend's right edge, weeks
    xs = np.linspace(XLIM_MC[0], x_right, 300)
    xin = xs[(xs >= p["x"].min()) & (xs <= p["x"].max())]
    under = [p["line"](xs).max()] + ([p["band"](xin)[:, 1].max()] if len(xin) else [])
    pts = p["y"][p["x"] <= x_right + 1]
    need = [(max(under), depth_in + LEGEND_CLEAR_IN_MC)]
    if len(pts):
        need.append((pts.max(), depth_in + LEGEND_CLEAR_IN_MC + MARKER_MC / 2 / 72))
    for y_hi, need_in in need:                                   # (Y1 - y_hi) / (Y1 - Y0) = need_in / MAIN_MC
        g = need_in / MAIN_MC
        Y1_MC = max(Y1_MC, (y_hi - g * Y0_MC) / (1 - g))

# ---- axes, spines, breaks, titles, R2 ----
bottom_row = {c: max(p["row"] for p in panels if p["col"] == c) for c in range(NCOL_MC)}
for k, p in enumerate(panels):
    ax, axl, res = p["ax"], p["axl"], p["res"]
    ax.set_ylim(Y0_MC, Y1_MC)
    ax.yaxis.set_major_locator(FixedLocator(Y_TICKS_MC) if Y_TICKS_MC
                               else MaxNLocator(nbins=4, steps=[1, 2, 2.5, 5, 10]))
    axl.set_ylim(0, (Y1_MC - Y0_MC) / MAIN_MC * LOW_IN_MC)       # same units per inch as the main panel
    axl.yaxis.set_major_locator(FixedLocator([0]))
    for a in (ax, axl):
        a.set_xlim(XLIM_MC)
        a.tick_params(labelsize=TICK_FONT_MC, length=TICK_LEN_MC, width=AXIS_LW_MC,
                      labelleft=(p["col"] == 0))                   # y tick labels on the left column only
        for s in a.spines.values():
            s.set_linewidth(AXIS_LW_MC)
    for a, side in ((ax, "bottom"), (ax, "right"), (axl, "top"), (axl, "right")):
        a.spines[side].set_visible(False)
    ax.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
    is_bottom = p["row"] == bottom_row[p["col"]]
    axl.tick_params(axis="x", labelbottom=is_bottom)              # x tick labels on the bottom row only
    if is_bottom:
        axl.set_xlabel(XLABEL_MC, fontsize=LABEL_FONT_MC)
    pm, ps = ax.get_position(), axl.get_position()
    fig.add_artist(Line2D([pm.x1, pm.x1], [ps.y0, pm.y1], transform=fig.transFigure, color="black",
                          lw=AXIS_LW_MC, solid_capstyle="projecting"))   # the unbroken right spine
    for a, yb, h_in in ((ax, 0.0, MAIN_MC), (axl, 1.0, LOW_IN_MC)):   # break marks on the left spine
        dx, dy = BREAK_IN_MC / PW_MC, BREAK_IN_MC / h_in
        a.plot([-dx, dx], [yb - dy, yb + dy], transform=a.transAxes, color="black",
               lw=AXIS_LW_MC, clip_on=False, zorder=6, solid_capstyle="butt")

    # title: the bold letter, then "Minimum coverage N", on one baseline above the top spine
    letter = ax.text(0, 1 + TITLE_IN_MC / MAIN_MC, "abcdefghijkl"[k], transform=ax.transAxes,
                     ha="left", va="baseline", fontsize=TITLE_FONT_MC, fontweight="bold")
    ax.annotate(TITLE_MC.format(floor=p["floor"]), xy=(1, 0), xycoords=letter,
                xytext=(0.3 * TITLE_FONT_MC, 0), textcoords="offset points", ha="left", va="bottom",
                fontsize=TITLE_FONT_MC)

    # R2 in black, lower right, R2_PAD_PT_MC above the x axis
    if R2_LABEL_MC:
        axl.annotate(R2_LABEL_MC.format(r2=res.rsquared), xy=(1, 0), xycoords="axes fraction",
                     xytext=(-R2_PAD_PT_MC[0], R2_PAD_PT_MC[1]), textcoords="offset points",
                     ha="right", va="bottom", fontsize=LEGEND_FONT_MC, color="black")

# ---- a y label for each row, centred on the row, left of its tick labels ----
fig.canvas.draw()
_rend = fig.canvas.get_renderer()
_ylabs = []
for p in panels:
    if p["col"] != 0:
        continue
    _x0 = min(t.get_window_extent(_rend).x0
              for t in p["ax"].get_yticklabels() + p["axl"].get_yticklabels() if t.get_text())
    _yc = (p["axl"].get_position().y0 + p["ax"].get_position().y1) / 2
    _ylabs.append(fig.text((_x0 - 4 * fig.dpi / 72) / fig.bbox.width, _yc, YLABEL_MC,
                           fontsize=LABEL_FONT_MC, rotation=90, ha="right", va="center",
                           multialignment="center"))
    _ylabs[-1].row_span = (p["axl"].get_position().y0, p["ax"].get_position().y1)

# ---- checks: no two pieces of text overlap; all text inside the figure ----
fig.canvas.draw()
_rend = fig.canvas.get_renderer()
_boxes = [(f"y label {i + 1}", t.get_window_extent(_rend)) for i, t in enumerate(_ylabs)]
for t in _ylabs:                                                # each y label within its row's height
    _b = t.get_window_extent(_rend)
    assert _b.y0 >= t.row_span[0] * fig.bbox.height - 1 and _b.y1 <= t.row_span[1] * fig.bbox.height + 1,         "a y label is taller than its row -- shorten YLABEL_MC or enlarge FIGSIZE"
for p in panels:
    tag = f"min cov {p['floor']}"
    _boxes.append((f"{tag} legend", p["ax"].get_legend().get_window_extent(_rend)))
    for a in (p["ax"], p["axl"]):
        (x_lo, x_hi), (y_lo, y_hi) = a.get_xlim(), a.get_ylim()
        drawn = ([t for t in a.get_xticklabels() if x_lo <= t.get_position()[0] <= x_hi]     # ticks in view
                 + [t for t in a.get_yticklabels() if y_lo <= t.get_position()[1] <= y_hi])
        for t in a.texts + drawn + [a.xaxis.label]:
            if t.get_visible() and t.get_text():
                _boxes.append((f"{tag} '{t.get_text()}'", t.get_window_extent(_rend)))
    assert all(t.get_window_extent(_rend).y1 <= p["ax"].get_window_extent(_rend).y0 for t in p["axl"].texts),         f"{tag}: R2 reaches the main panel -- lower R2_PAD_PT_MC[1]"
for i in range(len(_boxes)):
    for j in range(i + 1, len(_boxes)):
        (na, a), (nb, b) = _boxes[i], _boxes[j]
        assert not a.overlaps(b), f"text overlaps: {na} and {nb} -- enlarge FIGSIZE or the gaps"
for n, b in _boxes:
    assert b.x0 >= 0 and b.y0 >= 0 and b.x1 <= fig.bbox.x1 and b.y1 <= fig.bbox.y1, \
        f"{n} runs off the figure -- widen the margins or FIGSIZE"

save_figure(fig, SAVE_NAME_MC)                          # cropped; the panels keep their size
plt.show()

# ---------------------------------- report -----------------------------------
print(f"y = somatic SNVs / callable base pairs x {DIPLOID_BP:,} (diploid T2T autosomes); "
      f"{len(panels[0]['d'])} animals per panel")
print(f"{'min cov':>7s} {'SNVs':>6s} {'intercept [95% CI]':>20s} {'slope per year [95% CI]':>25s} "
      f"{'R2':>5s} {'slope P':>9s}")
for p in panels:
    res = p["res"]
    (b0lo, b0hi), (s_lo, s_hi) = res.conf_int(0.05)
    print(f"{p['floor']:>7d} {int(p['d'].singletons.sum()):>6,} "
          f"{f'{res.params[0]:.1f} [{b0lo:.1f}, {b0hi:.1f}]':>20s} "
          f"{f'{res.params[1] * k_yr:.1f} [{s_lo * k_yr:.1f}, {s_hi * k_yr:.1f}]':>25s} "
          f"{res.rsquared:>5.3f} {res.pvalues[1]:>9.2e}")
print(f"y range of the main panels: {Y0_MC:.1f} to {Y1_MC:.1f}")

## Supplementary Fig. 12. Somatic point mutations per diploid genome vs age: ours and Grońska-Pęski et al. (2026)

Both studies corrected to the genome's trinucleotide composition (setup 0f), with the OLS fit on our seven animals and its 95% confidence band.

In [ ]:
FIGSIZE    = SINGLE_SIZE                 # width, height in inches (setup 0b); the panel is FIGSIZE minus the margins
# =============================================================================
# Supplementary Fig. 12. Somatic point mutations per diploid genome vs age: ours and Grońska-Pęski et al.
#   (2026), both corrected to the genome's trinucleotide composition. Uses setup 0f.
#   ours  per animal, sum over the 32 folded contexts of (n_c / D_c) * 2 G_c: each context's rate times
#         its count on both copies of the T2T autosomes. The plain extrapolation (mutations / callable
#         bps x 2 x the T2T autosome length) is printed beside it.
#   CR    their Table S6 burden per bp (trinucleotide-corrected) x 2 x the same T2T autosome length
#         (ACGT positions), with its 95% CI.
#   Fit   OLS on our 7 animals with the 95% band of the line; dashed back to age 0, and dashed with a
#         lighter band from our oldest animals to the right edge, past the CR point.
#   Legends
#         upper left   the fit's key, the slope per year and its P; with R2_LABEL_5 set, R2 in black
#                      under it, left-aligned with the icon. The y top is solved so that what is drawn
#                      under them clears them by AGE_LEGEND_CLEAR_IN. The report prints R2 either way.
#         lower right  framed, just above the x axis (it belongs to the 0 strip, so it can reach over
#                      the gap into the main panel): our points (OURS_LABEL) and the CR point with its
#                      error bar (CR_LABEL).
#   Broken y axis: a short strip at the bottom (AGE_LOW_IN) holds the 0 tick; above a gap (AGE_GAP_IN)
#         the main panel starts at Y_BREAK, on the same scale. Break marks cross the left spine only;
#         the right spine is unbroken. The cell stops with an error if two pieces of text overlap or a
#         legend touches the data.
# =============================================================================
from matplotlib.legend_handler import HandlerErrorbar

# ------------------------------ knobs ----------------------------------------
YLABEL_5   = "Point mutations per diploid genome"
FIT_LABEL_5 = "slope = {slope:.1f} per year, p = {p}"   # {slope}, {lo}, {hi} per year; {p} the slope's P as text
R2_LABEL_5 = None                        # e.g. "R² = {r2:.2f}": R2 under the fit's legend, left-aligned with its icon; None: no R2
OURS_LABEL = "This study"
CR_LABEL   = "Grońska-Pęski et al. (2026)"
STUDY_PAD  = 0.5                          # gap between the framed study legend and the x axis / right spine, in font sizes
STUDY_ERR  = 0.9                         # half-length of the CR error bar in that legend, in font sizes
Y_BREAK    = 42                          # the main panel starts here (mutations per diploid genome); None: AGE_Y_PAD below the data
Y_TICKS    = [50, 70, 90]                # main panel's y ticks; None: chosen automatically
X_STEP     = 20                          # weeks between x ticks
SAVE_NAME_5 = "point_mutations_per_genome_vs_age_ours_and_CR"

fit_genome = Fit(ours.age_wk, ours.per_genome)

W, H = FIGSIZE
fig = plt.figure(figsize=FIGSIZE)
ax, axl = broken_age_axes(fig, (AGE_LEFT, AGE_BOTTOM, W - AGE_LEFT - AGE_RIGHT, H - AGE_TOP - AGE_BOTTOM))
lo, hi, handles = draw_age(ax, fit_genome, ours.per_genome, cr_genome)
ax.set_xlim(AGE_XLIM)
leg_fit = fit_legend(ax, fit_genome, FIT_LABEL_5)
under_top = [leg_fit] + ([r2_under_legend(ax, fit_genome, leg_fit, R2_LABEL_5)] if R2_LABEL_5 else [])
leg_study = axl.legend([handles["ours"], handles["cr"]], [OURS_LABEL, CR_LABEL], loc="lower right",
                       borderaxespad=STUDY_PAD, frameon=True, fontsize=AGE_LEGEND_FONT, handlelength=1.6,
                       handleheight=2.2,
                       handler_map={type(handles["cr"]): HandlerErrorbar(xerr_size=0.6, yerr_size=STUDY_ERR)})

# y range: from Y_BREAK to AGE_Y_PAD above the data, higher if the fit's legend and R2 need room
tops, bottoms = drawn_edges(fit_genome, ours.per_genome, AGE_XLIM, extend_right=True, cr=cr_genome)
y0 = lo - AGE_Y_PAD * (hi - lo) if Y_BREAK is None else Y_BREAK
y_top = max(hi + AGE_Y_PAD * (hi - lo), clear_top(ax, y0, tops, under_top))
break_y(ax, axl, lo, hi, Y_TICKS, y_break=Y_BREAK, y_top=y_top)
axl.xaxis.set_major_locator(ticker.MultipleLocator(X_STEP))
ylabel_beside(fig, (ax, axl), YLABEL_5)
for art in under_top:
    check_clear(ax, art, tops, above=True)
check_clear(ax, leg_study, bottoms, above=False)
check_text_apart(fig, (ax, axl), extra=[leg_fit, leg_study])

save_figure(fig, SAVE_NAME_5, crop=False)              # saved at FIGSIZE exactly
plt.show()

# ---------------------------------- report -----------------------------------
print(f"{'animal':>17s} {'age':>4s} {'calls':>6s} {'per genome, plain':>18s} {'per genome, corrected':>22s}")
for a, v in ours.iterrows():
    print(f"{a:>17s} {int(v.age_wk):>4d} {int(v.mutations):>6,} {v.per_genome_plain:>18.1f} {v.per_genome:>22.1f}")
report_fit(fit_genome, 1.0, "per diploid genome")
print(f"CR, AL cerebellar neurons at {CR_AGE_WK} wk: {cr_genome[0]:.1f} [{cr_genome[1]:.1f}, "
      f"{cr_genome[2]:.1f}] per diploid genome")

## Supplementary Fig. 13. Number of passes for each library

Histogram of the number of passes (np) per HiFi strand consensus read, one panel per library; dashed line: the mean; grey line: the minimum number of passes of the read filter.

In [ ]:
FIGSIZE = (PAGE_WIDTH, 3.25)          # width, height in inches; the panels fill it minus fixed margins
# =============================================================================
# Supplementary Fig. 13. Number of passes (np) — one panel per library
#   Input   read_tables/<library>.reads.npz
# =============================================================================
ROW_GAP_IN = 0.60                     # between the rows: the top row's x labels and the bottom row's titles

# ── STYLE KNOBS ──────────────────────────────────────────────────────────────
X_MAX        = 61               # x-axis limit (np is capped at 60)
NP_MIN       = 5                # minimum passes of the read filter; None hides its line
Y_DIV        = 1e6              # y tick labels are drawn in units of this
Y_LABEL      = r"Reads ($\times 10^6$)"
X_LABEL      = "Number of passes"
BAR_COLOR    = "#4C72B0"        # histogram
MEAN_COLOR   = "#B5452B"        # mean line and its legend text
MEAN_STYLE   = "--"
MEAN_LW      = 1.0
FILTER_COLOR = "#595959"        # filter line
FILTER_LW    = 0.8
SAVE_NAME    = "passes_distribution"

# ── PLOT ─────────────────────────────────────────────────────────────────────
edges = np.arange(-0.5, X_MAX + 1)          # one bin per pass count
fig, axes = library_grid(FIGSIZE, gaps=(0.16, ROW_GAP_IN), top_xticks=True)

for i, row in enumerate(GRID):
    for j, lib in enumerate(row):
        ax  = axes[i, j]
        npv = load_reads(lib, ("np",))["np"]

        heights, _ = np.histogram(npv, bins=edges)
        ax.stairs(heights, edges, fill=True, color=BAR_COLOR, lw=0)

        if NP_MIN is not None:
            ax.axvline(NP_MIN, color=FILTER_COLOR, lw=FILTER_LW)

        mean = npv.mean()
        ax.axvline(mean, color=MEAN_COLOR, lw=MEAN_LW, ls=MEAN_STYLE, label=f"Mean: {mean:.1f}")
        ax.legend(loc="upper right", fontsize=FONT["legend"], labelcolor=MEAN_COLOR, handlelength=1.6,
                  handletextpad=0.5, borderaxespad=0.3, facecolor="white", edgecolor="none", framealpha=1)
        del npv

axes[0, 0].set_xlim(0, X_MAX)
axes[0, 0].xaxis.set_major_locator(ticker.MultipleLocator(20))
axes[0, 0].yaxis.set_major_locator(ticker.MultipleLocator(0.5 * Y_DIV))
axes[0, 0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v / Y_DIV:g}"))
grid_labels(axes, X_LABEL, Y_LABEL, top_xlabel=True)

save_figure(fig, SAVE_NAME, crop=False)
plt.show()

## Supplementary Fig. 14. 96-channel spectrum of the strand-consensus mismatches

All mismatches (a), and split by whether the changed base is a pyrimidine (b) or a purine (c) on the read carrying the mismatch, as fractions of all mismatches, pooled over the libraries. Uses `strand_mismatches()` (setup 0d).

In [ ]:
FIGSIZE = (PAGE_WIDTH, 4.85)           # width, height in inches; the panels fill it minus fixed margins
# =============================================================================
# Supplementary Fig. 14. 96-channel spectrum of the strand-consensus mismatches, split by the strand
# of the read carrying the mismatch
#
# Uses strand_mismatches() (setup 0d). Pooled over the libraries, not corrected for trinucleotide
# opportunity, all three panels as fractions of all mismatches (b + c = a):
#   a  all mismatches
#   b  the changed base is a pyrimidine on the read carrying the mismatch: C>N
#      or T>N as that read shows it, 5' to 3'
#   c  the changed base is a purine on the read carrying the mismatch: G>N or
#      A>N as read, each bar under the channel of its complement and labelled
#      with its own change (T[G>T]G as read is drawn under C[C>A]A)
#
# Printed: per substitution class, the mismatches of each panel; how often the
# wrong base equals the base before it (5') or after it (3') on the read carrying
# the mismatch.
# =============================================================================
from matplotlib.patches import Rectangle

M = strand_mismatches()
mm, CHANNELS, SUBS, _comp, LIBS, EXT, SITE_FILTERS = M.mm, M.CHANNELS, M.SUBS, M.COMP, M.LIBS, M.EXT, M.SITE_FILTERS

# ------------------------------ style knobs ----------------------------------
SPEC_COLORS     = {"C>A": "#02BCED", "C>G": "#010101", "C>T": "#E32926",   # COSMIC palette
                   "T>A": "#CAC9C9", "T>C": "#A1CE63", "T>G": "#EDC6C5"}
SPEC_MARGINS_IN = (0.68, 0.03, 0.42, 0.42)  # left, right, top, bottom (inches)
SPEC_GAPS_IN    = (0.48, 0.80)  # between a and b (no context labels under a), between b and c
SPEC_BARW       = 0.78          # bar width
SPEC_YPAD       = 1.18          # headroom above the tallest bar, for the colour bands
SPEC_TITLE_IN   = 0.26          # panel titles: inches above the top spine
SPEC_YLABEL     = "fraction of\nmismatches"   # y label of each panel
SPEC_YSTEP      = 0.02          # between y ticks
SPEC_SAVE_NAME  = "strand_mismatch_spectrum"

# ------------------------------ spectra --------------------------------------
_pyr = mm.read_pyrimidine.to_numpy(dtype=bool)
spec_counts = pd.DataFrame({g: mm.channel[sel].value_counts().reindex(CHANNELS, fill_value=0)
                            for g, sel in (("all", np.ones(len(mm), bool)),
                                           ("pyrimidine", _pyr), ("purine", ~_pyr))})
spec_frac = spec_counts / len(mm)

panels = [("a", f"Folded strand consensus mismatches (n={len(mm):,})", "all"),
          ("b", f"Pyrimidine changed on the read carrying the mismatch (n={spec_counts.pyrimidine.sum():,})", "pyrimidine"),
          ("c", f"Purine changed on the read carrying the mismatch (n={spec_counts.purine.sum():,})", "purine")]
SUBS_PUR = [s.translate(_comp) for s in SUBS]                        # G>T, G>C, G>A, A>T, A>G, A>C
CTX_PYR  = [c[0] + c[6] for c in CHANNELS]                           # 5' and 3' base of each pyrimidine channel
CTX_PUR  = [c[6].translate(_comp) + c[0].translate(_comp) for c in CHANNELS]   # of the purine change drawn there

# ------------------------------ figure ---------------------------------------
W, H = FIGSIZE
left, right, top, bottom = SPEC_MARGINS_IN
gap_ab, gap_bc = SPEC_GAPS_IN
pw, ph = W - left - right, (H - top - bottom - gap_ab - gap_bc) / 3
fig = plt.figure(figsize=FIGSIZE)
y0s = [bottom + 2 * ph + gap_ab + gap_bc, bottom + ph + gap_bc, bottom]          # a, b, c from the top
axes = [fig.add_axes([left / W, y0 / H, pw / W, ph / H]) for y0 in y0s]
ymax = spec_frac.to_numpy().max() * SPEC_YPAD
for ax, (letter, label, col) in zip(axes, panels):
    ax.bar(range(96), spec_frac[col].to_numpy(), width=SPEC_BARW,
           color=[SPEC_COLORS[c[2:5]] for c in CHANNELS], linewidth=0)
    ax.set_ylim(0, ymax); ax.set_xlim(-0.8, 95.8)
    ax.set_ylabel(SPEC_YLABEL, fontsize=FONT["label"])
    ax.yaxis.set_major_locator(ticker.MultipleLocator(SPEC_YSTEP))
    ax.tick_params(axis="y", labelsize=FONT["tick"])
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.text(0, 1 + SPEC_TITLE_IN / ph, rf"$\mathbf{{{letter}}}$  {label}", transform=ax.transAxes, ha="left",
            va="baseline", fontsize=FONT["title"])
    for j, sub in enumerate(SUBS):
        ax.add_patch(Rectangle((j * 16 - .5, ymax * 1.005), 16, ymax * .055,
                               color=SPEC_COLORS[sub], clip_on=False, zorder=3))
        ax.text(j * 16 + 7.5, ymax * 1.075, sub if col != "purine" else SUBS_PUR[j], ha="center", va="bottom",
                fontsize=FONT["legend"], fontweight="bold", clip_on=False)
    ax.set_xticks(range(96))
    ax.tick_params(axis="x", length=1.5, pad=1)
    if col == "all":
        ax.set_xticklabels([])
    else:
        ax.set_xticklabels(CTX_PYR if col == "pyrimidine" else CTX_PUR, fontsize=FONT["context"],
                           family="monospace", rotation=90)
        ax.set_xlabel("trinucleotide context (5' and 3' base)", fontsize=FONT["label"])

save_figure(fig, SPEC_SAVE_NAME)
plt.show()

# ------------------------------ printed --------------------------------------
_cls = spec_counts.groupby([c[2:5] for c in CHANNELS]).sum()
_cls.loc["C>T at CpG"] = spec_counts.loc[[c for c in CHANNELS if c[2:5] == "C>T" and c[6] == "G"]].sum()
_cls["% pyrimidine"] = (100 * _cls.pyrimidine / _cls["all"]).round(1)
_cls.index = [f"{c} / {c[0].translate(_comp)}>{c[2].translate(_comp)}" if c in SUBS else c for c in _cls.index]
print(f"ext{EXT}, pooled over {len(LIBS)} libraries, site filters {SITE_FILTERS}")
print("substitution class, written as pyrimidine / purine on the read carrying the mismatch:")
print(_cls.to_string())

_rev = mm.err_reverse.to_numpy(dtype=bool)
_alt, _l, _r = mm.err_base.to_numpy(), mm.triplet.str[0].to_numpy(), mm.triplet.str[2].to_numpy()
_before = np.where(_rev, _alt == _r, _alt == _l)     # on a minus-strand read, the base before is the reference-right base
_after  = np.where(_rev, _alt == _l, _alt == _r)
_nb = pd.DataFrame({"n": [(_before & ~_after).sum(), (_after & ~_before).sum(),
                          (_before & _after).sum(), (~_before & ~_after).sum()]},
                   index=["the base before it (5') only", "the base after it (3') only", "both", "neither"])
_nb["%"] = (100 * _nb.n / len(mm)).round(1)
print("\nwrong base equals, on the read carrying the mismatch:")
print(_nb.to_string())

## Supplementary Fig. 15. Neighbour-joining tree of the seven animals

Neighbour-joining tree of the allelic differences between the animals at the sites callable in every library (`animal_genotypes()`, setup 0a), rooted on the reference, with bootstrap support and the length of one generation (Methods, Relationships among the animals).

In [ ]:
FIGSIZE = (PAGE_WIDTH, 3.6)            # width, height in inches
# =============================================================================
# Supplementary Fig. 15. Neighbour-joining tree of the seven animals
#   Distance: allelic differences, sum over the sites of |g_i - g_j| (REF = 0 at every site).
#   Neighbour joining (Biopython), rooted on REF and drawn from the animals' most recent common
#   ancestor, with REF as a leaf. Bootstrap: N_BOOT resamplings of the sites with replacement; the
#   support of a group is the percentage of replicate trees containing it. Branch lengths: allelic
#   differences per bp genotyped.
#   Input   animal_relationships/ (nj_sites.tsv, callable_L.tsv); callable_bps/trimmed/denominator.tsv
# =============================================================================
from Bio.Phylo.TreeConstruction import DistanceMatrix, DistanceTreeConstructor
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize

# ── KNOBS ────────────────────────────────────────────────────────────────────
MIN_COV  = 10            # coverage filter: untrimmed coverage from MIN_COV (8, 9 or 10) to the library's untrimmed p99
N_BOOT   = 10_000
SEED     = 0
MU       = 3.9e-9        # germline mutations per bp per haploid genome per generation (Lindsay et al., 2019);
                         # the scale bar is 2 * MU

# ── STYLE KNOBS ──────────────────────────────────────────────────────────────
X_UNIT       = 1e-7                       # x axis in units of 10^-7 allelic differences per bp
X_LABEL      = r"Allelic differences per bp ($\times10^{-7}$)"
INK          = "#1a1a1a"                  # text, axis, branches with 100% support
LW_BRANCH    = 1.2
TIP_SIZE     = FONT["label"]
LABEL_SIZE   = FONT["label"]
TICK_SIZE    = FONT["tick"]
Y_BOTTOM     = -0.6                       # lower y limit; tips are 1 apart, REF the top one
BOOT_LABELS  = False                      # print the bootstrap support (%) at each node
BOOT_LABEL_MAX = 100                      # print it only where it is below this (e.g. 90); 100 prints all
BOOT_SIZE    = FONT["legend"]
SHADE        = True                       # shade branches by bootstrap support, with a colour bar
BOOT_LOW     = 70                         # shading: support (%) drawn in BOOT_LIGHT, and anything below
BOOT_LIGHT   = "#c8c8c8"
CBAR_RECT    = (0.70, 0.12, 0.20, 0.035)  # bootstrap colour bar: x, y, width, height (axes fraction), lower right
CBAR_TICKS   = [70, 80, 90, 100]
CBAR_SIZE    = FONT["small"]
GEN_BAR_DY   = 0.45                       # generation scale bar: height above the reference's branch (tip units)
GEN_COLOR    = "#c0392b"
GEN_LW       = 2.0
GEN_CAP      = 0.12                       # half-height of its end caps (tip units)
GEN_LABEL    = "1 generation"
GEN_SIZE     = FONT["legend"]
SAVE_NAME    = "nj_tree_inherited_variants"
TIP_LABELS   = {"12wk_1": "12wk #1", "12wk_2": "12wk #2", "48wk_1": "48wk #1", "48wk_2": "48wk #2",
                "60wk": "60wk", "72wk_1": "72wk #1", "72wk_2": "72wk #2", "REF": "B6 T2T reference"}

# ── TREE AND BOOTSTRAP ───────────────────────────────────────────────────────
sites, counts = animal_genotypes(MIN_COV)
L     = counts["genotyped_bp"]
TAXA  = list(ANIMALS) + ["REF"]
NT    = len(TAXA)
PAIRS = [(i, j) for i in range(NT) for j in range(i + 1, NT)]
X     = np.vstack([sites[[f"{a}_GENOTYPE" for a in ANIMALS]].to_numpy().T, np.zeros((1, len(sites)))])   # taxa x sites
DP    = np.abs(X[[i for i, _ in PAIRS]] - X[[j for _, j in PAIRS]])                                      # pairs x sites


def square(vec):
    """Pairwise vector (PAIRS order) -> symmetric NT x NT matrix."""
    D = np.zeros((NT, NT))
    for k, (i, j) in enumerate(PAIRS):
        D[i, j] = D[j, i] = vec[k]
    return D


def nj_rooted(D):
    """Neighbour-joining tree of TAXA from the distance matrix D, rooted on REF."""
    t = DistanceTreeConstructor().nj(DistanceMatrix(names=TAXA, matrix=[[float(D[i, j]) for j in range(i + 1)] for i in range(NT)]))
    t.root_with_outgroup({"name": "REF"})
    return t


def groups(t):
    """Groups of animals of a rooted tree, as sets of tip names (2 to NT - 2 animals)."""
    out = set()
    for cl in t.get_nonterminals():
        tips = frozenset(x.name for x in cl.get_terminals())
        if 1 < len(tips) < NT - 1 and "REF" not in tips:
            out.add(tips)
    return out


D    = square(DP.sum(axis=1))
tree = nj_rooted(D)
tree.ladderize()
hits = dict.fromkeys(groups(tree), 0)

# drawn from the animals' most recent common ancestor: REF becomes a leaf attached there, with the whole
# distance from the ancestor to the reference as its branch, and is placed last (the top tip)
(ref,)  = [c for c in tree.root.clades if c.name == "REF"]
(mrca,) = [c for c in tree.root.clades if c.name != "REF"]
ref.branch_length = (ref.branch_length or 0.0) + (mrca.branch_length or 0.0)
mrca.branch_length = 0.0
mrca.clades.append(ref)
tree.root = mrca
rng  = np.random.default_rng(SEED)
for _ in range(N_BOOT):
    w = np.bincount(rng.integers(0, len(sites), len(sites)), minlength=len(sites)).astype(float)
    replicate = groups(nj_rooted(square(DP @ w)))
    for g in hits:
        hits[g] += g in replicate
support = {g: 100 * n / N_BOOT for g, n in hits.items()}

# ── PLOT ─────────────────────────────────────────────────────────────────────
x_of, y_of = {}, {}


def place(cl, dist):
    x_of[id(cl)] = dist / L / X_UNIT
    for c in cl.clades:
        place(c, dist + (c.branch_length or 0.0))


place(tree.root, 0.0)
for i, tip in enumerate(tree.get_terminals()):
    y_of[id(tip)] = float(i)
for cl in tree.get_nonterminals(order="postorder"):
    y_of[id(cl)] = float(np.mean([y_of[id(c)] for c in cl.clades]))

cmap = LinearSegmentedColormap.from_list("boot", [BOOT_LIGHT, INK])
norm = Normalize(BOOT_LOW, 100, clip=True)


def colour(cl):
    """Bootstrap shade for a group of animals (SHADE), INK otherwise and for the tips and the root."""
    tips = frozenset(t.name for t in cl.get_terminals())
    return cmap(norm(support[tips])) if SHADE and tips in support else INK


fig, ax = plt.subplots(figsize=FIGSIZE)
for cl in tree.find_clades():
    for ch in cl.clades:                                             # horizontal branches
        ax.plot([x_of[id(cl)], x_of[id(ch)]], [y_of[id(ch)]] * 2, color=colour(ch), lw=LW_BRANCH,
                solid_capstyle="round", zorder=3)
    if cl.clades:                                                    # vertical connectors
        ys = [y_of[id(c)] for c in cl.clades]
        ax.plot([x_of[id(cl)]] * 2, [min(ys), max(ys)], color=colour(cl), lw=LW_BRANCH,
                solid_capstyle="round", zorder=3)
for tip in tree.get_terminals():                                     # tip labels
    ax.annotate("  " + TIP_LABELS[tip.name], (x_of[id(tip)], y_of[id(tip)]), va="center",
                fontsize=TIP_SIZE, color=INK, zorder=5)
if BOOT_LABELS:                                                      # bootstrap support at each node
    for cl in tree.get_nonterminals():
        s = support.get(frozenset(t.name for t in cl.get_terminals()))
        if s is not None and s < BOOT_LABEL_MAX:
            ax.annotate(f"{s:.0f}", (x_of[id(cl)], y_of[id(cl)]), xytext=(-2, 1.5), textcoords="offset points",
                        ha="right", va="bottom", fontsize=BOOT_SIZE, color=INK, zorder=5)

# generation scale bar, 2 * MU long, centred above the reference's branch
x_mid, y_bar = x_of[id(ref)] / 2, y_of[id(ref)] + GEN_BAR_DY
x0, x1 = x_mid - MU / X_UNIT, x_mid + MU / X_UNIT
ax.plot([x0, x1], [y_bar] * 2, color=GEN_COLOR, lw=GEN_LW, solid_capstyle="butt", zorder=6)
for x in (x0, x1):
    ax.plot([x, x], [y_bar - GEN_CAP, y_bar + GEN_CAP], color=GEN_COLOR, lw=GEN_LW, solid_capstyle="butt", zorder=6)
ax.text(x_mid, y_bar + GEN_CAP + 0.08, GEN_LABEL, ha="center", va="bottom", fontsize=GEN_SIZE, color=INK, zorder=6)

if SHADE:                                                            # bootstrap colour bar
    cax = ax.inset_axes(CBAR_RECT)
    cb = fig.colorbar(ScalarMappable(norm=Normalize(BOOT_LOW, 100), cmap=cmap), cax=cax,
                      orientation="horizontal", ticks=CBAR_TICKS)
    cb.outline.set_linewidth(0.5)
    cb.outline.set_edgecolor(INK)
    cax.tick_params(labelsize=CBAR_SIZE, colors=INK, width=0.6, length=2)
    cax.set_title("Bootstrap support (%)", fontsize=CBAR_SIZE, color=INK, loc="left", pad=3)

x_max = max(x_of.values())
ax.set_ylim(Y_BOTTOM, y_bar + GEN_CAP + 0.55)
ax.set_xlabel(X_LABEL, fontsize=LABEL_SIZE, color=INK)
ax.set_yticks([])
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color(INK)
ax.spines["bottom"].set_linewidth(0.8)
ax.tick_params(colors=INK, labelsize=TICK_SIZE, width=0.8, length=3)
ax.set_xlim(-0.03 * x_max, 1.3 * x_max)
for _ in range(3):                                                   # right edge = end of the longest tip label
    fig.canvas.draw()
    inv = ax.transData.inverted()
    ax.set_xlim(-0.03 * x_max, max(inv.transform((t.get_window_extent().x1, 0))[0] for t in ax.texts) + 0.02 * x_max)

fig.tight_layout()
save_figure(fig, SAVE_NAME)
plt.show()

# ── NUMBERS FOR THE TEXT ─────────────────────────────────────────────────────
between = [D[i, j] for i, j in PAIRS if j < NT - 1]
print(f"{len(sites):,} sites, {L:,} bp genotyped, {N_BOOT:,} bootstrap replicates")
print(f"between animals: {min(between):.0f}–{max(between):.0f} allelic differences (mean {np.mean(between):.1f}), "
      f"{min(between) / L:.2e}–{max(between) / L:.2e} per bp")
print(f"to the reference: {min(D[-1, :-1]):.0f}–{max(D[-1, :-1]):.0f} allelic differences")
print(f"one generation: 2 x {MU:.1e} = {2 * MU:.2e} per bp, {2 * MU * L:.1f} allelic differences over the {L / 1e9:.2f} Gb")
print("groups (bootstrap support):")
for g, s in sorted(support.items(), key=lambda kv: (-kv[1], len(kv[0]))):
    print(f"  {s:5.1f}%  {', '.join(sorted(g))}")

# ── NUMBERS FOR THE TEXT: the variant sites ──────────────────────────────────
GT = sites[[f"{a}_GENOTYPE" for a in ANIMALS]].to_numpy()
carriers = (GT > 0).sum(axis=1)
private = [int(((GT[:, j] > 0) & (carriers == 1)).sum()) for j in range(len(ANIMALS))]
print(f"\nsites callable in every library, outside the RepeatMasker mask: {counts['genotyped_bp']:,} bp "
      f"({counts['genotyped_bp'] / counts['autosome_bp']:.1%} of the autosomes)")
print(f"called positions there: {counts['called']:,}; singletons removed: {counts['singletons']:,}; "
      f"variant sites: {len(sites):,}")
print(f"carried by one animal: {int((carriers == 1).sum())} ({min(private)}–{max(private)} per animal), "
      f"by 2–{len(ANIMALS) - 1}: {int(((carriers > 1) & (carriers < len(ANIMALS))).sum())}, "
      f"by all {len(ANIMALS)}: {int((carriers == len(ANIMALS)).sum())}")
print("private per animal: " + ", ".join(f"{a} {n}" for a, n in zip(ANIMALS, private)))
print(f"genotypes: 0 {int((GT == 0).sum()):,}, 1 {int((GT == 1).sum()):,}, 2 {int((GT == 2).sum()):,}")